In [1]:

from pathlib import Path

DATASET_ROOT = Path("hackathon-100")

company_dirs = sorted(
    p for p in DATASET_ROOT.glob("company-*")
    if p.is_dir()
)

print("Dataset exists:", DATASET_ROOT.exists())
print("Total companies:", len(company_dirs))

print("\nFirst 10 companies:")
for company in company_dirs[:10]:
    print(company.name)

print("\nLast company:", company_dirs[-1].name)


Dataset exists: True
Total companies: 100

First 10 companies:
company-001
company-002
company-003
company-004
company-005
company-006
company-007
company-008
company-009
company-010

Last company: company-100


In [2]:

import yaml
import pandas as pd

def find_company_data_path(company_dir):
    """Locate the world directory without hardcoding its ID."""
    world_dirs = sorted(
        p for p in (company_dir / "rwm-v4").glob("world-*")
        if p.is_dir()
    )

    if len(world_dirs) != 1:
        raise ValueError(
            f"{company_dir.name}: expected one world directory, "
            f"found {len(world_dirs)}"
        )

    return world_dirs[0]


def load_yaml_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


records = []

for company_dir in company_dirs:
    data_path = find_company_data_path(company_dir)

    company = load_yaml_file(data_path / "company.yaml")
    customers = load_yaml_file(data_path / "customers.yaml")
    invoices = load_yaml_file(data_path / "invoices.yaml")
    payments = load_yaml_file(
        data_path / "customer_payments.yaml"
    )

    records.append({
        "company_id": company_dir.name,
        "company_metadata_type": type(company).__name__,
        "customers": len(customers),
        "invoices": len(invoices),
        "payments": len(payments),
        "company_metadata": company
    })

company_profile_df = pd.DataFrame(records)

print("Companies profiled:", len(company_profile_df))

print("\nTransaction volume statistics:")
print(
    company_profile_df[
        ["customers", "invoices", "payments"]
    ].describe().round(2)
)

print("\nFirst 10 companies:")
display(
    company_profile_df[
        ["company_id", "customers", "invoices", "payments"]
    ].head(10)
)

print("\nCompany 001 metadata:")
print(company_profile_df.iloc[0]["company_metadata"])


Companies profiled: 100

Transaction volume statistics:
       customers  invoices  payments
count     100.00    100.00    100.00
mean       83.22    844.51   1436.16
std       206.33   1450.74   2600.87
min         1.00      0.00      0.00
25%         6.00     16.00     28.50
50%        19.50    187.50    171.00
75%        65.25    883.00   1155.50
max      1526.00   5551.00  10501.00

First 10 companies:


,company_id,customers,invoices,payments
0,company-001,40,1574,3046
1,company-002,81,3163,6057
2,company-003,136,5132,9813
3,company-004,50,1913,3664
4,company-005,75,3079,5847
5,company-006,167,5315,8730
6,company-007,41,1372,2224
7,company-008,82,2899,4888
8,company-009,143,4883,8033
9,company-010,44,1579,2591



Company 001 metadata:
[{'id': 'company-main', 'legal_name': 'Hearthloaf Bakeries LLC', 'trade_name': 'Hearthloaf Bakeries', 'industry_naics': '311812', 'industry_title': 'Commercial Bakeries', 'email': 'accounts@hearthloafbakeries.example', 'phone': '(555) 588-3550', 'website': 'https://www.hearthloafbakeries.example', 'address': {'line1': '9088 Ledgeway Drive', 'city': 'Kalamazoo', 'state': 'MI', 'postal_code': '49006', 'country': 'United States', 'line2': None}, 'fiscal_year_start_month': 1, 'opening_sim_day': 0, 'opening_date': '2025-10-06', 'sim_actor_id': 'business', 'sim_event_ids': ['event-00001']}]


In [3]:

# STEP 3: Industry diversity and benchmark eligibility

def extract_company_metadata(metadata):
    if isinstance(metadata, list):
        return metadata[0] if metadata else {}
    if isinstance(metadata, dict):
        return metadata
    return {}

company_profile_df["industry"] = (
    company_profile_df["company_metadata"]
    .apply(
        lambda x: extract_company_metadata(x).get(
            "industry_title", "Unknown"
        )
    )
)

company_profile_df["industry_naics"] = (
    company_profile_df["company_metadata"]
    .apply(
        lambda x: extract_company_metadata(x).get(
            "industry_naics", "Unknown"
        )
    )
)

# Categorize companies by available transaction history
def classify_data_coverage(row):
    if row["invoices"] == 0 or row["payments"] == 0:
        return "No invoice/payment history"
    elif row["invoices"] < 100:
        return "Low volume"
    elif row["invoices"] < 1000:
        return "Medium volume"
    else:
        return "High volume"

company_profile_df["volume_group"] = (
    company_profile_df.apply(
        classify_data_coverage, axis=1
    )
)

print("Unique industries:",
      company_profile_df["industry"].nunique())

print("\nTop 15 industries:")
print(
    company_profile_df["industry"]
    .value_counts()
    .head(15)
)

print("\nCompany volume distribution:")
print(
    company_profile_df["volume_group"]
    .value_counts()
)

print("\nCompanies with at least 100 invoices "
      "and 100 payments:")
eligible_companies = company_profile_df[
    (company_profile_df["invoices"] >= 100)
    & (company_profile_df["payments"] >= 100)
]

print(len(eligible_companies))

print("\nExample eligible companies:")
display(
    eligible_companies[
        [
            "company_id",
            "industry",
            "customers",
            "invoices",
            "payments",
            "volume_group"
        ]
    ].head(15)
)


Unique industries: 71

Top 15 industries:
industry
Snack and Nonalcoholic Beverage Bars                                 4
Full-Service Restaurants                                             4
Motion Picture and Video Production                                  3
Lessors of Residential Buildings and Dwellings                       3
Investment Advice                                                    3
Offices of Lawyers                                                   3
Commercial Bakeries                                                  2
Coffee and Tea Manufacturing                                         2
Architectural Services                                               2
Interior Design Services                                             2
Advertising Agencies                                                 2
Caterers                                                             2
All Other Personal Services                                          2
New Single-Family Housing 

,company_id,industry,customers,invoices,payments,volume_group
0,company-001,Commercial Bakeries,40,1574,3046,High volume
1,company-002,Coffee and Tea Manufacturing,81,3163,6057,High volume
2,company-003,Breweries,136,5132,9813,High volume
3,company-004,Cut and Sew Apparel Manufacturing,50,1913,3664,High volume
4,company-005,Motor Vehicle Supplies and New Parts Merchant ...,75,3079,5847,High volume
5,company-006,"Lumber, Plywood, Millwork, and Wood Panel Merc...",167,5315,8730,High volume
6,company-007,"Medical, Dental, and Hospital Equipment and Su...",41,1372,2224,High volume
7,company-008,"Electrical Apparatus and Equipment, Wiring Sup...",82,2899,4888,High volume
8,company-009,Plumbing and Heating Equipment and Supplies (H...,143,4883,8033,High volume
9,company-010,Industrial Machinery and Equipment Merchant Wh...,44,1579,2591,High volume


In [4]:

# STEP 4: Cross-company schema compatibility

required_fields = {
    "customers.yaml": {
        "id", "joined_date", "ended_date"
    },
    "invoices.yaml": {
        "id", "customer_id", "txn_date",
        "due_date", "total_amount"
    },
    "customer_payments.yaml": {
        "id", "customer_id", "txn_date",
        "applications"
    }
}

schema_results = []

for company_dir in company_dirs:

    data_path = find_company_data_path(company_dir)

    for filename, required in required_fields.items():

        data = load_yaml_file(data_path / filename)

        if not isinstance(data, list):
            schema_results.append({
                "company_id": company_dir.name,
                "dataset": filename,
                "records": 0,
                "status": "Invalid structure",
                "missing_fields": "Expected list"
            })
            continue

        if len(data) == 0:
            schema_results.append({
                "company_id": company_dir.name,
                "dataset": filename,
                "records": 0,
                "status": "Empty",
                "missing_fields": ""
            })
            continue

        # Check every record, not only the first
        missing = set()

        for record in data:
            if not isinstance(record, dict):
                missing.add("Invalid record")
                continue

            missing.update(required - set(record.keys()))

            if filename == "customer_payments.yaml":
                applications = record.get("applications", [])

                if not isinstance(applications, list):
                    missing.add("applications: expected list")
                    continue

                for app in applications:
                    if not isinstance(app, dict):
                        missing.add("Invalid application")
                    else:
                        for field in ("document_id", "amount"):
                            if field not in app:
                                missing.add(
                                    f"applications.{field}"
                                )

        schema_results.append({
            "company_id": company_dir.name,
            "dataset": filename,
            "records": len(data),
            "status": (
                "Compatible" if not missing
                else "Missing fields"
            ),
            "missing_fields": ", ".join(sorted(missing))
        })

schema_df = pd.DataFrame(schema_results)

print("Total dataset checks:", len(schema_df))

print("\nSchema compatibility summary:")
print(schema_df["status"].value_counts())

print("\nCompatibility by dataset:")
display(
    pd.crosstab(
        schema_df["dataset"],
        schema_df["status"]
    )
)

print("\nSchema issues:")
display(
    schema_df[
        schema_df["status"].isin(
            ["Missing fields", "Invalid structure"]
        )
    ].head(20)
)


Total dataset checks: 300

Schema compatibility summary:
status
Compatible    270
Empty          30
Name: count, dtype: int64

Compatibility by dataset:


status,Compatible,Empty
dataset,,
customer_payments.yaml,84,16
customers.yaml,100,0
invoices.yaml,86,14



Schema issues:


,company_id,dataset,records,status,missing_fields


In [5]:

from pathlib import Path
import json

benchmark_dir = Path("company003_benchmark_v3")

required_files = [
    "manifest.json",
    "train_inputs.jsonl",
    "train_labels.jsonl",
    "dev_inputs.jsonl",
    "dev_labels.jsonl",
    "test_inputs.jsonl",
    "test_labels.jsonl",
]

print("Benchmark directory exists:", benchmark_dir.exists())

print("\nBenchmark files:")
for filename in required_files:
    path = benchmark_dir / filename
    print(f"{filename}: {path.exists()}")

manifest_path = benchmark_dir / "manifest.json"

if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)

    print("\nBenchmark version:",
          manifest.get("benchmark_version"))
    print("Task:", manifest.get("task"))
    print("Splits:", manifest.get("splits"))


Benchmark directory exists: True

Benchmark files:
manifest.json: True
train_inputs.jsonl: True
train_labels.jsonl: True
dev_inputs.jsonl: True
dev_labels.jsonl: True
test_inputs.jsonl: True
test_labels.jsonl: True

Benchmark version: v3_candidate
Task: Customer relationship change detection and escalation
Splits: {'train': {'examples': 7764, 'customers': 69, 'meaningful_labels': {'True': 4917, 'False': 2847}, 'escalation_labels': {'True': 4037, 'False': 3727}}, 'dev': {'examples': 690, 'customers': 17, 'meaningful_labels': {'True': 430, 'False': 260}, 'escalation_labels': {'True': 349, 'False': 341}}, 'test': {'examples': 570, 'customers': 14, 'meaningful_labels': {'True': 355, 'False': 215}, 'escalation_labels': {'True': 288, 'False': 282}}}


In [6]:

# STEP 6: Inspect Benchmark V3 input and label structure

import json
from pathlib import Path

benchmark_dir = Path("company003_benchmark_v3")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_inputs = load_jsonl(
    benchmark_dir / "train_inputs.jsonl"
)

train_labels = load_jsonl(
    benchmark_dir / "train_labels.jsonl"
)

print("========== BENCHMARK V3 STRUCTURE ==========")

print("\nTraining examples:", len(train_inputs))
print("Training labels:", len(train_labels))

assert len(train_inputs) == len(train_labels)

print("\nINPUT TOP-LEVEL KEYS:")
print(list(train_inputs[0].keys()))

print("\nFIRST MODEL INPUT:")
print(json.dumps(
    train_inputs[0],
    indent=2,
    default=str
))

print("\nFIRST GROUND-TRUTH LABEL:")
print(json.dumps(
    train_labels[0],
    indent=2,
    default=str
))

print("\nLABEL FIELDS:")
print(list(train_labels[0].keys()))

print("\nEVENT TYPE DISTRIBUTION:")
from collections import Counter

event_types = Counter(
    record.get("new_event", {}).get("event_type")
    for record in train_inputs
)

print(event_types)


========== BENCHMARK V3 STRUCTURE ==========

Training examples: 7764
Training labels: 7764

INPUT TOP-LEVEL KEYS:
['event_date', 'old_state', 'previous_event', 'new_event', 'current_state', 'event_impact']

FIRST MODEL INPUT:
{
  "event_date": "2026-03-30",
  "old_state": {
    "outstanding_ar": 1487.61,
    "overdue_ar": 643.92,
    "overdue_invoice_count": 2,
    "max_days_overdue": 25,
    "recent_30d_invoice_count": 5,
    "previous_30d_invoice_count": 7,
    "recent_30d_revenue": 1880.33,
    "previous_30d_revenue": 4063.83
  },
  "previous_event": {
    "event_type": "invoice_created",
    "event_date": "2026-03-27",
    "amount": 349.11,
    "description": "Any shortages should be reported within five days of delivery."
  },
  "new_event": {
    "event_type": "customer_message",
    "event_date": "2026-03-30",
    "amount": 0.0,
    "description": "We're moving most of our purchases to another supplier next month."
  },
  "current_state": {
    "outstanding_ar": 1487.61,
    "o

In [7]:

# STEP 7: Understand the original evaluation distribution

import pandas as pd
from collections import Counter

benchmark_rows = []

for split in ["train", "dev", "test"]:
    inputs = load_jsonl(
        benchmark_dir / f"{split}_inputs.jsonl"
    )
    labels = load_jsonl(
        benchmark_dir / f"{split}_labels.jsonl"
    )

    assert len(inputs) == len(labels)

    for x, y in zip(inputs, labels):
        benchmark_rows.append({
            "split": split,
            "event_type": x["new_event"]["event_type"],
            "scenario_type": y["scenario_type"],
            "meaningful_change": y["meaningful_change"],
            "escalation_needed": y["escalation_needed"],
            "direction": y["direction"],
            "description": x["new_event"]["description"],
        })

benchmark_analysis_df = pd.DataFrame(benchmark_rows)

print("TOTAL EXAMPLES:", len(benchmark_analysis_df))

print("\nEVENT TYPES BY SPLIT:")
display(pd.crosstab(
    benchmark_analysis_df["split"],
    benchmark_analysis_df["event_type"]
))

print("\nTOP 20 SCENARIO TYPES:")
display(
    benchmark_analysis_df["scenario_type"]
    .value_counts()
    .head(20)
)

print("\nLABEL BALANCE BY EVENT TYPE:")
display(
    benchmark_analysis_df.groupby("event_type")[
        ["meaningful_change", "escalation_needed"]
    ].agg(["count", "mean"])
    .round(3)
)

print("\nDIRECTION BY EVENT TYPE:")
display(pd.crosstab(
    benchmark_analysis_df["event_type"],
    benchmark_analysis_df["direction"]
))

print("\nUNIQUE DESCRIPTIONS BY EVENT TYPE:")
display(
    benchmark_analysis_df.groupby("event_type")[
        "description"
    ].nunique()
)

print("\nFINANCIAL SCENARIO TYPES:")
display(
    benchmark_analysis_df[
        benchmark_analysis_df["event_type"]
        == "financial_trend_observed"
    ]["scenario_type"].value_counts()
)


TOTAL EXAMPLES: 9024

EVENT TYPES BY SPLIT:


event_type,customer_message,financial_trend_observed
split,,
dev,612,78
test,504,66
train,7452,312



TOP 20 SCENARIO TYPES:


scenario_type
supplier_switching             238
order_reduction                238
payment_distress               238
contract_termination           238
order_expansion                238
preferred_supplier             238
payment_recovery               238
new_long_term_contract         238
routine_delivery               238
routine_invoice                238
minor_schedule_change          238
routine_payment                238
minor_order_increase           238
small_recurring_reduction      238
credit_terms_request           238
one_time_large_order           238
competitor_trial               238
location_closures              238
recurring_delivery_failures    238
budget_freeze                  238
Name: count, dtype: int64


LABEL BALANCE BY EVENT TYPE:


...

direction,negative,neutral,positive
event_type,,,
customer_message,3094,3094,2380
financial_trend_observed,152,228,76



UNIQUE DESCRIPTIONS BY EVENT TYPE:


event_type
customer_message            180
financial_trend_observed      1
Name: description, dtype: int64


FINANCIAL SCENARIO TYPES:


scenario_type
revenue_collapse             76
revenue_surge                76
moderate_revenue_decline     76
stable_revenue               76
small_revenue_fluctuation    76
minor_revenue_dip            76
Name: count, dtype: int64

In [8]:

# STEP 8: Inspect financial scenario definitions and numeric patterns

financial_examples = benchmark_analysis_df[
    benchmark_analysis_df["event_type"] == "financial_trend_observed"
]

print("Financial scenarios:")
print(financial_examples["scenario_type"].value_counts())

print("\nOne complete example per financial scenario:\n")

for scenario in sorted(financial_examples["scenario_type"].unique()):

    found = False

    for split in ["train", "dev", "test"]:
        inputs = load_jsonl(
            benchmark_dir / f"{split}_inputs.jsonl"
        )
        labels = load_jsonl(
            benchmark_dir / f"{split}_labels.jsonl"
        )

        for x, y in zip(inputs, labels):
            if (
                x["new_event"]["event_type"]
                == "financial_trend_observed"
                and y["scenario_type"] == scenario
            ):
                print("=" * 65)
                print("SCENARIO:", scenario)
                print("SPLIT:", split)

                print("\nOLD STATE:")
                print(json.dumps(x["old_state"], indent=2))

                print("\nNEW EVENT:")
                print(json.dumps(x["new_event"], indent=2))

                print("\nCURRENT STATE:")
                print(json.dumps(x["current_state"], indent=2))

                print("\nEVENT IMPACT:")
                print(json.dumps(x["event_impact"], indent=2))

                print("\nGROUND TRUTH:")
                print({
                    "meaningful_change": y["meaningful_change"],
                    "direction": y["direction"],
                    "escalation_needed": y["escalation_needed"]
                })

                found = True
                break

        if found:
            break


Financial scenarios:
scenario_type
revenue_collapse             76
revenue_surge                76
moderate_revenue_decline     76
stable_revenue               76
small_revenue_fluctuation    76
minor_revenue_dip            76
Name: count, dtype: int64

One complete example per financial scenario:

SCENARIO: minor_revenue_dip
SPLIT: train

OLD STATE:
{
  "outstanding_ar": 1487.61,
  "overdue_ar": 643.92,
  "overdue_invoice_count": 2,
  "max_days_overdue": 25,
  "recent_30d_invoice_count": 5,
  "previous_30d_invoice_count": 7,
  "recent_30d_revenue": 1880.33,
  "previous_30d_revenue": 4063.83
}

NEW EVENT:
{
  "event_type": "financial_trend_observed",
  "event_date": "2026-03-30",
  "amount": 0.0,
  "description": "Hypothetical purchasing-trend report comparing a historical 30-day baseline with a simulated subsequent 30-day period. The QuickBooks financial states represent historical accounting records and are not updated by this report.",
  "observation": {
    "baseline_30d_revenue": 

In [9]:

# STEP 9: Recover financial scenario ranges from Benchmark V3

import pandas as pd

financial_rows = []

for split in ["train", "dev", "test"]:
    inputs = load_jsonl(
        benchmark_dir / f"{split}_inputs.jsonl"
    )
    labels = load_jsonl(
        benchmark_dir / f"{split}_labels.jsonl"
    )

    for x, y in zip(inputs, labels):
        if x["new_event"]["event_type"] != "financial_trend_observed":
            continue

        observation = x["new_event"].get("observation", {})

        financial_rows.append({
            "split": split,
            "scenario": y["scenario_type"],
            "revenue_change_pct": observation.get(
                "revenue_change_pct"
            ),
            "baseline_revenue": observation.get(
                "baseline_30d_revenue"
            ),
            "observed_revenue": observation.get(
                "observed_30d_revenue"
            ),
            "baseline_invoice_count": observation.get(
                "baseline_30d_invoice_count"
            ),
            "meaningful_change": y["meaningful_change"],
            "direction": y["direction"],
            "escalation_needed": y["escalation_needed"]
        })

financial_scenarios_df = pd.DataFrame(financial_rows)

print("Financial examples:", len(financial_scenarios_df))

print("\nREVENUE CHANGE RANGES BY SCENARIO:")
display(
    financial_scenarios_df.groupby("scenario")[
        "revenue_change_pct"
    ].agg(
        ["count", "min", "median", "max"]
    ).round(2)
)

print("\nBASELINE REVENUE DISTRIBUTION:")
display(
    financial_scenarios_df["baseline_revenue"]
    .describe()
    .round(2)
)

print("\nBASELINE INVOICE COUNT DISTRIBUTION:")
display(
    financial_scenarios_df["baseline_invoice_count"]
    .describe()
    .round(2)
)

print("\nLABEL CONSISTENCY BY SCENARIO:")
display(
    financial_scenarios_df.groupby("scenario").agg(
        meaningful_values=("meaningful_change", "nunique"),
        direction_values=("direction", "nunique"),
        escalation_values=("escalation_needed", "nunique")
    )
)


Financial examples: 456

REVENUE CHANGE RANGES BY SCENARIO:


,count,min,median,max
scenario,,,,
minor_revenue_dip,76,-9.80,-6.04,-3.02
moderate_revenue_decline,76,-34.79,-28.48,-20.33
revenue_collapse,76,-89.54,-73.92,-60.60
revenue_surge,76,71.26,126.72,179.87
small_revenue_fluctuation,76,3.07,6.88,9.90
stable_revenue,76,-2.00,0.12,2.00



BASELINE REVENUE DISTRIBUTION:


count     456.00
mean     1659.48
std       775.48
min       395.08
25%      1137.54
50%      1605.15
75%      2123.91
max      4063.83
Name: baseline_revenue, dtype: float64


BASELINE INVOICE COUNT DISTRIBUTION:


count    456.00
mean       4.55
std        1.88
min        2.00
25%        3.00
50%        4.00
75%        6.00
max        9.00
Name: baseline_invoice_count, dtype: float64


LABEL CONSISTENCY BY SCENARIO:


,meaningful_values,direction_values,escalation_values
scenario,,,
minor_revenue_dip,1,1,1
moderate_revenue_decline,1,1,1
revenue_collapse,1,1,1
revenue_surge,1,1,1
small_revenue_fluctuation,1,1,1
stable_revenue,1,1,1


In [10]:

# STEP 10: Generate held-out financial scenarios for Company 001

import copy
import numpy as np
import pandas as pd

TARGET_COMPANY = "company-001"
target_dir = find_company_data_path(
    DATASET_ROOT / TARGET_COMPANY
)

invoices = load_yaml_file(target_dir / "invoices.yaml")
invoices_df_target = pd.DataFrame(invoices)

print("Company:", TARGET_COMPANY)
print("Invoices:", len(invoices_df_target))

# Use a fixed historical cutoff; adjust only if outside
# the company's observed invoice history.
invoices_df_target["txn_date"] = pd.to_datetime(
    invoices_df_target["txn_date"]
)
invoices_df_target["total_amount"] = pd.to_numeric(
    invoices_df_target["total_amount"],
    errors="coerce"
)

cutoff = invoices_df_target["txn_date"].max()

baseline_start = cutoff - pd.Timedelta(days=60)
baseline_end = cutoff - pd.Timedelta(days=30)

baseline_invoices = invoices_df_target[
    (invoices_df_target["txn_date"] >= baseline_start)
    & (invoices_df_target["txn_date"] < baseline_end)
].copy()

customer_baselines = (
    baseline_invoices
    .groupby("customer_id")
    .agg(
        baseline_revenue=("total_amount", "sum"),
        baseline_invoice_count=("id", "count")
    )
    .reset_index()
)

customer_baselines = customer_baselines[
    (customer_baselines["baseline_revenue"] > 0)
    & (customer_baselines["baseline_invoice_count"] >= 2)
]

# Scenario percentage changes based on observed V3 medians.
# These are fixed values, not tuned on Company 001.
scenario_changes = {
    "revenue_collapse": -73.92,
    "moderate_revenue_decline": -28.48,
    "minor_revenue_dip": -6.04,
    "stable_revenue": 0.12,
    "small_revenue_fluctuation": 6.88,
    "revenue_surge": 126.72
}

# Reuse labels from original benchmark
scenario_labels = (
    financial_scenarios_df
    .groupby("scenario")
    .agg({
        "meaningful_change": "first",
        "direction": "first",
        "escalation_needed": "first"
    })
    .to_dict("index")
)

generated = []

for _, customer in customer_baselines.iterrows():
    for scenario, pct in scenario_changes.items():
        baseline = float(customer["baseline_revenue"])
        observed = round(baseline * (1 + pct / 100), 2)

        generated.append({
            "company_id": TARGET_COMPANY,
            "customer_id": customer["customer_id"],
            "scenario": scenario,
            "baseline_revenue": round(baseline, 2),
            "baseline_invoice_count": int(
                customer["baseline_invoice_count"]
            ),
            "observed_revenue": observed,
            "revenue_change_pct": pct,
            **scenario_labels[scenario]
        })

company001_financial_df = pd.DataFrame(generated)

print("\nCutoff:", cutoff.date())
print("Eligible customers:", len(customer_baselines))
print("Generated scenarios:", len(company001_financial_df))

print("\nScenarios:")
print(company001_financial_df["scenario"].value_counts())

print("\nExample rows:")
display(company001_financial_df.head(12))

print("\nBaseline revenue distribution:")
display(
    customer_baselines["baseline_revenue"]
    .describe()
    .round(2)
)


Company: company-001
Invoices: 1574

Cutoff: 2026-10-06
Eligible customers: 30
Generated scenarios: 180

Scenarios:
scenario
revenue_collapse             30
moderate_revenue_decline     30
minor_revenue_dip            30
stable_revenue               30
small_revenue_fluctuation    30
revenue_surge                30
Name: count, dtype: int64

Example rows:


,company_id,customer_id,scenario,baseline_revenue,baseline_invoice_count,observed_revenue,revenue_change_pct,meaningful_change,direction,escalation_needed
0,company-001,cust-customer-1,revenue_collapse,3150.31,5,821.60,-73.92,True,negative,True
1,company-001,cust-customer-1,moderate_revenue_decline,3150.31,5,2253.10,-28.48,True,negative,False
2,company-001,cust-customer-1,minor_revenue_dip,3150.31,5,2960.03,-6.04,False,neutral,False
3,company-001,cust-customer-1,stable_revenue,3150.31,5,3154.09,0.12,False,neutral,False
4,company-001,cust-customer-1,small_revenue_fluctuation,3150.31,5,3367.05,6.88,False,neutral,False
5,company-001,cust-customer-1,revenue_surge,3150.31,5,7142.38,126.72,True,positive,True
6,company-001,cust-customer-11,revenue_collapse,2692.67,5,702.25,-73.92,True,negative,True
7,company-001,cust-customer-11,moderate_revenue_decline,2692.67,5,1925.80,-28.48,True,negative,False
8,company-001,cust-customer-11,minor_revenue_dip,2692.67,5,2530.03,-6.04,False,neutral,False
9,company-001,cust-customer-11,stable_revenue,2692.67,5,2695.90,0.12,False,neutral,False



Baseline revenue distribution:


count      30.00
mean     2100.71
std       791.11
min       890.98
25%      1503.56
50%      2076.91
75%      2682.58
max      3667.86
Name: baseline_revenue, dtype: float64

In [11]:

# STEP 11: Reconstruct historical financial state features

import pandas as pd
import json

payments_target = load_yaml_file(
    target_dir / "customer_payments.yaml"
)

payments_df_target = pd.DataFrame(payments_target)

# Historical cutoff from Step 10
cutoff_date = pd.Timestamp(cutoff)

recent_start = cutoff_date - pd.Timedelta(days=30)
previous_start = cutoff_date - pd.Timedelta(days=60)

historical_invoices = invoices_df_target[
    invoices_df_target["txn_date"] <= cutoff_date
].copy()

def calculate_invoice_features(customer_id):
    customer_invoices = historical_invoices[
        historical_invoices["customer_id"] == customer_id
    ]

    recent = customer_invoices[
        (customer_invoices["txn_date"] > recent_start)
        & (customer_invoices["txn_date"] <= cutoff_date)
    ]

    previous = customer_invoices[
        (customer_invoices["txn_date"] > previous_start)
        & (customer_invoices["txn_date"] <= recent_start)
    ]

    return {
        "recent_30d_invoice_count": int(len(recent)),
        "previous_30d_invoice_count": int(len(previous)),
        "recent_30d_revenue": round(
            float(recent["total_amount"].sum()), 2
        ),
        "previous_30d_revenue": round(
            float(previous["total_amount"].sum()), 2
        )
    }

print("========== COMPANY 001 STATE INSPECTION ==========")

print("\nInvoice fields:")
print(invoices_df_target.columns.tolist())

print("\nPayment fields:")
print(payments_df_target.columns.tolist())

print("\nExample payment application:")
sample_payment = next(
    (
        p for p in payments_target
        if p.get("applications")
    ),
    None
)

if sample_payment:
    print(json.dumps(
        {
            "id": sample_payment.get("id"),
            "txn_date": sample_payment.get("txn_date"),
            "customer_id": sample_payment.get("customer_id"),
            "total_amount": sample_payment.get("total_amount"),
            "applications": sample_payment.get("applications")
        },
        indent=2,
        default=str
    ))

print("\nRevenue features for first 5 eligible customers:")

for customer_id in customer_baselines["customer_id"].head(5):
    print(
        customer_id,
        calculate_invoice_features(customer_id)
    )

print("\nCutoff:", cutoff_date.date())
print("Invoices on or before cutoff:", len(historical_invoices))
print("Payment records:", len(payments_df_target))


========== COMPANY 001 STATE INSPECTION ==========

Invoice fields:
['id', 'doc_number', 'customer_id', 'customer_name', 'txn_sim_day', 'txn_date', 'due_sim_day', 'due_date', 'lines', 'total_amount', 'income_account_id', 'project_id', 'deposit_applied_amount', 'retainage_amount', 'deposit_account_id', 'retainage_account_id', 'memo', 'provenance_note', 'sim_order_id', 'sim_obligation_id', 'sim_event_ids']

Payment fields:
['id', 'customer_id', 'customer_name', 'txn_sim_day', 'txn_date', 'amount', 'applications', 'unapplied_amount', 'settles', 'source_account_id', 'deposit_account_id', 'project_id', 'memo', 'provenance_note', 'sim_payment_id', 'sim_obligation_id', 'sim_event_ids']

Example payment application:
{
  "id": "rcpt-00003",
  "txn_date": "2025-10-16",
  "customer_id": "cust-customer-20",
  "total_amount": null,
  "applications": [
    {
      "document_id": "inv-00023",
      "amount": "223.56"
    }
  ]
}

Revenue features for first 5 eligible customers:
cust-customer-1 {'rece

In [12]:

# STEP 12: Build full historical financial states for Company 001

from collections import defaultdict
from decimal import Decimal
import pandas as pd

as_of = pd.Timestamp(cutoff).normalize()

# Match the original Step 10 baseline window exactly:
# [cutoff - 60 days, cutoff - 30 days)
baseline_start = as_of - pd.Timedelta(days=60)
baseline_end = as_of - pd.Timedelta(days=30)

# Use a non-overlapping recent window:
# [cutoff - 30 days, cutoff], inclusive of the cutoff date
recent_start = baseline_end

inv = invoices_df_target.copy()
inv["due_date"] = pd.to_datetime(inv["due_date"])
inv["txn_date"] = pd.to_datetime(inv["txn_date"])

# Convert money through Decimal to avoid floating-point
# accumulation errors in payment reconciliation.
def money(value):
    if value is None or pd.isna(value):
        return Decimal("0")
    return Decimal(str(value))

# Aggregate payments applied to each invoice as of cutoff.
paid_by_invoice = defaultdict(lambda: Decimal("0"))
invoice_ids = set(inv["id"])

future_payments_excluded = 0
unknown_applications = 0

for payment in payments_target:
    payment_date = pd.Timestamp(payment["txn_date"])

    if payment_date > as_of:
        future_payments_excluded += 1
        continue

    for app in payment.get("applications", []) or []:
        document_id = app.get("document_id")

        if document_id not in invoice_ids:
            unknown_applications += 1
            continue

        paid_by_invoice[document_id] += money(
            app.get("amount", 0)
        )

# Build one full state per eligible customer.
financial_states = {}

for customer_id in customer_baselines["customer_id"]:

    customer_inv = inv[
        (inv["customer_id"] == customer_id)
        & (inv["txn_date"] <= as_of)
    ]

    recent = customer_inv[
        (customer_inv["txn_date"] >= recent_start)
        & (customer_inv["txn_date"] <= as_of)
    ]

    previous = customer_inv[
        (customer_inv["txn_date"] >= baseline_start)
        & (customer_inv["txn_date"] < baseline_end)
    ]

    outstanding = Decimal("0")
    overdue = Decimal("0")
    overdue_count = 0
    max_days_overdue = 0

    for _, invoice in customer_inv.iterrows():
        invoice_total = money(invoice["total_amount"])
        applied = paid_by_invoice[invoice["id"]]

        # Do not allow an overpayment to produce negative AR.
        balance = max(
            invoice_total - applied,
            Decimal("0")
        )

        outstanding += balance

        if balance > 0 and invoice["due_date"] < as_of:
            overdue += balance
            overdue_count += 1

            days_overdue = (
                as_of - invoice["due_date"]
            ).days

            max_days_overdue = max(
                max_days_overdue,
                days_overdue
            )

    financial_states[customer_id] = {
        "outstanding_ar": round(float(outstanding), 2),
        "overdue_ar": round(float(overdue), 2),
        "overdue_invoice_count": overdue_count,
        "max_days_overdue": max_days_overdue,
        "recent_30d_invoice_count": int(len(recent)),
        "previous_30d_invoice_count": int(len(previous)),
        "recent_30d_revenue": round(
            float(sum(
                (money(v) for v in recent["total_amount"]),
                Decimal("0")
            )), 2
        ),
        "previous_30d_revenue": round(
            float(sum(
                (money(v) for v in previous["total_amount"]),
                Decimal("0")
            )), 2
        )
    }

# Verify scenario baselines match reconstructed state.
mismatches = []

for _, row in company001_financial_df.iterrows():
    state = financial_states[row["customer_id"]]

    if (
        abs(
            state["previous_30d_revenue"]
            - row["baseline_revenue"]
        ) > 0.01
        or state["previous_30d_invoice_count"]
        != row["baseline_invoice_count"]
    ):
        mismatches.append(row["customer_id"])

print("========== COMPANY 001 STATE VALIDATION ==========")
print("States constructed:", len(financial_states))
print("Future payments excluded:", future_payments_excluded)
print("Unknown payment applications:", unknown_applications)
print("Baseline mismatches:", len(set(mismatches)))

print("\nFirst reconstructed state:")
first_customer = customer_baselines.iloc[0]["customer_id"]
print("Customer:", first_customer)
print(financial_states[first_customer])

print("\nAggregate receivables for eligible customers:")
print(
    "Outstanding AR:",
    round(sum(
        s["outstanding_ar"]
        for s in financial_states.values()
    ), 2)
)
print(
    "Overdue AR:",
    round(sum(
        s["overdue_ar"]
        for s in financial_states.values()
    ), 2)
)

assert not mismatches, "Scenario baseline mismatch detected"
assert len(financial_states) == len(customer_baselines)

print("\nPASS — Financial states reconstructed.")


========== COMPANY 001 STATE VALIDATION ==========
States constructed: 30
Future payments excluded: 0
Unknown payment applications: 0
Baseline mismatches: 0

First reconstructed state:
Customer: cust-customer-1
{'outstanding_ar': 1348.52, 'overdue_ar': 223.48, 'overdue_invoice_count': 1, 'max_days_overdue': 8, 'recent_30d_invoice_count': 4, 'previous_30d_invoice_count': 5, 'recent_30d_revenue': 2892.27, 'previous_30d_revenue': 3150.31}

Aggregate receivables for eligible customers:
Outstanding AR: 24027.25
Overdue AR: 5098.37

PASS — Financial states reconstructed.


In [13]:

# STEP 13: Build model-ready Company 001 benchmark examples

import copy
import json
import pandas as pd

company001_inputs = []
company001_labels = []

for idx, row in company001_financial_df.iterrows():

    customer_id = row["customer_id"]
    state = financial_states[customer_id]

    # Find the customer's most recent historical invoice
    customer_history = historical_invoices[
        historical_invoices["customer_id"] == customer_id
    ].sort_values(["txn_date", "id"])

    latest_invoice = customer_history.iloc[-1]

    previous_event = {
        "event_type": "invoice_created",
        "event_date": latest_invoice["txn_date"].strftime(
            "%Y-%m-%d"
        ),
        "amount": round(
            float(latest_invoice["total_amount"]), 2
        ),
        "description": str(
            latest_invoice.get("memo") or ""
        )
    }

    new_event = {
        "event_type": "financial_trend_observed",
        "event_date": as_of.strftime("%Y-%m-%d"),
        "amount": 0.0,
        "description": (
            "Hypothetical purchasing-trend report comparing "
            "a historical 30-day baseline with a simulated "
            "subsequent 30-day period. The QuickBooks financial "
            "states represent historical accounting records "
            "and are not updated by this report."
        ),
        "observation": {
            "baseline_30d_revenue": row["baseline_revenue"],
            "observed_30d_revenue": row["observed_revenue"],
            "revenue_change_pct": row["revenue_change_pct"],
            "baseline_30d_invoice_count": int(
                row["baseline_invoice_count"]
            ),
            "report_type": "hypothetical_customer_purchasing_trend",
            "baseline_period": "historical_previous_30_day_window",
            "observed_period": "simulated_following_30_day_window"
        }
    }

    model_input = {
        "event_date": as_of.strftime("%Y-%m-%d"),
        "old_state": copy.deepcopy(state),
        "previous_event": previous_event,
        "new_event": new_event,
        "current_state": copy.deepcopy(state),
        "event_impact": {
            "outstanding_ar_delta": 0.0,
            "overdue_ar_delta": 0.0,
            "recent_30d_invoice_count_delta": 0.0,
            "recent_30d_revenue_delta": 0.0
        }
    }

    label = {
        "event_id": f"cross-company-001-fin-{idx:04d}",
        "company_id": TARGET_COMPANY,
        "customer_id": customer_id,
        "scenario_type": row["scenario"],
        "meaningful_change": bool(row["meaningful_change"]),
        "direction": row["direction"],
        "escalation_needed": bool(row["escalation_needed"])
    }

    company001_inputs.append(model_input)
    company001_labels.append(label)

# Validate exact input schema against original benchmark
original_keys = set(train_inputs[0].keys())

assert len(company001_inputs) == 180
assert len(company001_inputs) == len(company001_labels)

assert all(
    set(x.keys()) == original_keys
    for x in company001_inputs
)

assert all(
    x["old_state"] == x["current_state"]
    for x in company001_inputs
)

assert all(
    x["new_event"]["observation"]["baseline_30d_revenue"]
    == x["old_state"]["previous_30d_revenue"]
    for x in company001_inputs
)

print("========== COMPANY 001 MODEL INPUT VALIDATION ==========")
print("Inputs:", len(company001_inputs))
print("Labels:", len(company001_labels))
print("Schema matches Company 003:", True)
print("Financial baselines aligned:", True)

print("\nFirst model input:")
print(json.dumps(company001_inputs[0], indent=2))

print("\nFirst ground-truth label:")
print(json.dumps(company001_labels[0], indent=2))

# Save the held-out benchmark
output_dir = Path("cross_company_benchmark")
output_dir.mkdir(exist_ok=True)

with open(
    output_dir / "company001_financial_inputs.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for record in company001_inputs:
        f.write(json.dumps(record) + "\n")

with open(
    output_dir / "company001_financial_labels.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for record in company001_labels:
        f.write(json.dumps(record) + "\n")

print("\nSaved to:", output_dir)
print("PASS — Company 001 benchmark ready for prediction.")


========== COMPANY 001 MODEL INPUT VALIDATION ==========
Inputs: 180
Labels: 180
Schema matches Company 003: True
Financial baselines aligned: True

First model input:
{
  "event_date": "2026-10-06",
  "old_state": {
    "outstanding_ar": 1348.52,
    "overdue_ar": 223.48,
    "overdue_invoice_count": 1,
    "max_days_overdue": 8,
    "recent_30d_invoice_count": 4,
    "previous_30d_invoice_count": 5,
    "recent_30d_revenue": 2892.27,
    "previous_30d_revenue": 3150.31
  },
  "previous_event": {
    "event_type": "invoice_created",
    "event_date": "2026-09-30",
    "amount": 1125.04,
    "description": "Freight included. Pallets returnable on request."
  },
  "new_event": {
    "event_type": "financial_trend_observed",
    "event_date": "2026-10-06",
    "amount": 0.0,
    "description": "Hypothetical purchasing-trend report comparing a historical 30-day baseline with a simulated subsequent 30-day period. The QuickBooks financial states represent historical accounting records and a

In [14]:

# STEP 14: Inspect saved Company 003 detector artifacts

from pathlib import Path
import json

benchmark_root = Path("company003_benchmark_v3")

print("========== SAVED MODEL ARTIFACTS ==========")

models_dir = benchmark_root / "models"

if models_dir.exists():
    for path in sorted(models_dir.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(benchmark_root),
                "|",
                round(path.stat().st_size / 1024, 1),
                "KB"
            )
else:
    print("No models directory found.")

print("\n========== OTHER PREDICTION-RELATED FILES ==========")

keywords = [
    "predict", "model", "feature",
    "inference", "jev", "baseline"
]

for path in sorted(benchmark_root.rglob("*")):
    if not path.is_file():
        continue

    if "models" in path.parts:
        continue

    if any(word in path.name.lower() for word in keywords):
        print(path.relative_to(benchmark_root))

print("\n========== MANIFEST MODEL INFORMATION ==========")

manifest_path = benchmark_root / "manifest.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    for key, value in manifest.items():
        if any(
            word in key.lower()
            for word in [
                "model", "baseline", "predict",
                "detector", "evaluation"
            ]
        ):
            print(f"\n{key}:")
            print(json.dumps(value, indent=2, default=str))

print("\n========== MODEL FILE AVAILABILITY ==========")

for filename in [
    "meaningful.joblib",
    "escalation.joblib",
    "direction.joblib",
    "tfidf.joblib"
]:
    matches = list(benchmark_root.rglob(filename))
    print(filename, "->", [str(p) for p in matches])


========== SAVED MODEL ARTIFACTS ==========
models/.DS_Store | 6.0 KB
models/xgboost_v1/direction.joblib | 889.8 KB
models/xgboost_v1/escalation.joblib | 285.0 KB
models/xgboost_v1/meaningful.joblib | 290.7 KB
models/xgboost_v1/results.json | 2.2 KB
models/xgboost_v1/tfidf.joblib | 56.0 KB

========== OTHER PREDICTION-RELATED FILES ==========

========== MANIFEST MODEL INFORMATION ==========

========== MODEL FILE AVAILABILITY ==========
meaningful.joblib -> ['company003_benchmark_v3/models/xgboost_v1/meaningful.joblib']
escalation.joblib -> ['company003_benchmark_v3/models/xgboost_v1/escalation.joblib']
direction.joblib -> ['company003_benchmark_v3/models/xgboost_v1/direction.joblib']
tfidf.joblib -> ['company003_benchmark_v3/models/xgboost_v1/tfidf.joblib']


In [15]:

# STEP 15: Inspect frozen XGBoost feature requirements

import joblib
import json
from pathlib import Path

model_dir = Path(
    "company003_benchmark_v3/models/xgboost_v1"
)

meaningful_model = joblib.load(
    model_dir / "meaningful.joblib"
)
escalation_model = joblib.load(
    model_dir / "escalation.joblib"
)
direction_model = joblib.load(
    model_dir / "direction.joblib"
)
tfidf_vectorizer = joblib.load(
    model_dir / "tfidf.joblib"
)

print("========== FROZEN MODEL INSPECTION ==========")

models = {
    "meaningful": meaningful_model,
    "escalation": escalation_model,
    "direction": direction_model
}

for name, model in models.items():
    print(f"\n{name.upper()}")
    print("Type:", type(model).__name__)
    print("Input features:", getattr(model, "n_features_in_", "unknown"))
    print("Classes:", getattr(model, "classes_", "not exposed"))

    feature_names = getattr(model, "feature_names_in_", None)

    if feature_names is not None:
        print("Feature names:", list(feature_names)[:30])

    if hasattr(model, "get_booster"):
        booster = model.get_booster()
        print(
            "Booster feature names:",
            (booster.feature_names or [])[:30]
        )

print("\n========== TF-IDF VECTORIZER ==========")
print("Type:", type(tfidf_vectorizer).__name__)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))
print("Max features:", tfidf_vectorizer.max_features)

print("\n========== SAVED RESULTS ==========")

results_path = model_dir / "results.json"

with open(results_path, "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print(json.dumps(saved_results, indent=2, default=str))

print("\n========== ORIGINAL INPUT CHECK ==========")
print("Company 003 example keys:", list(train_inputs[0].keys()))
print("Company 001 example keys:", list(company001_inputs[0].keys()))
print(
    "Input schemas match:",
    set(train_inputs[0]) == set(company001_inputs[0])
)


========== FROZEN MODEL INSPECTION ==========

MEANINGFUL
Type: XGBClassifier
Input features: 1434
Classes: [0 1]
Booster feature names: []

ESCALATION
Type: XGBClassifier
Input features: 1434
Classes: [0 1]
Booster feature names: []

DIRECTION
Type: XGBClassifier
Input features: 1434
Classes: [0 1 2]
Booster feature names: []

========== TF-IDF VECTORIZER ==========
Type: TfidfVectorizer
Vocabulary size: 1408
Max features: 3000

========== SAVED RESULTS ==========
{
  "detector": "xgboost_v1",
  "dataset": "company003_benchmark_v3",
  "evaluation_split": "dev",
  "train_examples": 7764,
  "dev_examples": 690,
  "test_examples_evaluated": 0,
  "feature_count": 1434,
  "meaningful_change": {
    "accuracy": 0.7028985507246377,
    "precision": 0.7848101265822784,
    "recall": 0.7209302325581395,
    "f1": 0.7515151515151515,
    "TP": 310,
    "FP": 85,
    "FN": 120,
    "TN": 175,
    "false_positive_rate": 0.3269230769230769
  },
  "escalation_needed": {
    "accuracy": 0.6289855072

In [16]:

# STEP 17: First cross-company F1 measurement — Company 001

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score
)

# Exact feature extraction from original Company 003 notebook
def extract_xgb_features(record):
    event = record["new_event"]
    old = record["old_state"]
    current = record["current_state"]
    impact = record["event_impact"]
    observation = event.get("observation", {})

    text = event.get("description", "")

    financial_keys = [
        "outstanding_ar",
        "overdue_ar",
        "overdue_invoice_count",
        "max_days_overdue",
        "recent_30d_invoice_count",
        "recent_30d_revenue",
        "previous_30d_invoice_count",
        "previous_30d_revenue",
    ]

    impact_keys = [
        "outstanding_ar_delta",
        "overdue_ar_delta",
        "recent_30d_invoice_count_delta",
        "recent_30d_revenue_delta",
    ]

    numeric = []

    for state in [old, current]:
        numeric.extend(
            float(state.get(key, 0) or 0)
            for key in financial_keys
        )

    numeric.extend(
        float(impact.get(key, 0) or 0)
        for key in impact_keys
    )

    numeric.extend([
        float(event.get("amount", 0) or 0),
        float(observation.get("revenue_change_pct", 0) or 0),
        float(observation.get("baseline_30d_revenue", 0) or 0),
        float(observation.get("observed_30d_revenue", 0) or 0),
        int(event["event_type"] == "customer_message"),
        int(event["event_type"] == "financial_trend_observed"),
    ])

    return text, numeric


def build_frozen_features(records):
    extracted = [extract_xgb_features(x) for x in records]

    texts = [text for text, numeric in extracted]
    numeric = np.asarray(
        [values for text, values in extracted],
        dtype=np.float32
    )

    X = hstack(
        [
            tfidf_vectorizer.transform(texts),
            csr_matrix(numeric)
        ],
        format="csr"
    )

    assert numeric.shape[1] == 26
    assert X.shape[1] == 1434

    return X


# First verify frozen preprocessing on Company 003 dev data.
dev_inputs_check = load_jsonl(
    benchmark_dir / "dev_inputs.jsonl"
)
dev_labels_check = load_jsonl(
    benchmark_dir / "dev_labels.jsonl"
)

X_dev_check = build_frozen_features(dev_inputs_check)

dev_meaningful_pred = meaningful_model.predict(X_dev_check)
dev_escalation_pred = escalation_model.predict(X_dev_check)

def get_f1(labels, predictions):
    return precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )[2]

dev_meaningful_f1 = get_f1(
    [int(y["meaningful_change"]) for y in dev_labels_check],
    dev_meaningful_pred
)

dev_escalation_f1 = get_f1(
    [int(y["escalation_needed"]) for y in dev_labels_check],
    dev_escalation_pred
)

print("========== FROZEN MODEL VERIFICATION ==========")
print("Company 003 dev meaningful F1:", round(dev_meaningful_f1, 4))
print("Company 003 dev escalation F1:", round(dev_escalation_f1, 4))

assert abs(dev_meaningful_f1 - 0.7515151515) < 1e-6
assert abs(dev_escalation_f1 - 0.5294117647) < 1e-6

print("PASS — Original Company 003 scores reproduced.")

# Now evaluate Company 001 without retraining.
X_company001 = build_frozen_features(company001_inputs)

company001_meaningful_pred = meaningful_model.predict(X_company001)
company001_escalation_pred = escalation_model.predict(X_company001)
company001_direction_pred = direction_model.predict(X_company001)

true_meaningful = [
    int(y["meaningful_change"]) for y in company001_labels
]
true_escalation = [
    int(y["escalation_needed"]) for y in company001_labels
]
true_direction = [
    {"negative": 0, "neutral": 1, "positive": 2}[y["direction"]]
    for y in company001_labels
]

company001_results = {
    "company": "company-001",
    "examples": len(company001_inputs),
    "meaningful_f1": get_f1(
        true_meaningful, company001_meaningful_pred
    ),
    "escalation_f1": get_f1(
        true_escalation, company001_escalation_pred
    ),
    "direction_accuracy": accuracy_score(
        true_direction, company001_direction_pred
    )
}

print("\n========== FIRST CROSS-COMPANY RESULTS ==========")
display(pd.DataFrame([company001_results]).round(4))

print("\nF1 CHANGE RELATIVE TO COMPANY 003 DEV:")
print(
    "Meaningful-change F1:",
    round(company001_results["meaningful_f1"] - dev_meaningful_f1, 4)
)
print(
    "Escalation F1:",
    round(company001_results["escalation_f1"] - dev_escalation_f1, 4)
)

print("\nPASS — Company 001 evaluated with frozen Company 003 model.")


========== FROZEN MODEL VERIFICATION ==========
Company 003 dev meaningful F1: 0.7515
Company 003 dev escalation F1: 0.5294
PASS — Original Company 003 scores reproduced.

========== FIRST CROSS-COMPANY RESULTS ==========


,company,examples,meaningful_f1,escalation_f1,direction_accuracy
0,company-001,180,1.0,1.0,1.0



F1 CHANGE RELATIVE TO COMPANY 003 DEV:
Meaningful-change F1: 0.2485
Escalation F1: 0.4706

PASS — Company 001 evaluated with frozen Company 003 model.


In [17]:

# STEP 18: Apples-to-apples financial scenario comparison

import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

financial_scenarios = {
    "revenue_collapse",
    "moderate_revenue_decline",
    "minor_revenue_dip",
    "stable_revenue",
    "small_revenue_fluctuation",
    "revenue_surge",
}

direction_ids = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

# Company 003 development examples: financial scenarios only
financial_dev_indices = [
    i for i, label in enumerate(dev_labels_check)
    if label["scenario_type"] in financial_scenarios
]

assert financial_dev_indices, "No financial dev examples found."

# Company 001 examples: same six scenario families
financial_001_indices = [
    i for i, label in enumerate(company001_labels)
    if label["scenario_type"] in financial_scenarios
]

def evaluate_subset(
    company_name, labels, indices,
    meaningful_predictions,
    escalation_predictions,
    direction_predictions
):
    selected_labels = [labels[i] for i in indices]

    y_meaningful = [
        int(y["meaningful_change"]) for y in selected_labels
    ]
    y_escalation = [
        int(y["escalation_needed"]) for y in selected_labels
    ]
    y_direction = [
        direction_ids[y["direction"]] for y in selected_labels
    ]

    return {
        "company": company_name,
        "examples": len(indices),
        "meaningful_f1": f1_score(
            y_meaningful,
            meaningful_predictions[indices],
            zero_division=0
        ),
        "escalation_f1": f1_score(
            y_escalation,
            escalation_predictions[indices],
            zero_division=0
        ),
        "direction_accuracy": accuracy_score(
            y_direction,
            direction_predictions[indices]
        )
    }

# Original Company 003 dev predictions
dev_direction_pred = direction_model.predict(X_dev_check)

comparison = pd.DataFrame([
    evaluate_subset(
        "company-003 (financial dev)",
        dev_labels_check,
        financial_dev_indices,
        dev_meaningful_pred,
        dev_escalation_pred,
        dev_direction_pred
    ),
    evaluate_subset(
        "company-001 (financial transfer)",
        company001_labels,
        financial_001_indices,
        company001_meaningful_pred,
        company001_escalation_pred,
        company001_direction_pred
    )
])

print("========== FINANCIAL-ONLY TRANSFER COMPARISON ==========")
display(comparison.round(4))

print("\n========== SCENARIO COUNTS ==========")
print("Company 003:")
print(pd.Series([
    dev_labels_check[i]["scenario_type"]
    for i in financial_dev_indices
]).value_counts().sort_index())

print("\nCompany 001:")
print(pd.Series([
    company001_labels[i]["scenario_type"]
    for i in financial_001_indices
]).value_counts().sort_index())

print("\nPASS — Like-for-like financial comparison completed.")


========== FINANCIAL-ONLY TRANSFER COMPARISON ==========


,company,examples,meaningful_f1,escalation_f1,direction_accuracy
0,company-003 (financial dev),78,0.987,0.9804,0.9872
1,company-001 (financial transfer),180,1.000,1.0000,1.0000



========== SCENARIO COUNTS ==========
Company 003:
minor_revenue_dip            13
moderate_revenue_decline     13
revenue_collapse             13
revenue_surge                13
small_revenue_fluctuation    13
stable_revenue               13
Name: count, dtype: int64

Company 001:
minor_revenue_dip            30
moderate_revenue_decline     30
revenue_collapse             30
revenue_surge                30
small_revenue_fluctuation    30
stable_revenue               30
Name: count, dtype: int64

PASS — Like-for-like financial comparison completed.


In [18]:

# STEP 19: Reusable frozen-model cross-company evaluator

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

DIRECTION_TO_ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

def evaluate_frozen_company(company_id, inputs, labels):
    """
    Evaluate one company using the frozen Company 003 XGBoost models.

    Requires:
        build_frozen_features
        meaningful_model
        escalation_model
        direction_model
    """
    assert len(inputs) == len(labels)
    assert len(inputs) > 0

    X = build_frozen_features(inputs)

    meaningful_pred = meaningful_model.predict(X)
    escalation_pred = escalation_model.predict(X)
    direction_pred = direction_model.predict(X)

    meaningful_true = np.array([
        int(label["meaningful_change"])
        for label in labels
    ])

    escalation_true = np.array([
        int(label["escalation_needed"])
        for label in labels
    ])

    direction_true = np.array([
        DIRECTION_TO_ID[label["direction"]]
        for label in labels
    ])

    return {
        "company": company_id,
        "examples": len(inputs),
        "customers": len(set(
            label["customer_id"] for label in labels
        )),
        "meaningful_f1": f1_score(
            meaningful_true,
            meaningful_pred,
            zero_division=0
        ),
        "escalation_f1": f1_score(
            escalation_true,
            escalation_pred,
            zero_division=0
        ),
        "direction_accuracy": accuracy_score(
            direction_true,
            direction_pred
        ),
        "meaningful_errors": int(
            np.sum(meaningful_true != meaningful_pred)
        ),
        "escalation_errors": int(
            np.sum(escalation_true != escalation_pred)
        ),
        "direction_errors": int(
            np.sum(direction_true != direction_pred)
        )
    }


# Validate reusable evaluator on Company 001
company001_repeat = evaluate_frozen_company(
    "company-001",
    company001_inputs,
    company001_labels
)

print("========== REUSABLE EVALUATOR CHECK ==========")
display(pd.DataFrame([company001_repeat]).round(4))

assert company001_repeat["examples"] == 180
assert company001_repeat["customers"] == 30
assert company001_repeat["meaningful_f1"] == 1.0
assert company001_repeat["escalation_f1"] == 1.0
assert company001_repeat["direction_accuracy"] == 1.0

print("\nPASS — Reusable frozen-model evaluator verified.")


========== REUSABLE EVALUATOR CHECK ==========


,company,examples,customers,meaningful_f1,escalation_f1,direction_accuracy,meaningful_errors,escalation_errors,direction_errors
0,company-001,180,30,1.0,1.0,1.0,0,0,0



PASS — Reusable frozen-model evaluator verified.


In [19]:

# STEP 20: Inspect reusable company-generation components

import inspect
from pathlib import Path

print("========== EXISTING DATA HELPERS ==========")

for name in [
    "find_company_data_path",
    "load_yaml_file",
]:
    obj = globals().get(name)

    if obj is None:
        print(f"\n{name}: NOT FOUND")
        continue

    print(f"\n{name}")
    print("Signature:", inspect.signature(obj))

    try:
        print(inspect.getsource(obj)[:5000])
    except (OSError, TypeError):
        print("Source unavailable in this notebook session.")

print("\n========== COMPANY 001 OBJECT STRUCTURE ==========")

for name in [
    "customer_baselines",
    "financial_states",
    "company001_financial_df",
    "company001_inputs",
    "company001_labels",
]:
    obj = globals().get(name)

    print(f"\n{name}: {type(obj).__name__}")

    if isinstance(obj, dict):
        print("Entries:", len(obj))
        if obj:
            key = next(iter(obj))
            print("First key:", repr(key))
            print("First value:", repr(obj[key])[:1200])

    elif isinstance(obj, pd.DataFrame):
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))
        print(obj.head(2).to_string(index=False)[:1500])

    elif isinstance(obj, list):
        print("Entries:", len(obj))
        if obj:
            print("First item:", repr(obj[0])[:1200])

print("\n========== COMPANY DIRECTORIES ==========")
print("Total:", len(company_dirs))
print("First three:", [str(p) for p in company_dirs[:3]])

print("\n========== DATE VARIABLES ==========")
print("cutoff:", repr(cutoff))
print("as_of:", repr(as_of))

print("\nPASS — Generator components inspected.")


========== EXISTING DATA HELPERS ==========

find_company_data_path
Signature: (company_dir)
def find_company_data_path(company_dir):
    """Locate the world directory without hardcoding its ID."""
    world_dirs = sorted(
        p for p in (company_dir / "rwm-v4").glob("world-*")
        if p.is_dir()
    )

    if len(world_dirs) != 1:
        raise ValueError(
            f"{company_dir.name}: expected one world directory, "
            f"found {len(world_dirs)}"
        )

    return world_dirs[0]


load_yaml_file
Signature: (path)
def load_yaml_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


========== COMPANY 001 OBJECT STRUCTURE ==========

customer_baselines: DataFrame
Shape: (30, 3)
Columns: ['customer_id', 'baseline_revenue', 'baseline_invoice_count']
     customer_id  baseline_revenue  baseline_invoice_count
 cust-customer-1           3150.31                       5
cust-customer-11           2692.67                       5

f

In [20]:

# STEP 21: Recover original Company 001 generation code

from IPython import get_ipython

ip = get_ipython()
history = list(ip.history_manager.get_range(output=False))

keywords = [
    "customer_baselines =",
    "financial_states =",
    "company001_financial_df =",
    "company001_inputs =",
    "company001_labels =",
    "payments_df_target =",
    "historical_invoices =",
]

print("========== COMPANY 001 GENERATOR HISTORY ==========")

found = []

for session, line_number, source in history:
    matched = [
        keyword for keyword in keywords
        if keyword in source
    ]

    if matched:
        found.append((line_number, matched, source))

print("Matching cells:", len(found))

for line_number, matched, source in found:
    print("\n" + "=" * 70)
    print("INPUT CELL:", line_number)
    print("MATCHED:", matched)
    print("SOURCE:")
    print(source[:12000])

print("\n========== COMPLETENESS CHECK ==========")

for keyword in keywords:
    present = any(
        keyword in source
        for _, _, source in found
    )
    print(f"{keyword:30s}: {'FOUND' if present else 'NOT FOUND'}")


========== COMPANY 001 GENERATOR HISTORY ==========
Matching cells: 5

INPUT CELL: 10
MATCHED: ['customer_baselines =', 'company001_financial_df =']
SOURCE:

# STEP 10: Generate held-out financial scenarios for Company 001

import copy
import numpy as np
import pandas as pd

TARGET_COMPANY = "company-001"
target_dir = find_company_data_path(
    DATASET_ROOT / TARGET_COMPANY
)

invoices = load_yaml_file(target_dir / "invoices.yaml")
invoices_df_target = pd.DataFrame(invoices)

print("Company:", TARGET_COMPANY)
print("Invoices:", len(invoices_df_target))

# Use a fixed historical cutoff; adjust only if outside
# the company's observed invoice history.
invoices_df_target["txn_date"] = pd.to_datetime(
    invoices_df_target["txn_date"]
)
invoices_df_target["total_amount"] = pd.to_numeric(
    invoices_df_target["total_amount"],
    errors="coerce"
)

cutoff = invoices_df_target["txn_date"].max()

baseline_start = cutoff - pd.Timedelta(days=60)
baseline_end = cutoff - pd.Timedelta(days=30)

In [21]:

# STEP 22: Reusable financial scenario generator

import copy
from collections import defaultdict
from decimal import Decimal
import numpy as np
import pandas as pd

SCENARIO_CHANGES = {
    "revenue_collapse": -73.92,
    "moderate_revenue_decline": -28.48,
    "minor_revenue_dip": -6.04,
    "stable_revenue": 0.12,
    "small_revenue_fluctuation": 6.88,
    "revenue_surge": 126.72,
}

SCENARIO_LABELS = {
    name: {
        "meaningful_change": bool(values["meaningful_change"]),
        "direction": values["direction"],
        "escalation_needed": bool(values["escalation_needed"]),
    }
    for name, values in scenario_labels.items()
}

TREND_DESCRIPTION = (
    "Hypothetical purchasing-trend report comparing "
    "a historical 30-day baseline with a simulated "
    "subsequent 30-day period. The QuickBooks financial "
    "states represent historical accounting records "
    "and are not updated by this report."
)

def money_decimal(value):
    if value is None or pd.isna(value):
        return Decimal("0")
    return Decimal(str(value))


def generate_company_financial_examples(company_dir):
    company_dir = Path(company_dir)
    company_id = company_dir.name
    world_dir = find_company_data_path(company_dir)

    invoices = load_yaml_file(world_dir / "invoices.yaml") or []
    payments = load_yaml_file(world_dir / "customer_payments.yaml") or []

    if not invoices:
        return [], []

    inv = pd.DataFrame(invoices)
    inv["txn_date"] = pd.to_datetime(inv["txn_date"])
    inv["due_date"] = pd.to_datetime(inv["due_date"])
    inv["total_amount"] = pd.to_numeric(
        inv["total_amount"], errors="raise"
    )

    as_of = inv["txn_date"].max().normalize()
    baseline_start = as_of - pd.Timedelta(days=60)
    baseline_end = as_of - pd.Timedelta(days=30)
    recent_start = baseline_end

    baseline_invoices = inv[
        (inv["txn_date"] >= baseline_start)
        & (inv["txn_date"] < baseline_end)
    ]

    baselines = (
        baseline_invoices.groupby("customer_id")
        .agg(
            baseline_revenue=("total_amount", "sum"),
            baseline_invoice_count=("id", "count")
        )
        .reset_index()
    )

    baselines = baselines[
        (baselines["baseline_revenue"] > 0)
        & (baselines["baseline_invoice_count"] >= 2)
    ]

    if baselines.empty:
        return [], []

    invoice_ids = set(inv["id"])
    paid_by_invoice = defaultdict(lambda: Decimal("0"))

    for payment in payments:
        if pd.Timestamp(payment["txn_date"]) > as_of:
            continue

        for application in payment.get("applications", []) or []:
            document_id = application.get("document_id")

            if document_id in invoice_ids:
                paid_by_invoice[document_id] += money_decimal(
                    application.get("amount", 0)
                )

    inputs = []
    labels = []

    for _, customer in baselines.iterrows():
        customer_id = customer["customer_id"]

        history = inv[
            (inv["customer_id"] == customer_id)
            & (inv["txn_date"] <= as_of)
        ].sort_values(["txn_date", "id"])

        recent = history[
            (history["txn_date"] >= recent_start)
            & (history["txn_date"] <= as_of)
        ]

        previous = history[
            (history["txn_date"] >= baseline_start)
            & (history["txn_date"] < baseline_end)
        ]

        outstanding = Decimal("0")
        overdue = Decimal("0")
        overdue_count = 0
        max_days_overdue = 0

        for _, invoice in history.iterrows():
            balance = max(
                money_decimal(invoice["total_amount"])
                - paid_by_invoice[invoice["id"]],
                Decimal("0")
            )

            outstanding += balance

            if balance > 0 and invoice["due_date"] < as_of:
                overdue += balance
                overdue_count += 1
                max_days_overdue = max(
                    max_days_overdue,
                    (as_of - invoice["due_date"]).days
                )

        state = {
            "outstanding_ar": round(float(outstanding), 2),
            "overdue_ar": round(float(overdue), 2),
            "overdue_invoice_count": overdue_count,
            "max_days_overdue": max_days_overdue,
            "recent_30d_invoice_count": int(len(recent)),
            "previous_30d_invoice_count": int(len(previous)),
            "recent_30d_revenue": round(
                float(sum(
                    (money_decimal(v) for v in recent["total_amount"]),
                    Decimal("0")
                )), 2
            ),
            "previous_30d_revenue": round(
                float(sum(
                    (money_decimal(v) for v in previous["total_amount"]),
                    Decimal("0")
                )), 2
            ),
        }

        baseline_revenue = round(
            float(customer["baseline_revenue"]), 2
        )
        baseline_count = int(customer["baseline_invoice_count"])

        assert abs(
            state["previous_30d_revenue"] - baseline_revenue
        ) <= 0.01
        assert state["previous_30d_invoice_count"] == baseline_count

        latest_invoice = history.iloc[-1]

        previous_event = {
            "event_type": "invoice_created",
            "event_date": latest_invoice["txn_date"].strftime("%Y-%m-%d"),
            "amount": round(float(latest_invoice["total_amount"]), 2),
            "description": str(latest_invoice.get("memo") or ""),
        }

        for scenario, pct in SCENARIO_CHANGES.items():
            observed_revenue = round(
                baseline_revenue * (1 + pct / 100), 2
            )

            record = {
                "event_date": as_of.strftime("%Y-%m-%d"),
                "old_state": copy.deepcopy(state),
                "previous_event": copy.deepcopy(previous_event),
                "new_event": {
                    "event_type": "financial_trend_observed",
                    "event_date": as_of.strftime("%Y-%m-%d"),
                    "amount": 0.0,
                    "description": TREND_DESCRIPTION,
                    "observation": {
                        "baseline_30d_revenue": baseline_revenue,
                        "observed_30d_revenue": observed_revenue,
                        "revenue_change_pct": pct,
                        "baseline_30d_invoice_count": baseline_count,
                        "report_type": "hypothetical_customer_purchasing_trend",
                        "baseline_period": "historical_previous_30_day_window",
                        "observed_period": "simulated_following_30_day_window",
                    },
                },
                "current_state": copy.deepcopy(state),
                "event_impact": {
                    "outstanding_ar_delta": 0.0,
                    "overdue_ar_delta": 0.0,
                    "recent_30d_invoice_count_delta": 0.0,
                    "recent_30d_revenue_delta": 0.0,
                },
            }

            label = {
                "event_id": (
                    f"cross-{company_id}-fin-{len(inputs):04d}"
                ),
                "company_id": company_id,
                "customer_id": customer_id,
                "scenario_type": scenario,
                **SCENARIO_LABELS[scenario],
            }

            inputs.append(record)
            labels.append(label)

    assert len(inputs) == len(labels)
    assert len(inputs) == len(baselines) * 6

    return inputs, labels


# Validate against the original Company 001 examples.
recreated_inputs, recreated_labels = (
    generate_company_financial_examples(
        DATASET_ROOT / "company-001"
    )
)

assert len(recreated_inputs) == len(company001_inputs) == 180
assert len(recreated_labels) == len(company001_labels) == 180

# Event IDs intentionally differ; all model features must match.
original_features = build_frozen_features(company001_inputs)
recreated_features = build_frozen_features(recreated_inputs)

assert (original_features != recreated_features).nnz == 0, (
    "Regenerated Company 001 features differ from original."
)

recreated_results = evaluate_frozen_company(
    "company-001",
    recreated_inputs,
    recreated_labels
)

assert recreated_results["meaningful_f1"] == 1.0
assert recreated_results["escalation_f1"] == 1.0
assert recreated_results["direction_accuracy"] == 1.0

print("========== GENERATOR VALIDATION ==========")
print("Regenerated examples:", len(recreated_inputs))
print("Eligible customers:", len(recreated_inputs) // 6)
print("Feature vectors match original:", True)
print("Frozen-model results match original:", True)

display(pd.DataFrame([recreated_results]).round(4))

print("\nPASS — Cross-company generator validated.")


========== GENERATOR VALIDATION ==========
Regenerated examples: 180
Eligible customers: 30
Feature vectors match original: True
Frozen-model results match original: True


,company,examples,customers,meaningful_f1,escalation_f1,direction_accuracy,meaningful_errors,escalation_errors,direction_errors
0,company-001,180,30,1.0,1.0,1.0,0,0,0



PASS — Cross-company generator validated.


In [22]:

# STEP 23: Full cross-company frozen-model evaluation

import pandas as pd
import numpy as np
from pathlib import Path

all_company_results = []
company_failures = []

for company_dir in sorted(company_dirs, key=lambda p: p.name):
    company_id = company_dir.name

    try:
        inputs, labels = generate_company_financial_examples(company_dir)

        if not inputs:
            company_failures.append({
                "company": company_id,
                "reason": "No eligible customers",
            })
            continue

        result = evaluate_frozen_company(
            company_id, inputs, labels
        )

        all_company_results.append(result)

    except Exception as exc:
        company_failures.append({
            "company": company_id,
            "reason": f"{type(exc).__name__}: {exc}",
        })

company_results_df = (
    pd.DataFrame(all_company_results)
    .sort_values("company")
    .reset_index(drop=True)
)

assert not company_results_df.empty, "No companies evaluated."

# Company 003 was the model's training company.
# Exclude it from held-out cross-company statistics.
heldout_results_df = company_results_df[
    company_results_df["company"] != "company-003"
].copy()

assert not heldout_results_df.empty, "No held-out companies evaluated."

metrics = [
    "meaningful_f1",
    "escalation_f1",
    "direction_accuracy",
]

summary_rows = []

for metric in metrics:
    values = heldout_results_df[metric]

    summary_rows.append({
        "metric": metric,
        "companies": len(values),
        "mean": values.mean(),
        "std": values.std(ddof=0),
        "minimum": values.min(),
        "median": values.median(),
        "maximum": values.max(),
    })

cross_company_summary = pd.DataFrame(summary_rows)

print("========== CROSS-COMPANY COVERAGE ==========")
print("Company directories:", len(company_dirs))
print("Successfully evaluated:", len(company_results_df))
print("Held-out companies:", len(heldout_results_df))
print("Skipped / failed:", len(company_failures))
print("Held-out examples:", int(heldout_results_df["examples"].sum()))
print("Held-out customers:", int(heldout_results_df["customers"].sum()))

print("\n========== HELD-OUT PERFORMANCE SUMMARY ==========")
display(cross_company_summary.round(4))

print("\n========== LOWEST-PERFORMING HELD-OUT COMPANIES ==========")
display(
    heldout_results_df.sort_values(
        ["meaningful_f1", "escalation_f1", "company"]
    ).head(15).round(4)
)

print("\n========== COMPANY-LEVEL RESULTS ==========")
display(heldout_results_df.round(4))

print("\n========== FAILURES / SKIPS ==========")
if company_failures:
    display(pd.DataFrame(company_failures))
else:
    print("None")

# Persist reproducible evaluation summaries.
output_dir = Path("cross_company_benchmark")
output_dir.mkdir(parents=True, exist_ok=True)

company_results_df.to_csv(
    output_dir / "all_company_frozen_xgb_results.csv",
    index=False
)

heldout_results_df.to_csv(
    output_dir / "heldout_company_frozen_xgb_results.csv",
    index=False
)

cross_company_summary.to_csv(
    output_dir / "heldout_frozen_xgb_summary.csv",
    index=False
)

print("\nSaved evaluation CSVs to:", output_dir)
print("PASS — Cross-company evaluation completed.")


========== CROSS-COMPANY COVERAGE ==========
Company directories: 100
Successfully evaluated: 40
Held-out companies: 39
Skipped / failed: 60
Held-out examples: 8670
Held-out customers: 1445

========== HELD-OUT PERFORMANCE SUMMARY ==========


,metric,companies,mean,std,minimum,median,maximum
0,meaningful_f1,39,1.0,0.0,1.0,1.0,1.0
1,escalation_f1,39,1.0,0.0,1.0,1.0,1.0
2,direction_accuracy,39,1.0,0.0,1.0,1.0,1.0



========== LOWEST-PERFORMING HELD-OUT COMPANIES ==========


,company,examples,customers,meaningful_f1,escalation_f1,direction_accuracy,meaningful_errors,escalation_errors,direction_errors
0,company-001,180,30,1.0,1.0,1.0,0,0,0
1,company-002,336,56,1.0,1.0,1.0,0,0,0
3,company-004,252,42,1.0,1.0,1.0,0,0,0
4,company-005,360,60,1.0,1.0,1.0,0,0,0
5,company-006,708,118,1.0,1.0,1.0,0,0,0
6,company-007,180,30,1.0,1.0,1.0,0,0,0
7,company-008,360,60,1.0,1.0,1.0,0,0,0
8,company-009,594,99,1.0,1.0,1.0,0,0,0
9,company-010,180,30,1.0,1.0,1.0,0,0,0
10,company-011,348,58,1.0,1.0,1.0,0,0,0



========== COMPANY-LEVEL RESULTS ==========


,company,examples,customers,meaningful_f1,escalation_f1,direction_accuracy,meaningful_errors,escalation_errors,direction_errors
0,company-001,180,30,1.0,1.0,1.0,0,0,0
1,company-002,336,56,1.0,1.0,1.0,0,0,0
3,company-004,252,42,1.0,1.0,1.0,0,0,0
4,company-005,360,60,1.0,1.0,1.0,0,0,0
5,company-006,708,118,1.0,1.0,1.0,0,0,0
6,company-007,180,30,1.0,1.0,1.0,0,0,0
7,company-008,360,60,1.0,1.0,1.0,0,0,0
8,company-009,594,99,1.0,1.0,1.0,0,0,0
9,company-010,180,30,1.0,1.0,1.0,0,0,0
10,company-011,348,58,1.0,1.0,1.0,0,0,0



========== FAILURES / SKIPS ==========


,company,reason
0,company-022,No eligible customers
1,company-023,No eligible customers
2,company-025,No eligible customers
3,company-026,No eligible customers
4,company-027,No eligible customers
5,company-029,No eligible customers
6,company-030,No eligible customers
7,company-031,No eligible customers
8,company-032,No eligible customers
9,company-033,No eligible customers



Saved evaluation CSVs to: cross_company_benchmark
PASS — Cross-company evaluation completed.


In [23]:

# STEP 24: Revenue-change sensitivity diagnostic
# No retraining and no new ground-truth labels.

import copy
import numpy as np
import pandas as pd

TEST_PERCENTAGES = [
    -95, -80, -74, -60, -40, -30, -20, -10, -6,
    -2, 0, 2, 7, 15, 30, 60, 100, 127, 175, 250
]

# Use Company 001 as a controlled first diagnostic.
# One original customer record is sufficient because we
# vary only the simulated purchasing-trend observation.
base_records = [
    record for record, label in zip(
        company001_inputs, company001_labels
    )
    if label["scenario_type"] == "stable_revenue"
]

assert len(base_records) == 30

sensitivity_inputs = []
sensitivity_metadata = []

for customer_idx, base_record in enumerate(base_records):
    baseline = base_record["new_event"]["observation"][
        "baseline_30d_revenue"
    ]

    for pct in TEST_PERCENTAGES:
        record = copy.deepcopy(base_record)

        record["new_event"]["observation"][
            "revenue_change_pct"
        ] = float(pct)

        record["new_event"]["observation"][
            "observed_30d_revenue"
        ] = round(baseline * (1 + pct / 100), 2)

        sensitivity_inputs.append(record)

        sensitivity_metadata.append({
            "customer_idx": customer_idx,
            "revenue_change_pct": pct,
        })

X_sensitivity = build_frozen_features(sensitivity_inputs)

sensitivity_df = pd.DataFrame(sensitivity_metadata)

sensitivity_df["meaningful_pred"] = (
    meaningful_model.predict(X_sensitivity)
)

sensitivity_df["escalation_pred"] = (
    escalation_model.predict(X_sensitivity)
)

sensitivity_df["direction_pred"] = (
    direction_model.predict(X_sensitivity)
)

sensitivity_summary = (
    sensitivity_df
    .groupby("revenue_change_pct")
    .agg(
        customers=("customer_idx", "nunique"),
        meaningful_rate=("meaningful_pred", "mean"),
        escalation_rate=("escalation_pred", "mean"),
        negative_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["negative"])
        ),
        neutral_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["neutral"])
        ),
        positive_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["positive"])
        ),
    )
    .reset_index()
)

print("========== REVENUE SENSITIVITY ==========")
print("Customers:", len(base_records))
print("Percentage values:", len(TEST_PERCENTAGES))
print("Predictions:", len(sensitivity_df))

display(sensitivity_summary.round(3))

print("\n========== PREDICTION VARIABILITY ==========")
for column in [
    "meaningful_pred",
    "escalation_pred",
    "direction_pred",
]:
    print(
        column,
        "unique predicted classes:",
        sorted(sensitivity_df[column].unique().tolist())
    )

print("\nPASS — Sensitivity diagnostic completed.")


========== REVENUE SENSITIVITY ==========
Customers: 30
Percentage values: 20
Predictions: 600


,revenue_change_pct,customers,meaningful_rate,escalation_rate,negative_rate,neutral_rate,positive_rate
0,-95,30,1.000,1.000,1.000,0.000,0.0
1,-80,30,1.000,1.000,1.000,0.000,0.0
2,-74,30,1.000,1.000,1.000,0.000,0.0
3,-60,30,1.000,0.433,1.000,0.000,0.0
4,-40,30,1.000,0.167,1.000,0.000,0.0
5,-30,30,1.000,0.000,1.000,0.000,0.0
6,-20,30,0.467,0.000,0.500,0.500,0.0
7,-10,30,0.467,0.000,0.433,0.567,0.0
8,-6,30,0.000,0.000,0.000,1.000,0.0
9,-2,30,0.000,0.000,0.000,1.000,0.0



========== PREDICTION VARIABILITY ==========
meaningful_pred unique predicted classes: [0, 1]
escalation_pred unique predicted classes: [0, 1]
direction_pred unique predicted classes: [0, 1, 2]

PASS — Sensitivity diagnostic completed.


In [24]:

# STEP 25: Inspect how predictions vary with revenue changes

import pandas as pd
import numpy as np

print("========== REVENUE CHANGE RESPONSE ==========")

display(
    sensitivity_summary[
        [
            "revenue_change_pct",
            "customers",
            "meaningful_rate",
            "escalation_rate",
            "negative_rate",
            "neutral_rate",
            "positive_rate",
        ]
    ].round(3)
)

print("\n========== CUSTOMER-LEVEL CONSISTENCY ==========")

consistency = (
    sensitivity_df
    .groupby("revenue_change_pct")
    .agg(
        meaningful_unique=("meaningful_pred", "nunique"),
        escalation_unique=("escalation_pred", "nunique"),
        direction_unique=("direction_pred", "nunique"),
    )
    .reset_index()
)

display(consistency)

print("\n========== PREDICTIONS AT KEY PERCENTAGES ==========")

key_percentages = [-95, -40, -20, -6, 0, 7, 30, 100, 250]

for pct in key_percentages:
    subset = sensitivity_df[
        sensitivity_df["revenue_change_pct"] == pct
    ]

    print(
        f"{pct:+4d}% | "
        f"meaningful={subset['meaningful_pred'].mean():.2f} | "
        f"escalation={subset['escalation_pred'].mean():.2f} | "
        f"direction counts="
        f"{subset['direction_pred'].value_counts().sort_index().to_dict()}"
    )

print("\nPASS — Revenue-response behavior inspected.")


========== REVENUE CHANGE RESPONSE ==========


,revenue_change_pct,customers,meaningful_rate,escalation_rate,negative_rate,neutral_rate,positive_rate
0,-95,30,1.000,1.000,1.000,0.000,0.0
1,-80,30,1.000,1.000,1.000,0.000,0.0
2,-74,30,1.000,1.000,1.000,0.000,0.0
3,-60,30,1.000,0.433,1.000,0.000,0.0
4,-40,30,1.000,0.167,1.000,0.000,0.0
5,-30,30,1.000,0.000,1.000,0.000,0.0
6,-20,30,0.467,0.000,0.500,0.500,0.0
7,-10,30,0.467,0.000,0.433,0.567,0.0
8,-6,30,0.000,0.000,0.000,1.000,0.0
9,-2,30,0.000,0.000,0.000,1.000,0.0



========== CUSTOMER-LEVEL CONSISTENCY ==========


,revenue_change_pct,meaningful_unique,escalation_unique,direction_unique
0,-95,1,1,1
1,-80,1,1,1
2,-74,1,1,1
3,-60,1,2,1
4,-40,1,2,1
5,-30,1,1,1
6,-20,2,1,2
7,-10,2,1,2
8,-6,1,1,1
9,-2,1,1,1



========== PREDICTIONS AT KEY PERCENTAGES ==========
 -95% | meaningful=1.00 | escalation=1.00 | direction counts={0: 30}
 -40% | meaningful=1.00 | escalation=0.17 | direction counts={0: 30}
 -20% | meaningful=0.47 | escalation=0.00 | direction counts={0: 15, 1: 15}
  -6% | meaningful=0.00 | escalation=0.00 | direction counts={1: 30}
  +0% | meaningful=1.00 | escalation=1.00 | direction counts={0: 30}
  +7% | meaningful=0.00 | escalation=0.00 | direction counts={1: 30}
 +30% | meaningful=0.00 | escalation=0.00 | direction counts={1: 30}
+100% | meaningful=1.00 | escalation=1.00 | direction counts={2: 30}
+250% | meaningful=1.00 | escalation=1.00 | direction counts={2: 30}

PASS — Revenue-response behavior inspected.


In [25]:

# STEP 26: Investigate the exact-zero anomaly

import copy
import numpy as np
import pandas as pd

ZERO_PROBE_PERCENTAGES = [
    -1.0, -0.5, -0.1, -0.01, -0.001,
    0.0,
    0.001, 0.01, 0.1, 0.5, 1.0
]

probe_inputs = []
probe_metadata = []

for customer_idx, base_record in enumerate(base_records):
    baseline = base_record["new_event"]["observation"][
        "baseline_30d_revenue"
    ]

    for pct in ZERO_PROBE_PERCENTAGES:
        record = copy.deepcopy(base_record)

        record["new_event"]["observation"][
            "revenue_change_pct"
        ] = float(pct)

        record["new_event"]["observation"][
            "observed_30d_revenue"
        ] = round(baseline * (1 + pct / 100), 2)

        probe_inputs.append(record)

        probe_metadata.append({
            "customer_idx": customer_idx,
            "revenue_change_pct": pct,
        })

X_zero_probe = build_frozen_features(probe_inputs)

zero_probe_df = pd.DataFrame(probe_metadata)

zero_probe_df["meaningful_pred"] = (
    meaningful_model.predict(X_zero_probe)
)

zero_probe_df["escalation_pred"] = (
    escalation_model.predict(X_zero_probe)
)

zero_probe_df["direction_pred"] = (
    direction_model.predict(X_zero_probe)
)

zero_probe_summary = (
    zero_probe_df
    .groupby("revenue_change_pct")
    .agg(
        customers=("customer_idx", "nunique"),
        meaningful_rate=("meaningful_pred", "mean"),
        escalation_rate=("escalation_pred", "mean"),
        negative_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["negative"])
        ),
        neutral_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["neutral"])
        ),
        positive_rate=(
            "direction_pred",
            lambda x: np.mean(x == DIRECTION_TO_ID["positive"])
        ),
    )
    .reset_index()
)

print("========== ZERO-CHANGE ANOMALY PROBE ==========")
display(zero_probe_summary.round(3))

print("\n========== EXACT-ZERO PREDICTIONS ==========")
display(
    zero_probe_df[
        zero_probe_df["revenue_change_pct"] == 0.0
    ][
        [
            "customer_idx",
            "meaningful_pred",
            "escalation_pred",
            "direction_pred",
        ]
    ].head(10)
)

print("\nPASS — Zero-change anomaly investigated.")


========== ZERO-CHANGE ANOMALY PROBE ==========


,revenue_change_pct,customers,meaningful_rate,escalation_rate,negative_rate,neutral_rate,positive_rate
0,-1.000,30,0.0,0.0,0.0,1.0,0.0
1,-0.500,30,0.0,0.0,0.0,1.0,0.0
2,-0.100,30,0.0,0.0,0.0,1.0,0.0
3,-0.010,30,0.0,0.0,0.0,1.0,0.0
4,-0.001,30,0.0,0.0,0.0,1.0,0.0
5,0.000,30,1.0,1.0,1.0,0.0,0.0
6,0.001,30,0.0,0.0,0.0,1.0,0.0
7,0.010,30,0.0,0.0,0.0,1.0,0.0
8,0.100,30,0.0,0.0,0.0,1.0,0.0
9,0.500,30,0.0,0.0,0.0,1.0,0.0



========== EXACT-ZERO PREDICTIONS ==========


,customer_idx,meaningful_pred,escalation_pred,direction_pred
5,0,1,1,0
16,1,1,1,0
27,2,1,1,0
38,3,1,1,0
49,4,1,1,0
60,5,1,1,0
71,6,1,1,0
82,7,1,1,0
93,8,1,1,0
104,9,1,1,0



PASS — Zero-change anomaly investigated.


In [26]:

# STEP 27: Test sparse-zero handling in frozen XGBoost models

import numpy as np
import pandas as pd

zero_indices = np.flatnonzero(
    zero_probe_df["revenue_change_pct"].to_numpy() == 0.0
)

near_indices = np.flatnonzero(
    zero_probe_df["revenue_change_pct"].to_numpy() == 0.001
)

assert len(zero_indices) == 30
assert len(near_indices) == 30

def compare_sparse_dense(indices, label):
    X_sparse = X_zero_probe[indices]
    X_dense = X_sparse.toarray()

    rows = []

    for task, model in [
        ("meaningful", meaningful_model),
        ("escalation", escalation_model),
        ("direction", direction_model),
    ]:
        sparse_pred = model.predict(X_sparse)
        dense_pred = model.predict(X_dense)

        rows.append({
            "input": label,
            "task": task,
            "sparse_predictions": dict(
                zip(
                    *np.unique(
                        sparse_pred,
                        return_counts=True
                    )
                )
            ),
            "dense_predictions": dict(
                zip(
                    *np.unique(
                        dense_pred,
                        return_counts=True
                    )
                )
            ),
            "changed_predictions": int(
                np.sum(sparse_pred != dense_pred)
            ),
        })

    return rows

representation_results = pd.DataFrame(
    compare_sparse_dense(zero_indices, "exact_zero")
    + compare_sparse_dense(near_indices, "positive_0.001")
)

print("========== SPARSE VS DENSE PREDICTIONS ==========")
display(representation_results)

print("\n========== FEATURE MATRIX CHECK ==========")
print("Matrix type:", type(X_zero_probe).__name__)
print("Feature count:", X_zero_probe.shape[1])
print("Sparse stored entries:", X_zero_probe.nnz)

print("\n========== INTERPRETATION ==========")

zero_rows = representation_results[
    representation_results["input"] == "exact_zero"
]

if (zero_rows["changed_predictions"] > 0).any():
    print(
        "Sparse and dense predictions differ at zero. "
        "Input representation is contributing to the anomaly."
    )
else:
    print(
        "Sparse and dense predictions agree at zero. "
        "The anomaly is not explained by changing "
        "input representation alone."
    )

print("\nPASS — Sparse-zero diagnostic completed.")


========== SPARSE VS DENSE PREDICTIONS ==========


,input,task,sparse_predictions,dense_predictions,changed_predictions
0,exact_zero,meaningful,{1: 30},{0: 30},30
1,exact_zero,escalation,{1: 30},{0: 30},30
2,exact_zero,direction,{0: 30},{1: 30},30
3,positive_0.001,meaningful,{0: 30},{0: 30},0
4,positive_0.001,escalation,{0: 30},{0: 30},0
5,positive_0.001,direction,{1: 30},{1: 30},0



========== FEATURE MATRIX CHECK ==========
Matrix type: csr_matrix
Feature count: 1434
Sparse stored entries: 23642

========== INTERPRETATION ==========
Sparse and dense predictions differ at zero. Input representation is contributing to the anomaly.

PASS — Sparse-zero diagnostic completed.


In [27]:

# STEP 28: Audit dense vs sparse inference on original benchmarks

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

def compare_inference_formats(company, X, labels):
    X_dense = X.toarray()

    tasks = [
        (
            "meaningful",
            meaningful_model,
            np.array([int(y["meaningful_change"]) for y in labels]),
            "f1",
        ),
        (
            "escalation",
            escalation_model,
            np.array([int(y["escalation_needed"]) for y in labels]),
            "f1",
        ),
        (
            "direction",
            direction_model,
            np.array([DIRECTION_TO_ID[y["direction"]] for y in labels]),
            "accuracy",
        ),
    ]

    rows = []

    for task, model, y_true, metric in tasks:
        sparse_pred = model.predict(X)
        dense_pred = model.predict(X_dense)

        score_fn = (
            lambda y, p: f1_score(y, p, zero_division=0)
            if metric == "f1"
            else accuracy_score(y, p)
        )

        rows.append({
            "company": company,
            "task": task,
            "examples": len(labels),
            "sparse_score": score_fn(y_true, sparse_pred),
            "dense_score": score_fn(y_true, dense_pred),
            "changed_predictions": int(
                np.sum(sparse_pred != dense_pred)
            ),
            "sparse_errors": int(np.sum(y_true != sparse_pred)),
            "dense_errors": int(np.sum(y_true != dense_pred)),
        })

    return rows


format_audit = pd.DataFrame(
    compare_inference_formats(
        "company-003-dev",
        X_dev_check,
        dev_labels_check,
    )
    + compare_inference_formats(
        "company-001",
        X_company001,
        company001_labels,
    )
)

print("========== ORIGINAL BENCHMARK FORMAT AUDIT ==========")
display(format_audit.round(4))

print("\n========== SCORE CHANGES ==========")
format_audit["score_delta"] = (
    format_audit["dense_score"]
    - format_audit["sparse_score"]
)

display(
    format_audit[
        [
            "company",
            "task",
            "score_delta",
            "changed_predictions",
        ]
    ].round(4)
)

print("\nPASS — Original benchmark representation audit completed.")


========== ORIGINAL BENCHMARK FORMAT AUDIT ==========


,company,task,examples,sparse_score,dense_score,changed_predictions,sparse_errors,dense_errors
0,company-003-dev,meaningful,690,0.7515,0.0000,395,205,430
1,company-003-dev,escalation,690,0.5294,0.0929,212,256,332
2,company-003-dev,direction,690,0.6043,0.4188,468,273,401
3,company-001,meaningful,180,1.0000,0.0000,90,0,90
4,company-001,escalation,180,1.0000,0.0000,60,0,60
5,company-001,direction,180,1.0000,0.6667,60,0,60



========== SCORE CHANGES ==========


,company,task,score_delta,changed_predictions
0,company-003-dev,meaningful,-0.7515,395
1,company-003-dev,escalation,-0.4365,212
2,company-003-dev,direction,-0.1855,468
3,company-001,meaningful,-1.0000,90
4,company-001,escalation,-1.0000,60
5,company-001,direction,-0.3333,60



PASS — Original benchmark representation audit completed.


In [28]:

# STEP 29: Test targeted sparse-zero correction
# No retraining. Original frozen models remain unchanged.

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score, accuracy_score

# Numeric feature order from original extract_xgb_features:
# 16 state features + 4 impact features + event amount
# + revenue_change_pct + 4 remaining numeric features.
REVENUE_PCT_COLUMN = X_dev_check.shape[1] - 26 + 21

def preserve_revenue_zero(X):
    """
    Keep the original CSR matrix but explicitly store zero
    in the revenue_change_pct feature column.
    """
    X_fixed = X.tolil(copy=True)

    # Only insert explicit zeros where the feature is zero.
    values = np.asarray(
        X_fixed[:, REVENUE_PCT_COLUMN].toarray()
    ).ravel()

    zero_rows = np.flatnonzero(values == 0.0)

    for row in zero_rows:
        X_fixed[row, REVENUE_PCT_COLUMN] = 1e-7

    return X_fixed.tocsr()


def audit_targeted_fix(name, X, labels):
    X_fixed = preserve_revenue_zero(X)

    results = []

    for task, model, label_key in [
        ("meaningful", meaningful_model, "meaningful_change"),
        ("escalation", escalation_model, "escalation_needed"),
        ("direction", direction_model, "direction"),
    ]:
        original_pred = model.predict(X)
        fixed_pred = model.predict(X_fixed)

        if task == "direction":
            y_true = np.array([
                DIRECTION_TO_ID[y[label_key]] for y in labels
            ])
            score = accuracy_score
        else:
            y_true = np.array([
                int(y[label_key]) for y in labels
            ])
            score = lambda a, b: f1_score(
                a, b, zero_division=0
            )

        results.append({
            "dataset": name,
            "task": task,
            "original_score": score(y_true, original_pred),
            "corrected_score": score(y_true, fixed_pred),
            "changed_predictions": int(
                np.sum(original_pred != fixed_pred)
            ),
        })

    return results


# Check original benchmarks.
targeted_audit = pd.DataFrame(
    audit_targeted_fix(
        "company-003-dev",
        X_dev_check,
        dev_labels_check,
    )
    + audit_targeted_fix(
        "company-001",
        X_company001,
        company001_labels,
    )
)

print("========== ORIGINAL BENCHMARK AUDIT ==========")
display(targeted_audit.round(4))

# Check exact-zero anomaly.
X_zero_only = X_zero_probe[zero_indices]
X_zero_fixed = preserve_revenue_zero(X_zero_only)

print("\n========== EXACT-ZERO FIX ==========")

for task, model in [
    ("meaningful", meaningful_model),
    ("escalation", escalation_model),
    ("direction", direction_model),
]:
    before = model.predict(X_zero_only)
    after = model.predict(X_zero_fixed)

    print(
        task,
        "| before:", np.unique(before, return_counts=True),
        "| after:", np.unique(after, return_counts=True)
    )

print("\nPASS — Targeted correction experiment completed.")


========== ORIGINAL BENCHMARK AUDIT ==========


,dataset,task,original_score,corrected_score,changed_predictions
0,company-003-dev,meaningful,0.7515,0.4316,255
1,company-003-dev,escalation,0.5294,0.2892,136
2,company-003-dev,direction,0.6043,0.5058,204
3,company-001,meaningful,1.0000,1.0000,0
4,company-001,escalation,1.0000,1.0000,0
5,company-001,direction,1.0000,1.0000,0



========== EXACT-ZERO FIX ==========
meaningful | before: (array([1]), array([30])) | after: (array([0]), array([30]))
escalation | before: (array([1]), array([30])) | after: (array([0]), array([30]))
direction | before: (array([0]), array([30])) | after: (array([1]), array([30]))

PASS — Targeted correction experiment completed.


In [29]:

# STEP 30: Identify where the targeted correction causes regressions

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

dev_event_types = np.array([
    record["new_event"]["event_type"]
    for record in dev_inputs_check
])

X_dev_targeted = preserve_revenue_zero(X_dev_check)

rows = []

for event_type in sorted(set(dev_event_types)):
    indices = np.flatnonzero(dev_event_types == event_type)

    for task, model, label_key in [
        ("meaningful", meaningful_model, "meaningful_change"),
        ("escalation", escalation_model, "escalation_needed"),
        ("direction", direction_model, "direction"),
    ]:
        original = model.predict(X_dev_check[indices])
        corrected = model.predict(X_dev_targeted[indices])

        if task == "direction":
            truth = np.array([
                DIRECTION_TO_ID[dev_labels_check[i][label_key]]
                for i in indices
            ])
            metric = accuracy_score
        else:
            truth = np.array([
                int(dev_labels_check[i][label_key])
                for i in indices
            ])
            metric = lambda y, p: f1_score(
                y, p, zero_division=0
            )

        rows.append({
            "event_type": event_type,
            "task": task,
            "examples": len(indices),
            "original_score": metric(truth, original),
            "corrected_score": metric(truth, corrected),
            "changed_predictions": int(
                np.sum(original != corrected)
            ),
        })

event_type_audit = pd.DataFrame(rows)

event_type_audit["score_delta"] = (
    event_type_audit["corrected_score"]
    - event_type_audit["original_score"]
)

print("========== CORRECTION IMPACT BY EVENT TYPE ==========")
display(event_type_audit.round(4))

print("\n========== OVERALL CHANGED PREDICTIONS ==========")
display(
    event_type_audit.groupby("event_type")[
        ["examples", "changed_predictions"]
    ].agg({
        "examples": "first",
        "changed_predictions": "sum",
    })
)

print("\nPASS — Event-type regression audit completed.")


========== CORRECTION IMPACT BY EVENT TYPE ==========


,event_type,task,examples,original_score,corrected_score,changed_predictions,score_delta
0,customer_message,meaningful,612,0.7273,0.3448,255,-0.3824
1,customer_message,escalation,612,0.4828,0.1905,136,-0.2923
2,customer_message,direction,612,0.5556,0.4444,204,-0.1111
3,financial_trend_observed,meaningful,78,0.9870,0.9870,0,0.0000
4,financial_trend_observed,escalation,78,0.9804,0.9804,0,0.0000
5,financial_trend_observed,direction,78,0.9872,0.9872,0,0.0000



========== OVERALL CHANGED PREDICTIONS ==========


,examples,changed_predictions
event_type,,
customer_message,612,595
financial_trend_observed,78,0



PASS — Event-type regression audit completed.


In [30]:

# STEP 31: Audit event-specific zero-change correction
# Experimental only — no retraining or model replacement.

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

def conditional_zero_fix(X, records):
    """
    Replace the revenue-change feature with a tiny nonzero
    value only for financial-trend events with exactly 0% change.
    All other rows retain their original sparse features.
    """
    assert X.shape[0] == len(records)

    selected = [
        i for i, record in enumerate(records)
        if (
            record["new_event"]["event_type"]
            == "financial_trend_observed"
            and float(
                record["new_event"]
                .get("observation", {})
                .get("revenue_change_pct", 0)
            ) == 0.0
        )
    ]

    if not selected:
        return X.copy(), 0

    X_fixed = X.tolil(copy=True)

    for i in selected:
        X_fixed[i, REVENUE_PCT_COLUMN] = 1e-7

    return X_fixed.tocsr(), len(selected)


def evaluate_conditional_fix(name, X, records, labels=None):
    X_fixed, corrected_rows = conditional_zero_fix(X, records)

    rows = []

    for task, model, label_key in [
        ("meaningful", meaningful_model, "meaningful_change"),
        ("escalation", escalation_model, "escalation_needed"),
        ("direction", direction_model, "direction"),
    ]:
        before = model.predict(X)
        after = model.predict(X_fixed)

        result = {
            "dataset": name,
            "task": task,
            "examples": len(records),
            "corrected_rows": corrected_rows,
            "changed_predictions": int(np.sum(before != after)),
        }

        if labels is not None:
            if task == "direction":
                truth = np.array([
                    DIRECTION_TO_ID[y[label_key]]
                    for y in labels
                ])
                score = accuracy_score
            else:
                truth = np.array([
                    int(y[label_key])
                    for y in labels
                ])
                score = lambda y, p: f1_score(
                    y, p, zero_division=0
                )

            result["original_score"] = score(truth, before)
            result["corrected_score"] = score(truth, after)

        rows.append(result)

    return rows


conditional_audit = pd.DataFrame(
    evaluate_conditional_fix(
        "company-003-dev",
        X_dev_check,
        dev_inputs_check,
        dev_labels_check,
    )
    + evaluate_conditional_fix(
        "company-001",
        X_company001,
        company001_inputs,
        company001_labels,
    )
    + evaluate_conditional_fix(
        "company-001-zero-stress",
        X_zero_probe[zero_indices],
        [probe_inputs[i] for i in zero_indices],
    )
)

print("========== CONDITIONAL FIX AUDIT ==========")
display(conditional_audit.round(4))

print("\n========== REGRESSION CHECK ==========")

benchmark_rows = conditional_audit[
    conditional_audit["original_score"].notna()
].copy()

regressions = benchmark_rows[
    benchmark_rows["corrected_score"]
    < benchmark_rows["original_score"] - 1e-10
]

print("Benchmark regressions:", len(regressions))

if regressions.empty:
    print("PASS — No regressions on checked benchmarks.")
else:
    print("WARNING — Correction caused benchmark regressions.")
    display(regressions)

print("\nNOTE: The correction remains experimental.")


========== CONDITIONAL FIX AUDIT ==========


,dataset,task,examples,corrected_rows,changed_predictions,original_score,corrected_score
0,company-003-dev,meaningful,690,0,0,0.7515,0.7515
1,company-003-dev,escalation,690,0,0,0.5294,0.5294
2,company-003-dev,direction,690,0,0,0.6043,0.6043
3,company-001,meaningful,180,0,0,1.0000,1.0000
4,company-001,escalation,180,0,0,1.0000,1.0000
5,company-001,direction,180,0,0,1.0000,1.0000
6,company-001-zero-stress,meaningful,30,30,30,NaN,NaN
7,company-001-zero-stress,escalation,30,30,30,NaN,NaN
8,company-001-zero-stress,direction,30,30,30,NaN,NaN



========== REGRESSION CHECK ==========
Benchmark regressions: 0
PASS — No regressions on checked benchmarks.

NOTE: The correction remains experimental.


In [31]:

# STEP 32: Unseen-percentage financial stress test
# Frozen XGBoost baseline, original sparse inference.
# No retraining and no changes to the saved models.

import copy
import numpy as np
import pandas as pd

UNSEEN_PERCENTAGES = [
    -90, -65, -50, -35, -25, -15, -8, -3,
    0, 3, 10, 20, 40, 75, 110, 160, 220
]

stress_records = []
stress_metadata = []

for pct in UNSEEN_PERCENTAGES:
    for customer_idx, original in enumerate(base_records):
        record = copy.deepcopy(original)

        observation = record["new_event"]["observation"]
        baseline = float(observation["baseline_30d_revenue"])

        observation["revenue_change_pct"] = float(pct)
        observation["observed_30d_revenue"] = round(
            baseline * (1 + pct / 100),
            2
        )

        stress_records.append(record)
        stress_metadata.append({
            "revenue_change_pct": pct,
            "customer_idx": customer_idx,
        })

X_stress = build_frozen_features(stress_records)

stress_df = pd.DataFrame(stress_metadata)
stress_df["meaningful_pred"] = meaningful_model.predict(X_stress)
stress_df["escalation_pred"] = escalation_model.predict(X_stress)
stress_df["direction_pred"] = direction_model.predict(X_stress)

stress_summary = (
    stress_df.groupby("revenue_change_pct")
    .agg(
        customers=("customer_idx", "count"),
        meaningful_rate=("meaningful_pred", "mean"),
        escalation_rate=("escalation_pred", "mean"),
        negative_rate=("direction_pred", lambda x: (x == 0).mean()),
        neutral_rate=("direction_pred", lambda x: (x == 1).mean()),
        positive_rate=("direction_pred", lambda x: (x == 2).mean()),
    )
    .reset_index()
)

print("========== UNSEEN-PERCENTAGE STRESS TEST ==========")
display(stress_summary.round(3))

print("\nTotal examples:", len(stress_df))
print("Frozen model: XGBoost")
print("Inference: original sparse pipeline")
print("Ground-truth F1: not calculated (no independent labels)")

print("\nPASS — Unseen-percentage stress test completed.")


========== UNSEEN-PERCENTAGE STRESS TEST ==========


,revenue_change_pct,customers,meaningful_rate,escalation_rate,negative_rate,neutral_rate,positive_rate
0,-90,30,1.000,1.000,1.000,0.000,0.0
1,-65,30,1.000,1.000,1.000,0.000,0.0
2,-50,30,1.000,0.233,1.000,0.000,0.0
3,-35,30,1.000,0.167,1.000,0.000,0.0
4,-25,30,1.000,0.000,1.000,0.000,0.0
5,-15,30,0.433,0.000,0.433,0.567,0.0
6,-8,30,0.000,0.000,0.000,1.000,0.0
7,-3,30,0.000,0.000,0.000,1.000,0.0
8,0,30,1.000,1.000,1.000,0.000,0.0
9,3,30,0.000,0.000,0.000,1.000,0.0



Total examples: 510
Frozen model: XGBoost
Inference: original sparse pipeline
Ground-truth F1: not calculated (no independent labels)

PASS — Unseen-percentage stress test completed.


In [32]:

# STEP 33 — CORRECTED: Cross-company unseen-percentage audit

import copy
import numpy as np
import pandas as pd

PERCENTAGES = [
    -90, -65, -50, -35, -25, -15, -8, -3,
    0, 3, 10, 20, 40, 75, 110, 160, 220
]

cross_stress_rows = []
skipped_companies = []

for company_dir in company_dirs:
    company_id = company_dir.name

    # Exclude the original training company.
    if company_id.lower().replace("-", "").replace("_", "") in (
        "company003", "003"
    ):
        continue

    inputs, labels = generate_company_financial_examples(company_dir)

    if not inputs:
        skipped_companies.append(company_id)
        continue

    # Identify stable-revenue records using labels, not new_event.
    # Falls back to scenario position only if labels lack scenario_type.
    if labels and "scenario_type" in labels[0]:
        baselines = [
            record
            for record, label in zip(inputs, labels)
            if label["scenario_type"] == "stable_revenue"
        ]
    else:
        # Generator produces six scenarios per customer.
        # stable_revenue is the fourth scenario (index 3).
        baselines = inputs[3::6]

    assert baselines, f"No baseline records for {company_id}"

    records = []
    metadata = []

    for pct in PERCENTAGES:
        for customer_idx, original in enumerate(baselines):
            record = copy.deepcopy(original)
            observation = record["new_event"]["observation"]
            baseline = float(observation["baseline_30d_revenue"])

            observation["revenue_change_pct"] = float(pct)
            observation["observed_30d_revenue"] = round(
                baseline * (1 + pct / 100),
                2
            )

            records.append(record)
            metadata.append((pct, customer_idx))

    X = build_frozen_features(records)

    meaningful = meaningful_model.predict(X)
    escalation = escalation_model.predict(X)
    direction = direction_model.predict(X)

    for (pct, customer_idx), m, e, d in zip(
        metadata, meaningful, escalation, direction
    ):
        cross_stress_rows.append({
            "company": company_id,
            "customer_idx": customer_idx,
            "revenue_change_pct": pct,
            "meaningful_pred": int(m),
            "escalation_pred": int(e),
            "direction_pred": int(d),
        })

cross_stress_df = pd.DataFrame(cross_stress_rows)

cross_stress_summary = (
    cross_stress_df.groupby("revenue_change_pct")
    .agg(
        companies=("company", "nunique"),
        examples=("company", "size"),
        meaningful_rate=("meaningful_pred", "mean"),
        escalation_rate=("escalation_pred", "mean"),
        negative_rate=("direction_pred", lambda x: (x == 0).mean()),
        neutral_rate=("direction_pred", lambda x: (x == 1).mean()),
        positive_rate=("direction_pred", lambda x: (x == 2).mean()),
    )
    .reset_index()
)

print("========== CROSS-COMPANY UNSEEN-PERCENTAGE AUDIT ==========")
display(cross_stress_summary.round(3))

print("\nHeld-out companies:", cross_stress_df["company"].nunique())
print(
    "Unique customers:",
    cross_stress_df[["company", "customer_idx"]]
    .drop_duplicates()
    .shape[0]
)
print("Total examples:", len(cross_stress_df))
print("Skipped companies:", len(skipped_companies))

print("\nGround-truth F1 not calculated: this is a prediction stress test.")
print("PASS — Cross-company robustness audit completed.")


========== CROSS-COMPANY UNSEEN-PERCENTAGE AUDIT ==========


,revenue_change_pct,companies,examples,meaningful_rate,escalation_rate,negative_rate,neutral_rate,positive_rate
0,-90,39,1445,1.000,1.000,1.000,0.000,0.0
1,-65,39,1445,1.000,1.000,1.000,0.000,0.0
2,-50,39,1445,1.000,0.306,1.000,0.000,0.0
3,-35,39,1445,1.000,0.210,1.000,0.000,0.0
4,-25,39,1445,1.000,0.000,1.000,0.000,0.0
5,-15,39,1445,0.618,0.000,0.441,0.559,0.0
6,-8,39,1445,0.000,0.000,0.000,1.000,0.0
7,-3,39,1445,0.000,0.000,0.000,1.000,0.0
8,0,39,1445,1.000,1.000,1.000,0.000,0.0
9,3,39,1445,0.000,0.000,0.000,1.000,0.0



Held-out companies: 39
Unique customers: 1445
Total examples: 24565
Skipped companies: 60

Ground-truth F1 not calculated: this is a prediction stress test.
PASS — Cross-company robustness audit completed.


In [33]:

# STEP 34: Save final cross-company generalization evidence

from pathlib import Path
import json

output_dir = Path("cross_company_benchmark")
output_dir.mkdir(parents=True, exist_ok=True)

cross_stress_df.to_csv(
    output_dir / "unseen_percentage_predictions.csv",
    index=False
)

cross_stress_summary.to_csv(
    output_dir / "unseen_percentage_summary.csv",
    index=False
)

report = {
    "model": "Frozen XGBoost baseline",
    "training_company": "Company 003",
    "heldout_companies": int(
        cross_stress_df["company"].nunique()
    ),
    "eligible_customers": int(
        cross_stress_df[
            ["company", "customer_idx"]
        ].drop_duplicates().shape[0]
    ),
    "controlled_scenario_examples": 8670,
    "controlled_meaningful_f1": 1.0,
    "controlled_escalation_f1": 1.0,
    "controlled_direction_accuracy": 1.0,
    "unseen_percentage_examples": int(len(cross_stress_df)),
    "unseen_percentage_levels": int(
        cross_stress_df["revenue_change_pct"].nunique()
    ),
    "stress_test_has_ground_truth": False,
    "known_limitation": (
        "At exactly 0% revenue change, the original sparse "
        "pipeline predicts meaningful change, escalation, "
        "and negative direction for all tested customers."
    ),
    "scope": (
        "Controlled synthetic financial scenarios grounded "
        "in held-out company accounting histories; "
        "not evidence of real-world outcome accuracy or "
        "Jev/System One cross-company performance."
    ),
}

with open(
    output_dir / "generalization_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("========== SAVED GENERALIZATION EVIDENCE ==========")

for name in [
    "unseen_percentage_predictions.csv",
    "unseen_percentage_summary.csv",
    "generalization_report.json",
]:
    path = output_dir / name
    print(f"{path} — {path.stat().st_size:,} bytes")

print("\nPASS — Generalization evidence saved.")


========== SAVED GENERALIZATION EVIDENCE ==========
cross_company_benchmark/unseen_percentage_predictions.csv — 595,767 bytes
cross_company_benchmark/unseen_percentage_summary.csv — 721 bytes
cross_company_benchmark/generalization_report.json — 766 bytes

PASS — Generalization evidence saved.


In [34]:
#rule based generalization

In [35]:

import re

NEGATIVE_ESCALATE = [
    r"\b(other|another|competing|competitor)\s+(supplier|distributor|vendor)\b",
    r"\b(stop|discontinue|terminate|cancel|end)\b.{0,50}\b(purchas|order|contract|agreement|relationship)\b",
    r"\b(cash shortage|cash flow problems|financial difficulties)\b",
    r"\b(delayed payments?|unable to pay|struggling to pay)\b",
    r"\b(closing|shutting down)\b.{0,40}\b(location|branch|store|outlet)s?\b",
    r"\b(budget freeze|spending freeze|purchasing freeze)\b",
    r"\b(repeated|recurring|multiple)\b.{0,40}\b(delivery failures?|quality issues?|defects?)\b",
    r"\b(credit limit|credit ceiling|longer credit terms|net.sixty|net.60)\b",
    r"\b(reduc|cut|decreas|dropp|shrink)\w*\b.{0,55}\b(order|purchas|volume|quantit|deliver)\w*\b",
]

POSITIVE_ESCALATE = [
    r"\b(primary|preferred|main|sole|exclusive|first.choice)\b.{0,45}\b(supplier|vendor|distributor|partner)\b",
    r"\b(expand|expansion|opening|launching|additional)\b.{0,55}\b(location|branch|store|outlet|region)s?\b",
    r"\b(double|doubling|substantially more|significantly more)\b.{0,50}\b(purchas|order|inventory|volume)\b",
    r"\b(two.year|three.year|multi.year|long.term)\b.{0,55}\b(contract|agreement|partnership|relationship)\b",
    r"\b(financing|funding)\b.{0,50}\b(cleared|resolved|came through)\b",
    r"\b(payment schedule|payments?)\b.{0,50}\b(back to normal|resume normally)\b",
]

MEANINGFUL_ONLY = [
    r"\b(ten percent|10%|one tenth)\b.{0,65}\b(order|purchas|volume|quantit)\w*\b",
    r"\b(permanent|recurring|standing)\b.{0,55}\b(small|minor|modest|slight)\b.{0,30}\b(order|purchas|product)\w*\b",
    r"\b(small|minor|modest|slight)\b.{0,55}\b(permanent|recurring|standing)\b.{0,40}\b(order|purchas|product)\w*\b",
]

def rules_detector_v1(record):
    event = record["new_event"]
    event_type = event["event_type"]

    result = {
        "meaningful_change": False,
        "direction": "neutral",
        "escalation_needed": False,
    }

    if event_type == "financial_trend_observed":
        change = float(event["observation"]["revenue_change_pct"])

        if change <= -50:
            return {
                "meaningful_change": True,
                "direction": "negative",
                "escalation_needed": True,
            }

        if change >= 60:
            return {
                "meaningful_change": True,
                "direction": "positive",
                "escalation_needed": True,
            }

        if change <= -20:
            return {
                "meaningful_change": True,
                "direction": "negative",
                "escalation_needed": False,
            }

        return result

    text = event.get("description", "").lower()

    if any(re.search(pattern, text) for pattern in NEGATIVE_ESCALATE):
        return {
            "meaningful_change": True,
            "direction": "negative",
            "escalation_needed": True,
        }

    if any(re.search(pattern, text) for pattern in POSITIVE_ESCALATE):
        return {
            "meaningful_change": True,
            "direction": "positive",
            "escalation_needed": True,
        }

    if any(re.search(pattern, text) for pattern in MEANINGFUL_ONLY):
        return {
            "meaningful_change": True,
            "direction": "neutral",
            "escalation_needed": False,
        }

    return result

print("PASS — Original Rules Detector V1 loaded.")

# Quick check on existing financial examples
for record in company001_inputs[:3]:
    print(rules_detector_v1(record))


PASS — Original Rules Detector V1 loaded.
{'meaningful_change': True, 'direction': 'negative', 'escalation_needed': True}
{'meaningful_change': True, 'direction': 'negative', 'escalation_needed': False}
{'meaningful_change': False, 'direction': 'neutral', 'escalation_needed': False}


In [36]:

# STEP 35: Cross-company Rules vs XGBoost comparison
# Reuse the original rules_detector_v1 without changing thresholds.

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

assert callable(rules_detector_v1), (
    "Run the original Rules Detector V1 definition cell first."
)

comparison_rows = []

for company_dir in company_dirs:
    company_id = company_dir.name

    if company_id.lower().replace("-", "").replace("_", "") in (
        "company003", "003"
    ):
        continue

    inputs, labels = generate_company_financial_examples(company_dir)

    if not inputs:
        continue

    rules_preds = [
        rules_detector_v1(record)
        for record in inputs
    ]

    X = build_frozen_features(inputs)

    xgb_meaningful = meaningful_model.predict(X)
    xgb_escalation = escalation_model.predict(X)
    xgb_direction = direction_model.predict(X)

    truth_meaningful = np.array([
        int(label["meaningful_change"])
        for label in labels
    ])
    truth_escalation = np.array([
        int(label["escalation_needed"])
        for label in labels
    ])
    truth_direction = np.array([
        DIRECTION_TO_ID[label["direction"]]
        for label in labels
    ])

    model_predictions = {
        "XGBoost": (
            xgb_meaningful,
            xgb_escalation,
            xgb_direction,
        ),
        "Rules V1": (
            np.array([
                int(pred["meaningful_change"])
                for pred in rules_preds
            ]),
            np.array([
                int(pred["escalation_needed"])
                for pred in rules_preds
            ]),
            np.array([
                DIRECTION_TO_ID[pred["direction"]]
                for pred in rules_preds
            ]),
        ),
    }

    for model_name, (m, e, d) in model_predictions.items():
        comparison_rows.append({
            "company": company_id,
            "model": model_name,
            "examples": len(inputs),
            "meaningful_f1": f1_score(
                truth_meaningful, m, zero_division=0
            ),
            "escalation_f1": f1_score(
                truth_escalation, e, zero_division=0
            ),
            "direction_accuracy": accuracy_score(
                truth_direction, d
            ),
        })

rules_xgb_company_df = pd.DataFrame(comparison_rows)

comparison_summary = (
    rules_xgb_company_df.groupby("model")
    .agg(
        companies=("company", "nunique"),
        examples_per_company_total=("examples", "sum"),
        mean_company_meaningful_f1=("meaningful_f1", "mean"),
        mean_company_escalation_f1=("escalation_f1", "mean"),
        mean_company_direction_accuracy=("direction_accuracy", "mean"),
    )
    .reset_index()
)

print("========== CROSS-COMPANY RULES VS XGBOOST ==========")
display(comparison_summary.round(4))

print("\n========== COMPANY-LEVEL RESULTS ==========")
display(rules_xgb_company_df.head(10).round(4))

print("\nPASS — Rules and XGBoost evaluated on identical examples.")


========== CROSS-COMPANY RULES VS XGBOOST ==========


,model,companies,examples_per_company_total,mean_company_meaningful_f1,mean_company_escalation_f1,mean_company_direction_accuracy
0,Rules V1,39,8670,1.0,1.0,1.0
1,XGBoost,39,8670,1.0,1.0,1.0



========== COMPANY-LEVEL RESULTS ==========


,company,model,examples,meaningful_f1,escalation_f1,direction_accuracy
0,company-001,XGBoost,180,1.0,1.0,1.0
1,company-001,Rules V1,180,1.0,1.0,1.0
2,company-002,XGBoost,336,1.0,1.0,1.0
3,company-002,Rules V1,336,1.0,1.0,1.0
4,company-004,XGBoost,252,1.0,1.0,1.0
5,company-004,Rules V1,252,1.0,1.0,1.0
6,company-005,XGBoost,360,1.0,1.0,1.0
7,company-005,Rules V1,360,1.0,1.0,1.0
8,company-006,XGBoost,708,1.0,1.0,1.0
9,company-006,Rules V1,708,1.0,1.0,1.0



PASS — Rules and XGBoost evaluated on identical examples.


In [37]:

# STEP 36: Compare Rules V1 and XGBoost on unseen percentages
# Uses existing cross_stress_df; no regeneration or model calls.

import numpy as np
import pandas as pd

def rules_financial_prediction(pct):
    # Same financial thresholds as the original Rules V1.
    if pct <= -50:
        return (1, 1, 0)
    if pct >= 60:
        return (1, 1, 2)
    if pct <= -20:
        return (1, 0, 0)
    return (0, 0, 1)

comparison = cross_stress_df.copy()

rules_outputs = [
    rules_financial_prediction(pct)
    for pct in comparison["revenue_change_pct"]
]

comparison[
    ["rules_meaningful", "rules_escalation", "rules_direction"]
] = pd.DataFrame(
    rules_outputs,
    index=comparison.index
)

comparison["meaningful_disagree"] = (
    comparison["meaningful_pred"]
    != comparison["rules_meaningful"]
)

comparison["escalation_disagree"] = (
    comparison["escalation_pred"]
    != comparison["rules_escalation"]
)

comparison["direction_disagree"] = (
    comparison["direction_pred"]
    != comparison["rules_direction"]
)

disagreement_summary = (
    comparison.groupby("revenue_change_pct")
    .agg(
        examples=("company", "size"),
        meaningful_disagreement=("meaningful_disagree", "mean"),
        escalation_disagreement=("escalation_disagree", "mean"),
        direction_disagreement=("direction_disagree", "mean"),
    )
    .reset_index()
)

print("========== RULES VS XGBOOST: UNSEEN PERCENTAGES ==========")
display(disagreement_summary.round(3))

print("\n========== OVERALL DISAGREEMENT ==========")
for task in ["meaningful", "escalation", "direction"]:
    print(
        f"{task}: "
        f"{comparison[f'{task}_disagree'].mean():.1%}"
    )

print("\nPASS — Multi-model stress-test comparison completed.")


========== RULES VS XGBOOST: UNSEEN PERCENTAGES ==========


,revenue_change_pct,examples,meaningful_disagreement,escalation_disagreement,direction_disagreement
0,-90,1445,0.000,0.000,0.000
1,-65,1445,0.000,0.000,0.000
2,-50,1445,0.000,0.694,0.000
3,-35,1445,0.000,0.210,0.000
4,-25,1445,0.000,0.000,0.000
5,-15,1445,0.618,0.000,0.441
6,-8,1445,0.000,0.000,0.000
7,-3,1445,0.000,0.000,0.000
8,0,1445,1.000,1.000,1.000
9,3,1445,0.000,0.000,0.000



========== OVERALL DISAGREEMENT ==========
meaningful: 9.5%
escalation: 11.2%
direction: 8.5%

PASS — Multi-model stress-test comparison completed.


In [38]:

# STEP 37: Create a reproducible cross-company multi-model set
# No API calls, no model training.

import copy
import numpy as np
import pandas as pd

TEST_PERCENTAGES = [-65, -35, -15, 0, 75]
CUSTOMERS_PER_COMPANY = 10
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

multi_model_inputs = []
multi_model_metadata = []

for company_dir in company_dirs:
    company_id = company_dir.name

    if company_id.lower().replace("-", "").replace("_", "") in (
        "company003", "003"
    ):
        continue

    inputs, labels = generate_company_financial_examples(company_dir)

    if not inputs:
        continue

    # One stable-revenue template per eligible customer.
    baselines = [
        record for record, label in zip(inputs, labels)
        if label.get("scenario_type") == "stable_revenue"
    ]

    if not baselines:
        baselines = inputs[3::6]

    chosen = rng.choice(
        len(baselines),
        size=min(CUSTOMERS_PER_COMPANY, len(baselines)),
        replace=False
    )

    for customer_idx in sorted(chosen):
        original = baselines[int(customer_idx)]

        for pct in TEST_PERCENTAGES:
            record = copy.deepcopy(original)
            obs = record["new_event"]["observation"]
            baseline = float(obs["baseline_30d_revenue"])

            obs["revenue_change_pct"] = float(pct)
            obs["observed_30d_revenue"] = round(
                baseline * (1 + pct / 100), 2
            )

            example_id = f"{company_id}-c{customer_idx}-p{pct}"

            multi_model_inputs.append(record)
            multi_model_metadata.append({
                "example_id": example_id,
                "company": company_id,
                "customer_idx": int(customer_idx),
                "revenue_change_pct": pct,
            })

multi_model_df = pd.DataFrame(multi_model_metadata)

assert len(multi_model_df) == len(multi_model_inputs)
assert multi_model_df["example_id"].is_unique

print("========== MULTI-MODEL CROSS-COMPANY SET ==========")
print("Companies:", multi_model_df["company"].nunique())
print(
    "Customers:",
    multi_model_df[["company", "customer_idx"]]
    .drop_duplicates().shape[0]
)
print("Examples:", len(multi_model_df))

display(
    multi_model_df.groupby("revenue_change_pct")
    .agg(
        companies=("company", "nunique"),
        examples=("example_id", "count")
    )
)

print("\nPASS — Shared evaluation inputs created.")


========== MULTI-MODEL CROSS-COMPANY SET ==========
Companies: 39
Customers: 308
Examples: 1540


,companies,examples
revenue_change_pct,,
-65,39,308
-35,39,308
-15,39,308
0,39,308
75,39,308



PASS — Shared evaluation inputs created.


In [39]:

# STEP 38: Freeze the shared multi-model evaluation set

from pathlib import Path
import json
import hashlib

output_dir = Path("cross_company_benchmark")
output_dir.mkdir(parents=True, exist_ok=True)

inputs_path = output_dir / "multi_model_inputs.jsonl"
metadata_path = output_dir / "multi_model_metadata.csv"
manifest_path = output_dir / "multi_model_manifest.json"

with inputs_path.open("w", encoding="utf-8") as f:
    for record in multi_model_inputs:
        f.write(json.dumps(record, sort_keys=True, default=str) + "\n")

multi_model_df.to_csv(metadata_path, index=False)

dataset_sha256 = hashlib.sha256(
    inputs_path.read_bytes()
).hexdigest()

manifest = {
    "name": "Cross-company multi-model financial evaluation",
    "source_training_company": "company-003",
    "heldout_companies": int(multi_model_df["company"].nunique()),
    "customers": int(
        multi_model_df[["company", "customer_idx"]]
        .drop_duplicates().shape[0]
    ),
    "examples": len(multi_model_inputs),
    "revenue_change_percentages": [-65, -35, -15, 0, 75],
    "random_seed": 42,
    "customers_per_company_max": 10,
    "inputs_sha256": dataset_sha256,
    "ground_truth_labels": "not yet established",
    "evaluation_scope": (
        "Controlled hypothetical financial trends based on "
        "held-out company accounting histories"
    ),
}

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("========== FROZEN MULTI-MODEL DATASET ==========")
print("Inputs:", inputs_path)
print("Metadata:", metadata_path)
print("Manifest:", manifest_path)
print("SHA-256:", dataset_sha256)
print("Examples:", len(multi_model_inputs))

assert sum(1 for _ in inputs_path.open()) == 1540
assert multi_model_df["example_id"].is_unique

print("\nPASS — Shared evaluation set saved and fingerprinted.")


========== FROZEN MULTI-MODEL DATASET ==========
Inputs: cross_company_benchmark/multi_model_inputs.jsonl
Metadata: cross_company_benchmark/multi_model_metadata.csv
Manifest: cross_company_benchmark/multi_model_manifest.json
SHA-256: 313cb7bc119271f6a8626aacc4af70a263df5876c6b034bfdcf91452fad335db
Examples: 1540

PASS — Shared evaluation set saved and fingerprinted.


In [40]:
print("Frozen XGBoost available:", "model" in globals())
print("Rules V1 available:", "rules_detector_v1" in globals())
print("Benchmark inputs available:", "multi_model_inputs" in globals())
print("Benchmark metadata available:", "multi_model_df" in globals())

if "multi_model_inputs" in globals():
    print("Total examples:", len(multi_model_inputs))

Frozen XGBoost available: True
Rules V1 available: True
Benchmark inputs available: True
Benchmark metadata available: True
Total examples: 1540


In [41]:

# STEP 43 — Freeze synthetic policy-reference labels
# Does not call any model or API.

import json
import hashlib
from pathlib import Path

OUTPUT_DIR = Path("cross_company_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_PATH = OUTPUT_DIR / "multi_model_policy_labels.jsonl"

assert len(multi_model_inputs) == len(multi_model_df) == 1540
assert multi_model_df["example_id"].is_unique

# Match the original Rules V1 financial policy.
def reference_policy(pct):
    pct = float(pct)

    if pct <= -50:
        return True, "negative", True
    if pct >= 60:
        return True, "positive", True
    if pct <= -20:
        return True, "negative", False

    return False, "neutral", False


labels = []

for row in multi_model_df.itertuples(index=False):
    meaningful, direction, escalation = reference_policy(
        row.revenue_change_pct
    )

    labels.append({
        "example_id": row.example_id,
        "meaningful_change": meaningful,
        "direction": direction,
        "escalation_needed": escalation,
    })

# Never silently overwrite an existing reference-label file.
serialized = "".join(
    json.dumps(label, sort_keys=True) + "\n"
    for label in labels
)

if LABELS_PATH.exists():
    existing = LABELS_PATH.read_text(encoding="utf-8")
    assert existing == serialized, (
        "Existing label file differs. "
        "Do not overwrite without investigating."
    )
    print("Existing frozen labels verified.")
else:
    LABELS_PATH.write_text(
        serialized,
        encoding="utf-8"
    )
    print("New policy-reference labels saved.")

digest = hashlib.sha256(
    LABELS_PATH.read_bytes()
).hexdigest()

print("\n========== POLICY LABEL VERIFICATION ==========")
print("Total labels:", len(labels))
print("Unique IDs:", len(set(x["example_id"] for x in labels)))
print("Meaningful positives:",
      sum(x["meaningful_change"] for x in labels))
print("Escalation positives:",
      sum(x["escalation_needed"] for x in labels))
print("SHA-256:", digest)
print("Saved to:", LABELS_PATH)


New policy-reference labels saved.

========== POLICY LABEL VERIFICATION ==========
Total labels: 1540
Unique IDs: 1540
Meaningful positives: 924
Escalation positives: 616
SHA-256: 8a9221d5c8f872e3bd6bdbd90464b1d1dc4e103530a06a2b0988230cef2debfa
Saved to: cross_company_benchmark/multi_model_policy_labels.jsonl


In [42]:

# STEP 44 — Generate XGBoost and Rules V1 predictions
# Uses frozen models and the exact same 1,540 inputs.
# No training and no API calls.

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

OUTPUT_DIR = Path("cross_company_benchmark")

# Verify required objects exist before running.
required = [
    "multi_model_inputs",
    "multi_model_df",
    "rules_detector_v1",
    "build_frozen_features",
    "meaningful_model",
    "escalation_model",
    "direction_model",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing notebook objects: " + ", ".join(missing)
    )

assert len(multi_model_inputs) == len(multi_model_df) == 1540

# ---------------------------------------------------------
# RULES V1
# ---------------------------------------------------------

rules_predictions = []

rules_start = time.perf_counter()

for record in multi_model_inputs:
    rules_predictions.append(
        rules_detector_v1(record)
    )

rules_elapsed = time.perf_counter() - rules_start

# ---------------------------------------------------------
# FROZEN XGBOOST
# ---------------------------------------------------------

xgb_start = time.perf_counter()

X = build_frozen_features(multi_model_inputs)

xgb_meaningful = meaningful_model.predict(X)
xgb_escalation = escalation_model.predict(X)
xgb_direction = direction_model.predict(X)

xgb_elapsed = time.perf_counter() - xgb_start

direction_mapping = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

# ---------------------------------------------------------
# SAVE PREDICTIONS
# ---------------------------------------------------------

def save_predictions(filename, predictions):
    path = OUTPUT_DIR / filename

    with path.open("w", encoding="utf-8") as f:
        for i, pred in enumerate(predictions):
            row = {
                "example_id": multi_model_df.iloc[i]["example_id"],
                "meaningful_change": bool(pred["meaningful_change"]),
                "direction": str(pred["direction"]),
                "escalation_needed": bool(pred["escalation_needed"]),
            }

            f.write(json.dumps(row) + "\n")

    return path


rules_path = save_predictions(
    "rules_v1_predictions.jsonl",
    rules_predictions,
)

xgb_predictions = [
    {
        "meaningful_change": bool(xgb_meaningful[i]),
        "direction": direction_mapping[int(xgb_direction[i])],
        "escalation_needed": bool(xgb_escalation[i]),
    }
    for i in range(len(multi_model_inputs))
]

xgb_path = save_predictions(
    "xgboost_predictions.jsonl",
    xgb_predictions,
)

print("========== STEP 44 RESULTS ==========")
print("Examples:", len(multi_model_inputs))
print("Rules predictions:", len(rules_predictions))
print("XGBoost predictions:", len(xgb_predictions))
print("Rules total inference time:", round(rules_elapsed, 4), "seconds")
print("XGBoost total inference time:", round(xgb_elapsed, 4), "seconds")
print("Rules saved:", rules_path)
print("XGBoost saved:", xgb_path)

print("\nPASS — Both models evaluated on frozen inputs.")


========== STEP 44 RESULTS ==========
Examples: 1540
Rules predictions: 1540
XGBoost predictions: 1540
Rules total inference time: 0.008 seconds
XGBoost total inference time: 0.0428 seconds
Rules saved: cross_company_benchmark/rules_v1_predictions.jsonl
XGBoost saved: cross_company_benchmark/xgboost_predictions.jsonl

PASS — Both models evaluated on frozen inputs.


In [43]:

# STEP 45 — Evaluate Rules V1 and XGBoost
# No training, no API calls.

import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

BASE = Path("cross_company_benchmark")

def load_jsonl_df(filename):
    path = BASE / filename

    with path.open(encoding="utf-8") as f:
        rows = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    df = pd.DataFrame(rows)

    assert len(df) == 1540, (
        f"{filename}: expected 1540 rows, got {len(df)}"
    )
    assert df["example_id"].is_unique

    return df


labels = load_jsonl_df(
    "multi_model_policy_labels.jsonl"
)

model_files = {
    "Rules V1": "rules_v1_predictions.jsonl",
    "XGBoost": "xgboost_predictions.jsonl",
}

results = []

for model_name, filename in model_files.items():
    preds = load_jsonl_df(filename)

    merged = labels.merge(
        preds,
        on="example_id",
        suffixes=("_true", "_pred"),
        validate="one_to_one",
    )

    assert len(merged) == 1540

    row = {"model": model_name, "examples": len(merged)}

    for task, prefix in [
        ("meaningful_change", "meaningful"),
        ("escalation_needed", "escalation"),
    ]:
        y_true = merged[f"{task}_true"].astype(int)
        y_pred = merged[f"{task}_pred"].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        row[f"{prefix}_precision"] = precision_score(
            y_true, y_pred, zero_division=0
        )
        row[f"{prefix}_recall"] = recall_score(
            y_true, y_pred, zero_division=0
        )
        row[f"{prefix}_f1"] = f1_score(
            y_true, y_pred, zero_division=0
        )
        row[f"{prefix}_tp"] = int(tp)
        row[f"{prefix}_fp"] = int(fp)
        row[f"{prefix}_fn"] = int(fn)
        row[f"{prefix}_tn"] = int(tn)

    row["direction_accuracy"] = accuracy_score(
        merged["direction_true"],
        merged["direction_pred"],
    )

    results.append(row)


results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "model",
            "examples",
            "meaningful_precision",
            "meaningful_recall",
            "meaningful_f1",
            "escalation_precision",
            "escalation_recall",
            "escalation_f1",
            "direction_accuracy",
        ]
    ].round(4)
)

results_path = BASE / "local_models_policy_metrics.csv"
results_df.to_csv(results_path, index=False)

print("\nSaved:", results_path)
print("PASS — Local model policy-reference evaluation complete.")


,model,examples,meaningful_precision,meaningful_recall,meaningful_f1,escalation_precision,escalation_recall,escalation_f1,direction_accuracy
0,Rules V1,1540,1.0000,1.0,1.0000,1.0000,1.0,1.0000,1.0000
1,XGBoost,1540,0.6351,1.0,0.7768,0.6117,1.0,0.7591,0.6909



Saved: cross_company_benchmark/local_models_policy_metrics.csv
PASS — Local model policy-reference evaluation complete.


In [46]:

# STEP 46 — Evaluate Jev against frozen policy labels
# No API calls.

import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

BASE = Path("cross_company_benchmark")

def load_eval_file(filename):
    with (BASE / filename).open(encoding="utf-8") as f:
        df = pd.DataFrame(
            json.loads(line)
            for line in f
            if line.strip()
        )

    assert len(df) == 1540, (
        f"{filename}: expected 1540 rows, got {len(df)}"
    )
    assert df["example_id"].is_unique
    return df


labels = load_eval_file("multi_model_policy_labels.jsonl")
jev = pd.DataFrame(
    json.loads(line)
    for line in Path("/Users/anushreenema/jev_predictions.jsonl")
        .read_text(encoding="utf-8").splitlines()
    if line.strip()
)

assert len(jev) == 1540, f"Expected 1540 Jev predictions, got {len(jev)}"
assert jev["example_id"].is_unique

merged = labels.merge(
    jev,
    on="example_id",
    suffixes=("_true", "_pred"),
    validate="one_to_one",
)

assert len(merged) == 1540

result = {
    "model": "Jev System One",
    "examples": len(merged),
}

for task, prefix in [
    ("meaningful_change", "meaningful"),
    ("escalation_needed", "escalation"),
]:
    truth = merged[f"{task}_true"].astype(int)
    predicted = merged[f"{task}_pred"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        truth, predicted, labels=[0, 1]
    ).ravel()

    result[f"{prefix}_precision"] = precision_score(
        truth, predicted, zero_division=0
    )
    result[f"{prefix}_recall"] = recall_score(
        truth, predicted, zero_division=0
    )
    result[f"{prefix}_f1"] = f1_score(
        truth, predicted, zero_division=0
    )
    result[f"{prefix}_tp"] = int(tp)
    result[f"{prefix}_fp"] = int(fp)
    result[f"{prefix}_fn"] = int(fn)
    result[f"{prefix}_tn"] = int(tn)

result["direction_accuracy"] = accuracy_score(
    merged["direction_true"],
    merged["direction_pred"],
)

jev_metrics = pd.DataFrame([result])

display(
    jev_metrics[
        [
            "model",
            "examples",
            "meaningful_precision",
            "meaningful_recall",
            "meaningful_f1",
            "escalation_precision",
            "escalation_recall",
            "escalation_f1",
            "direction_accuracy",
        ]
    ].round(4)
)

output_path = BASE / "jev_policy_metrics.csv"
jev_metrics.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("PASS — Jev policy-reference evaluation complete.")


,model,examples,meaningful_precision,meaningful_recall,meaningful_f1,escalation_precision,escalation_recall,escalation_f1,direction_accuracy
0,Jev System One,1540,0.7131,0.8095,0.7582,0.5508,0.625,0.5856,0.8



Saved: cross_company_benchmark/jev_policy_metrics.csv
PASS — Jev policy-reference evaluation complete.


In [45]:

from pathlib import Path

home = Path.home()

matches = list(home.glob("jev_predictions.jsonl"))

print("Home directory:", home)
print("Jev files found:", len(matches))

for path in matches:
    print("Found:", path)


Home directory: /Users/anushreenema
Jev files found: 1
Found: /Users/anushreenema/jev_predictions.jsonl


In [47]:

import pandas as pd
from pathlib import Path

BASE = Path("cross_company_benchmark")

local = pd.read_csv(
    BASE / "local_models_policy_metrics.csv"
)

jev = pd.read_csv(
    BASE / "jev_policy_metrics.csv"
)

comparison = pd.concat(
    [local, jev],
    ignore_index=True
)

assert len(comparison) == 3
assert comparison["model"].is_unique
assert (comparison["examples"] == 1540).all()

columns = [
    "model",
    "examples",
    "meaningful_precision",
    "meaningful_recall",
    "meaningful_f1",
    "escalation_precision",
    "escalation_recall",
    "escalation_f1",
    "direction_accuracy",
]

comparison = comparison[columns]

display(comparison.round(4))

output = BASE / "three_model_comparison.csv"
comparison.to_csv(output, index=False)

print("Saved:", output)
print("PASS — Three-model comparison complete.")


,model,examples,meaningful_precision,meaningful_recall,meaningful_f1,escalation_precision,escalation_recall,escalation_f1,direction_accuracy
0,Rules V1,1540,1.0000,1.0000,1.0000,1.0000,1.000,1.0000,1.0000
1,XGBoost,1540,0.6351,1.0000,0.7768,0.6117,1.000,0.7591,0.6909
2,Jev System One,1540,0.7131,0.8095,0.7582,0.5508,0.625,0.5856,0.8000


Saved: cross_company_benchmark/three_model_comparison.csv
PASS — Three-model comparison complete.


In [48]:

import pandas as pd
import json
from pathlib import Path

BASE = Path("cross_company_benchmark")

metadata = multi_model_df[
    ["example_id", "revenue_change_pct"]
].copy()

def read_predictions(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame(
            json.loads(line) for line in f if line.strip()
        )

labels = read_predictions(
    BASE / "multi_model_policy_labels.jsonl"
)[["example_id", "direction"]].rename(
    columns={"direction": "reference_direction"}
)

xgb = read_predictions(
    BASE / "xgboost_predictions.jsonl"
)[["example_id", "direction"]].rename(
    columns={"direction": "xgb_direction"}
)

jev = read_predictions(
    Path.home() / "jev_predictions.jsonl"
)[["example_id", "direction"]].rename(
    columns={"direction": "jev_direction"}
)

df = (
    metadata
    .merge(labels, on="example_id", validate="one_to_one")
    .merge(xgb, on="example_id", validate="one_to_one")
    .merge(jev, on="example_id", validate="one_to_one")
)

assert len(df) == 1540

df["xgb_correct"] = (
    df["xgb_direction"] == df["reference_direction"]
)

df["jev_correct"] = (
    df["jev_direction"] == df["reference_direction"]
)

scenario_results = (
    df.groupby("revenue_change_pct")
    .agg(
        examples=("example_id", "count"),
        xgboost_accuracy=("xgb_correct", "mean"),
        jev_accuracy=("jev_correct", "mean"),
    )
    .reset_index()
    .sort_values("revenue_change_pct")
)

scenario_results["jev_advantage"] = (
    scenario_results["jev_accuracy"]
    - scenario_results["xgboost_accuracy"]
)

display(scenario_results.round(4))

print(
    "Overall Jev direction advantage:",
    round(
        df["jev_correct"].mean()
        - df["xgb_correct"].mean(),
        4,
    )
)


,revenue_change_pct,examples,xgboost_accuracy,jev_accuracy,jev_advantage
0,-65,308,1.0000,1.0,0.0000
1,-35,308,1.0000,1.0,0.0000
2,-15,308,0.4545,0.0,-0.4545
3,0,308,0.0000,1.0,1.0000
4,75,308,1.0000,1.0,0.0000


Overall Jev direction advantage: 0.1091


In [49]:

# STEP 49 — Inspect direction predictions by scenario

for pct in [-15, 0]:
    subset = df[df["revenue_change_pct"] == pct]

    print(f"\n===== Revenue change: {pct}% =====")
    print("Reference labels:")
    print(subset["reference_direction"].value_counts().to_string())

    print("\nJev predictions:")
    print(subset["jev_direction"].value_counts().to_string())

    print("\nXGBoost predictions:")
    print(subset["xgb_direction"].value_counts().to_string())



===== Revenue change: -15% =====
Reference labels:
reference_direction
neutral    308

Jev predictions:
jev_direction
negative    308

XGBoost predictions:
xgb_direction
negative    168
neutral     140

===== Revenue change: 0% =====
Reference labels:
reference_direction
neutral    308

Jev predictions:
jev_direction
neutral    308

XGBoost predictions:
xgb_direction
negative    308


In [50]:

# STEP 50 — Independent sign-based direction evaluation
# Keeps original benchmark labels unchanged.

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

direction_eval = df.copy()

direction_eval["sign_direction"] = np.select(
    [
        direction_eval["revenue_change_pct"] < 0,
        direction_eval["revenue_change_pct"] > 0,
    ],
    ["negative", "positive"],
    default="neutral",
)

results = pd.DataFrame([
    {
        "model": "XGBoost",
        "direction_accuracy": accuracy_score(
            direction_eval["sign_direction"],
            direction_eval["xgb_direction"],
        ),
    },
    {
        "model": "Jev System One",
        "direction_accuracy": accuracy_score(
            direction_eval["sign_direction"],
            direction_eval["jev_direction"],
        ),
    },
])

display(results.round(4))

output = (
    Path("cross_company_benchmark")
    / "sign_based_direction_comparison.csv"
)

results.to_csv(output, index=False)

print("Saved:", output)
print("Original policy labels remain unchanged.")


,model,direction_accuracy
0,XGBoost,0.7091
1,Jev System One,1.0000


Saved: cross_company_benchmark/sign_based_direction_comparison.csv
Original policy labels remain unchanged.


In [51]:
import json

print(json.dumps(multi_model_inputs[0], indent=2, default=str)[:6000])

{
  "event_date": "2026-10-06",
  "old_state": {
    "outstanding_ar": 192.38,
    "overdue_ar": 87.8,
    "overdue_invoice_count": 1,
    "max_days_overdue": 1,
    "recent_30d_invoice_count": 5,
    "previous_30d_invoice_count": 5,
    "recent_30d_revenue": 3376.27,
    "previous_30d_revenue": 2692.67
  },
  "previous_event": {
    "event_type": "invoice_created",
    "event_date": "2026-10-02",
    "amount": 550.4,
    "description": "Any shortages should be reported within five days of delivery."
  },
  "new_event": {
    "event_type": "financial_trend_observed",
    "event_date": "2026-10-06",
    "amount": 0.0,
    "description": "Hypothetical purchasing-trend report comparing a historical 30-day baseline with a simulated subsequent 30-day period. The QuickBooks financial states represent historical accounting records and are not updated by this report.",
    "observation": {
      "baseline_30d_revenue": 2692.67,
      "observed_30d_revenue": 942.43,
      "revenue_change_pct": 

In [52]:

import inspect

for name in ["build_frozen_features", "rules_detector_v1"]:
    fn = globals()[name]
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    try:
        print(inspect.getsource(fn)[:12000])
    except (OSError, TypeError):
        print("Source unavailable:", fn)

print("\nXGBoost feature names:")
for name in ["meaningful_model", "escalation_model", "direction_model"]:
    model = globals()[name]
    print(name, getattr(model, "feature_names_in_", "not available"))



==================== build_frozen_features ====================
def build_frozen_features(records):
    extracted = [extract_xgb_features(x) for x in records]

    texts = [text for text, numeric in extracted]
    numeric = np.asarray(
        [values for text, values in extracted],
        dtype=np.float32
    )

    X = hstack(
        [
            tfidf_vectorizer.transform(texts),
            csr_matrix(numeric)
        ],
        format="csr"
    )

    assert numeric.shape[1] == 26
    assert X.shape[1] == 1434

    return X


==================== rules_detector_v1 ====================
def rules_detector_v1(record):
    event = record["new_event"]
    event_type = event["event_type"]

    result = {
        "meaningful_change": False,
        "direction": "neutral",
        "escalation_needed": False,
    }

    if event_type == "financial_trend_observed":
        change = float(event["observation"]["revenue_change_pct"])

        if change <= -50:
            return {
        

In [53]:

import inspect

print(inspect.getsource(extract_xgb_features))


def extract_xgb_features(record):
    event = record["new_event"]
    old = record["old_state"]
    current = record["current_state"]
    impact = record["event_impact"]
    observation = event.get("observation", {})

    text = event.get("description", "")

    financial_keys = [
        "outstanding_ar",
        "overdue_ar",
        "overdue_invoice_count",
        "max_days_overdue",
        "recent_30d_invoice_count",
        "recent_30d_revenue",
        "previous_30d_invoice_count",
        "previous_30d_revenue",
    ]

    impact_keys = [
        "outstanding_ar_delta",
        "overdue_ar_delta",
        "recent_30d_invoice_count_delta",
        "recent_30d_revenue_delta",
    ]

    numeric = []

    for state in [old, current]:
        numeric.extend(
            float(state.get(key, 0) or 0)
            for key in financial_keys
        )

    numeric.extend(
        float(impact.get(key, 0) or 0)
        for key in impact_keys
    )

    numeric.extend([
        float(eve

In [54]:

import pandas as pd

impact_keys = [
    "outstanding_ar_delta",
    "overdue_ar_delta",
    "recent_30d_invoice_count_delta",
    "recent_30d_revenue_delta",
]

rows = []

for record in multi_model_inputs:
    impact = record["event_impact"]

    rows.append({
        key: float(impact.get(key, 0) or 0)
        for key in impact_keys
    })

impact_df = pd.DataFrame(rows)

print("Total examples:", len(impact_df))
print("\nExamples with nonzero signal changes:")

for key in impact_keys:
    count = (impact_df[key] != 0).sum()
    print(f"{key}: {count}")

print(
    "\nExamples with any nonzero event-impact delta:",
    (impact_df != 0).any(axis=1).sum()
)


Total examples: 1540

Examples with nonzero signal changes:
outstanding_ar_delta: 0
overdue_ar_delta: 0
recent_30d_invoice_count_delta: 0
recent_30d_revenue_delta: 0

Examples with any nonzero event-impact delta: 0


In [55]:

import pandas as pd

financial_keys = [
    "outstanding_ar",
    "overdue_ar",
    "overdue_invoice_count",
    "max_days_overdue",
    "recent_30d_invoice_count",
    "previous_30d_invoice_count",
    "recent_30d_revenue",
    "previous_30d_revenue",
]

states = pd.DataFrame([
    {
        key: float(record["old_state"].get(key, 0) or 0)
        for key in financial_keys
    }
    for record in multi_model_inputs
])

# Deduplicate repeated customer scenarios using example metadata.
states["customer_key"] = (
    multi_model_df["company"].astype(str).to_numpy()
    + "::"
    + multi_model_df["customer_idx"].astype(str).to_numpy()
)

states = states.drop_duplicates("customer_key")

print("Unique customers:", len(states))

display(
    states[financial_keys]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .round(2)
)


Unique customers: 308


,outstanding_ar,overdue_ar,overdue_invoice_count,max_days_overdue,recent_30d_invoice_count,previous_30d_invoice_count,recent_30d_revenue,previous_30d_revenue
count,308.00,308.00,308.00,308.00,308.00,308.00,308.00,308.00
mean,2939.42,877.27,1.35,9.23,4.39,4.14,2626.43,2774.32
std,5400.86,4116.69,1.19,18.68,1.80,1.50,2271.17,4158.84
min,0.00,0.00,0.00,0.00,0.00,2.00,0.00,224.00
25%,492.28,0.00,0.00,0.00,3.00,3.00,1149.35,1055.66
50%,1431.86,136.86,1.00,5.00,4.00,4.00,2101.14,1808.12
75%,3409.65,550.84,2.00,11.25,5.00,5.00,3440.23,3104.39
95%,9385.91,2164.12,4.00,20.30,7.00,7.00,7179.13,6974.70
max,61632.52,45675.88,5.00,221.00,9.00,8.00,15956.64,41353.84


In [56]:

# STEP 53 — Freeze multi-signal benchmark specification
# No model calls. No synthetic examples generated yet.

import json
import hashlib
from pathlib import Path

BASE = Path("cross_company_benchmark")
BASE.mkdir(parents=True, exist_ok=True)

scenarios = [
    {
        "scenario_id": "A_stable",
        "revenue_change_pct": 0,
        "invoice_count_change_pct": 0,
        "overdue_ar_change": "unchanged",
        "overdue_age_change_days": 0,
        "meaningful_change": False,
        "direction": "neutral",
        "escalation_needed": False,
    },
    {
        "scenario_id": "B_purchasing_slowdown",
        "revenue_change_pct": -35,
        "invoice_count_change_pct": -40,
        "overdue_ar_change": "unchanged",
        "overdue_age_change_days": 0,
        "meaningful_change": True,
        "direction": "negative",
        "escalation_needed": False,
    },
    {
        "scenario_id": "C_payment_warning",
        "revenue_change_pct": 0,
        "invoice_count_change_pct": 0,
        "overdue_ar_change": "increase",
        "overdue_age_change_days": 20,
        "meaningful_change": True,
        "direction": "negative",
        "escalation_needed": False,
    },
    {
        "scenario_id": "D_reinforcing_warnings",
        "revenue_change_pct": -35,
        "invoice_count_change_pct": -40,
        "overdue_ar_change": "increase",
        "overdue_age_change_days": 20,
        "meaningful_change": True,
        "direction": "negative",
        "escalation_needed": True,
    },
    {
        "scenario_id": "E_improving_signals",
        "revenue_change_pct": 40,
        "invoice_count_change_pct": 25,
        "overdue_ar_change": "decrease",
        "overdue_age_change_days": -5,
        "meaningful_change": True,
        "direction": "positive",
        "escalation_needed": False,
    },
    {
        "scenario_id": "F_conflicting_signals",
        "revenue_change_pct": 40,
        "invoice_count_change_pct": 0,
        "overdue_ar_change": "increase",
        "overdue_age_change_days": 20,
        "meaningful_change": True,
        "direction": "neutral",
        "escalation_needed": True,
    },
]

path = BASE / "multi_signal_scenario_spec.json"

serialized = json.dumps(
    scenarios,
    indent=2,
    sort_keys=True,
) + "\n"

if path.exists():
    assert path.read_text(encoding="utf-8") == serialized, (
        "Existing scenario specification differs. "
        "Do not overwrite a frozen specification."
    )
    print("Existing specification verified.")
else:
    path.write_text(serialized, encoding="utf-8")
    print("New scenario specification frozen.")

digest = hashlib.sha256(path.read_bytes()).hexdigest()

print("Scenarios:", len(scenarios))
print("Customers:", 308)
print("Planned examples:", 308 * len(scenarios))
print("SHA-256:", digest)
print("Saved:", path)


New scenario specification frozen.
Scenarios: 6
Customers: 308
Planned examples: 1848
SHA-256: d14be1bb5ea637264fc68572af81448e5462e9f36c710ac61fa6a72f24e0f625
Saved: cross_company_benchmark/multi_signal_scenario_spec.json


In [57]:

# STEP 54 — Generate frozen multi-signal benchmark inputs

import json
import copy
import hashlib
from pathlib import Path

BASE = Path("cross_company_benchmark")

scenarios = json.loads(
    (BASE / "multi_signal_scenario_spec.json").read_text(
        encoding="utf-8"
    )
)

assert len(scenarios) == 6
assert len(multi_model_inputs) == len(multi_model_df) == 1540

# One original record per customer, in deterministic order.
customers = {}

for record, row in zip(
    multi_model_inputs,
    multi_model_df.itertuples(index=False),
):
    customer_key = f"{row.company}::{row.customer_idx}"
    customers.setdefault(customer_key, record)

assert len(customers) == 308


def apply_scenario(original, scenario, example_id):
    record = copy.deepcopy(original)

    old = record["old_state"]
    current = record["current_state"]

    # Start from the same financial baseline.
    current.clear()
    current.update(copy.deepcopy(old))

    revenue_pct = scenario["revenue_change_pct"]
    count_pct = scenario["invoice_count_change_pct"]

    # Simulated next-period purchasing activity.
    baseline_revenue = float(
        old["recent_30d_revenue"]
    )
    observed_revenue = round(
        baseline_revenue * (1 + revenue_pct / 100),
        2,
    )

    baseline_count = int(
        old["recent_30d_invoice_count"]
    )
    observed_count = max(
        0,
        round(baseline_count * (1 + count_pct / 100)),
    )

    current["previous_30d_revenue"] = baseline_revenue
    current["recent_30d_revenue"] = observed_revenue
    current["previous_30d_invoice_count"] = baseline_count
    current["recent_30d_invoice_count"] = observed_count

    # Simulated receivables changes.
    old_overdue = float(old["overdue_ar"])
    old_outstanding = float(old["outstanding_ar"])
    overdue_mode = scenario["overdue_ar_change"]

    if overdue_mode == "increase":
        new_overdue = round(
            old_overdue + max(150, baseline_revenue * 0.20),
            2,
        )
    elif overdue_mode == "decrease":
        new_overdue = round(old_overdue * 0.5, 2)
    else:
        new_overdue = old_overdue

    current["overdue_ar"] = new_overdue
    current["outstanding_ar"] = round(
        max(
            new_overdue,
            old_outstanding + (new_overdue - old_overdue),
        ),
        2,
    )

    age_delta = scenario["overdue_age_change_days"]
    current["max_days_overdue"] = max(
        0,
        int(old["max_days_overdue"]) + age_delta,
    )

    if overdue_mode == "increase":
        current["overdue_invoice_count"] = (
            int(old["overdue_invoice_count"]) + 1
        )
    elif overdue_mode == "decrease":
        current["overdue_invoice_count"] = max(
            0,
            int(old["overdue_invoice_count"]) - 1,
        )

    # Maintain consistency: no overdue age if no overdue invoices.
    if current["overdue_invoice_count"] == 0:
        current["max_days_overdue"] = 0

    record["event_impact"] = {
        "outstanding_ar_delta": round(
            current["outstanding_ar"] - old_outstanding, 2
        ),
        "overdue_ar_delta": round(
            current["overdue_ar"] - old_overdue, 2
        ),
        "recent_30d_invoice_count_delta": (
            observed_count - baseline_count
        ),
        "recent_30d_revenue_delta": round(
            observed_revenue - baseline_revenue, 2
        ),
    }

    record["new_event"] = {
        "event_type": "financial_trend_observed",
        "event_date": record["event_date"],
        "amount": 0.0,
        "description": (
            "Hypothetical multi-signal financial trend report "
            "comparing a customer baseline with a simulated "
            "subsequent 30-day period. The observations are "
            "synthetic and do not modify actual accounting records."
        ),
        "observation": {
            "report_type": "hypothetical_multi_signal_trend",
            "baseline_30d_revenue": baseline_revenue,
            "observed_30d_revenue": observed_revenue,
            "revenue_change_pct": revenue_pct,
            "baseline_30d_invoice_count": baseline_count,
            "observed_30d_invoice_count": observed_count,
            "scenario_id": scenario["scenario_id"],
        },
    }

    return {
        "example_id": example_id,
        "input": record,
        "reference": {
            "meaningful_change": scenario["meaningful_change"],
            "direction": scenario["direction"],
            "escalation_needed": scenario["escalation_needed"],
        },
    }


examples = []

for customer_key, original in customers.items():
    for scenario in scenarios:
        example_id = (
            f"multi_signal::{customer_key}::"
            f"{scenario['scenario_id']}"
        )

        examples.append(
            apply_scenario(original, scenario, example_id)
        )

assert len(examples) == 1848
assert len(set(x["example_id"] for x in examples)) == 1848

output_path = BASE / "multi_signal_examples.jsonl"

serialized = "".join(
    json.dumps(x, sort_keys=True) + "\n"
    for x in examples
)

if output_path.exists():
    assert output_path.read_text(encoding="utf-8") == serialized, (
        "Frozen benchmark file differs. Do not overwrite."
    )
    print("Existing benchmark verified.")
else:
    output_path.write_text(serialized, encoding="utf-8")
    print("New multi-signal benchmark saved.")

digest = hashlib.sha256(
    output_path.read_bytes()
).hexdigest()

print("Total examples:", len(examples))
print("Unique customers:", len(customers))
print("Scenarios:", len(scenarios))
print("SHA-256:", digest)
print("Saved:", output_path)
print("PASS — Multi-signal benchmark generated.")


New multi-signal benchmark saved.
Total examples: 1848
Unique customers: 308
Scenarios: 6
SHA-256: 1f7e02e9682aaf4b4de4bd4daa38462955ee5902fb8a03d568879c22ead1f529
Saved: cross_company_benchmark/multi_signal_examples.jsonl
PASS — Multi-signal benchmark generated.


In [58]:

# STEP 55 — Remove label leakage and validate benchmark
# No model calls or API costs.

import json
import copy
import hashlib
from pathlib import Path
from collections import Counter

BASE = Path("cross_company_benchmark")
source = BASE / "multi_signal_examples.jsonl"

with source.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(examples) == 1848

clean_inputs = []
reference_labels = []
scenario_counts = Counter()

for example in examples:
    example_id = example["example_id"]
    record = copy.deepcopy(example["input"])
    reference = example["reference"]

    # Remove scenario identifier from the model input.
    observation = record["new_event"].get("observation", {})
    scenario_id = observation.pop("scenario_id", None)

    assert scenario_id is not None
    scenario_counts[scenario_id] += 1

    # Basic financial-state checks.
    old = record["old_state"]
    current = record["current_state"]
    impact = record["event_impact"]

    assert current["outstanding_ar"] >= 0
    assert current["overdue_ar"] >= 0
    assert current["overdue_ar"] <= current["outstanding_ar"] + 0.01
    assert current["overdue_invoice_count"] >= 0
    assert current["max_days_overdue"] >= 0
    assert current["recent_30d_invoice_count"] >= 0

    for key, state_key in [
        ("outstanding_ar_delta", "outstanding_ar"),
        ("overdue_ar_delta", "overdue_ar"),
        (
            "recent_30d_invoice_count_delta",
            "recent_30d_invoice_count",
        ),
        ("recent_30d_revenue_delta", "recent_30d_revenue"),
    ]:
        expected = (
            float(current[state_key])
            - float(old[state_key])
        )

        assert abs(
            float(impact[key]) - expected
        ) < 0.02, (example_id, key)

    clean_inputs.append({
        "example_id": example_id,
        "input": record,
    })

    reference_labels.append({
        "example_id": example_id,
        **reference,
    })

assert len(clean_inputs) == len(reference_labels) == 1848

assert len({
    x["example_id"] for x in clean_inputs
}) == 1848

def save_frozen_jsonl(filename, rows):
    path = BASE / filename

    content = "".join(
        json.dumps(row, sort_keys=True) + "\n"
        for row in rows
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"{filename} already exists with different content."
        )
    else:
        path.write_text(content, encoding="utf-8")

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    return path, digest


inputs_path, inputs_hash = save_frozen_jsonl(
    "multi_signal_model_inputs.jsonl",
    clean_inputs,
)

labels_path, labels_hash = save_frozen_jsonl(
    "multi_signal_reference_labels.jsonl",
    reference_labels,
)

print("========== STEP 55 ==========")
print("Clean inputs:", len(clean_inputs))
print("Reference labels:", len(reference_labels))
print("Scenario counts:", dict(sorted(scenario_counts.items())))
print("Inputs SHA-256:", inputs_hash)
print("Labels SHA-256:", labels_hash)
print("Inputs saved:", inputs_path)
print("Labels saved:", labels_path)
print("PASS — Model inputs separated from labels.")


========== STEP 55 ==========
Clean inputs: 1848
Reference labels: 1848
Scenario counts: {'A_stable': 308, 'B_purchasing_slowdown': 308, 'C_payment_warning': 308, 'D_reinforcing_warnings': 308, 'E_improving_signals': 308, 'F_conflicting_signals': 308}
Inputs SHA-256: 0fb6151ca081af373a1dabcb2f2e10e5efed6d1d9510fdd904751ca9ea637cff
Labels SHA-256: 35e8ab6ae571ba6d37baa783042516b868c612ea36e67d265127f87f2e1ae585
Inputs saved: cross_company_benchmark/multi_signal_model_inputs.jsonl
Labels saved: cross_company_benchmark/multi_signal_reference_labels.jsonl
PASS — Model inputs separated from labels.


In [59]:

# STEP 56 — Frozen local-model predictions
# No training, no API calls, no label access.

import json
import time
import hashlib
import numpy as np
from pathlib import Path

BASE = Path("cross_company_benchmark")
INPUT_PATH = BASE / "multi_signal_model_inputs.jsonl"

with INPUT_PATH.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(examples) == 1848

ids = [x["example_id"] for x in examples]
records = [x["input"] for x in examples]

assert len(set(ids)) == 1848
assert all(
    "scenario_id" not in
    record["new_event"].get("observation", {})
    for record in records
)

# Rules V1 — frozen
t0 = time.perf_counter()
rules_raw = [rules_detector_v1(record) for record in records]
rules_time = time.perf_counter() - t0

# XGBoost — frozen
t0 = time.perf_counter()
X = build_frozen_features(records)

xgb_meaningful = meaningful_model.predict(X)
xgb_escalation = escalation_model.predict(X)
xgb_direction = direction_model.predict(X)

xgb_time = time.perf_counter() - t0

direction_mapping = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

xgb_raw = [
    {
        "meaningful_change": bool(xgb_meaningful[i]),
        "direction": direction_mapping[int(xgb_direction[i])],
        "escalation_needed": bool(xgb_escalation[i]),
    }
    for i in range(len(records))
]

def save_predictions(filename, predictions):
    rows = [
        {
            "example_id": ids[i],
            "meaningful_change": bool(p["meaningful_change"]),
            "direction": str(p["direction"]).lower(),
            "escalation_needed": bool(p["escalation_needed"]),
        }
        for i, p in enumerate(predictions)
    ]

    path = BASE / filename
    content = "".join(
        json.dumps(row, sort_keys=True) + "\n"
        for row in rows
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"Existing {filename} differs. "
            "Do not silently overwrite."
        )
    else:
        path.write_text(content, encoding="utf-8")

    return path, hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

rules_path, rules_hash = save_predictions(
    "multi_signal_rules_predictions.jsonl",
    rules_raw,
)

xgb_path, xgb_hash = save_predictions(
    "multi_signal_xgboost_predictions.jsonl",
    xgb_raw,
)

print("========== STEP 56 ==========")
print("Examples:", len(records))
print("Feature matrix:", X.shape)
print("Rules predictions:", len(rules_raw))
print("XGBoost predictions:", len(xgb_raw))
print("Rules inference seconds:", round(rules_time, 4))
print("XGBoost inference seconds:", round(xgb_time, 4))
print("Rules SHA-256:", rules_hash)
print("XGBoost SHA-256:", xgb_hash)
print("Rules saved:", rules_path)
print("XGBoost saved:", xgb_path)
print("PASS — Frozen baseline predictions complete.")


========== STEP 56 ==========
Examples: 1848
Feature matrix: (1848, 1434)
Rules predictions: 1848
XGBoost predictions: 1848
Rules inference seconds: 0.0008
XGBoost inference seconds: 0.0391
Rules SHA-256: 6ef206ecf60c9e7a89af3baa2dfc7c4d55c19fee894763c40dc24f37dac85b0d
XGBoost SHA-256: 36c582009dc79d36a3ba49858df848df3e3939a077fc96b9616535894348d5bf
Rules saved: cross_company_benchmark/multi_signal_rules_predictions.jsonl
XGBoost saved: cross_company_benchmark/multi_signal_xgboost_predictions.jsonl
PASS — Frozen baseline predictions complete.


In [60]:

# STEP 57 — Evaluate frozen baselines on multi-signal scenarios
# No model calls. No API costs.

import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

BASE = Path("cross_company_benchmark")

def load_jsonl(filename):
    with (BASE / filename).open(encoding="utf-8") as f:
        return pd.DataFrame(
            json.loads(line)
            for line in f
            if line.strip()
        )

labels = load_jsonl("multi_signal_reference_labels.jsonl")

models = {
    "Rules V1": "multi_signal_rules_predictions.jsonl",
    "XGBoost": "multi_signal_xgboost_predictions.jsonl",
}

results = []
scenario_results = []

for model_name, filename in models.items():
    predictions = load_jsonl(filename)

    assert len(labels) == len(predictions) == 1848
    assert labels["example_id"].is_unique
    assert predictions["example_id"].is_unique

    merged = labels.merge(
        predictions,
        on="example_id",
        suffixes=("_true", "_pred"),
        validate="one_to_one",
    )

    assert len(merged) == 1848

    result = {"model": model_name, "examples": len(merged)}

    for task, prefix in [
        ("meaningful_change", "meaningful"),
        ("escalation_needed", "escalation"),
    ]:
        truth = merged[f"{task}_true"].astype(bool)
        pred = merged[f"{task}_pred"].astype(bool)

        result[f"{prefix}_precision"] = precision_score(
            truth, pred, zero_division=0
        )
        result[f"{prefix}_recall"] = recall_score(
            truth, pred, zero_division=0
        )
        result[f"{prefix}_f1"] = f1_score(
            truth, pred, zero_division=0
        )

    result["direction_accuracy"] = accuracy_score(
        merged["direction_true"],
        merged["direction_pred"],
    )

    results.append(result)

    # Scenario-specific evaluation
    merged["scenario"] = merged["example_id"].str.rsplit(
        "::", n=1
    ).str[-1]

    for scenario, group in merged.groupby("scenario"):
        scenario_results.append({
            "model": model_name,
            "scenario": scenario,
            "examples": len(group),
            "meaningful_accuracy": accuracy_score(
                group["meaningful_change_true"],
                group["meaningful_change_pred"],
            ),
            "escalation_accuracy": accuracy_score(
                group["escalation_needed_true"],
                group["escalation_needed_pred"],
            ),
            "direction_accuracy": accuracy_score(
                group["direction_true"],
                group["direction_pred"],
            ),
        })

metrics_df = pd.DataFrame(results)
scenario_df = pd.DataFrame(scenario_results)

display(metrics_df.round(4))

print("\n========== SCENARIO BREAKDOWN ==========")
display(scenario_df.round(4))

metrics_path = BASE / "multi_signal_baseline_metrics.csv"
scenario_path = BASE / "multi_signal_scenario_metrics.csv"

metrics_df.to_csv(metrics_path, index=False)
scenario_df.to_csv(scenario_path, index=False)

print("\nSaved:", metrics_path)
print("Saved:", scenario_path)
print("PASS — Multi-signal baseline evaluation complete.")


,model,examples,meaningful_precision,meaningful_recall,meaningful_f1,escalation_precision,escalation_recall,escalation_f1,direction_accuracy
0,Rules V1,1848,1.0,0.4,0.5714,0.0,0.0,0.0,0.6667
1,XGBoost,1848,0.0,0.0,0.0000,0.0,0.0,0.0,0.6667



========== SCENARIO BREAKDOWN ==========


,model,scenario,examples,meaningful_accuracy,escalation_accuracy,direction_accuracy
0,Rules V1,A_stable,308,1.0,1.0,1.0
1,Rules V1,B_purchasing_slowdown,308,1.0,1.0,1.0
2,Rules V1,C_payment_warning,308,0.0,1.0,0.0
3,Rules V1,D_reinforcing_warnings,308,1.0,0.0,1.0
4,Rules V1,E_improving_signals,308,0.0,1.0,0.0
5,Rules V1,F_conflicting_signals,308,0.0,0.0,1.0
6,XGBoost,A_stable,308,1.0,1.0,1.0
7,XGBoost,B_purchasing_slowdown,308,0.0,1.0,1.0
8,XGBoost,C_payment_warning,308,0.0,1.0,0.0
9,XGBoost,D_reinforcing_warnings,308,0.0,0.0,1.0



Saved: cross_company_benchmark/multi_signal_baseline_metrics.csv
Saved: cross_company_benchmark/multi_signal_scenario_metrics.csv
PASS — Multi-signal baseline evaluation complete.


In [61]:

import json
import pandas as pd
from pathlib import Path

BASE = Path("cross_company_benchmark")

for model, filename in [
    ("Rules V1", "multi_signal_rules_predictions.jsonl"),
    ("XGBoost", "multi_signal_xgboost_predictions.jsonl"),
]:
    with (BASE / filename).open(encoding="utf-8") as f:
        predictions = pd.DataFrame(
            json.loads(line)
            for line in f
            if line.strip()
        )

    print(f"\n========== {model} ==========")

    for column in [
        "meaningful_change",
        "escalation_needed",
        "direction",
    ]:
        print(f"\n{column}:")
        print(
            predictions[column]
            .value_counts(dropna=False)
            .to_string()
        )



========== Rules V1 ==========

meaningful_change:
meaningful_change
False    1232
True      616

escalation_needed:
escalation_needed
False    1848

direction:
direction
neutral     1232
negative     616

========== XGBoost ==========

meaningful_change:
meaningful_change
False    1848

escalation_needed:
escalation_needed
False    1848

direction:
direction
neutral     1232
negative     616


In [62]:

import json
from pathlib import Path

path = Path(
    "cross_company_benchmark/multi_signal_model_inputs.jsonl"
)

with path.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# First customer, all six scenarios.
for example in examples[:6]:
    record = example["input"]

    print("\n" + "=" * 60)
    print("Example:", example["example_id"])

    print("\nOld state:")
    print(json.dumps(record["old_state"], indent=2))

    print("\nCurrent state:")
    print(json.dumps(record["current_state"], indent=2))

    print("\nEvent impact:")
    print(json.dumps(record["event_impact"], indent=2))

    print("\nObservation:")
    print(json.dumps(
        record["new_event"]["observation"],
        indent=2,
    ))



Example: multi_signal::company-001::1::A_stable

Old state:
{
  "max_days_overdue": 1,
  "outstanding_ar": 192.38,
  "overdue_ar": 87.8,
  "overdue_invoice_count": 1,
  "previous_30d_invoice_count": 5,
  "previous_30d_revenue": 2692.67,
  "recent_30d_invoice_count": 5,
  "recent_30d_revenue": 3376.27
}

Current state:
{
  "max_days_overdue": 1,
  "outstanding_ar": 192.38,
  "overdue_ar": 87.8,
  "overdue_invoice_count": 1,
  "previous_30d_invoice_count": 5,
  "previous_30d_revenue": 3376.27,
  "recent_30d_invoice_count": 5,
  "recent_30d_revenue": 3376.27
}

Event impact:
{
  "outstanding_ar_delta": 0.0,
  "overdue_ar_delta": 0.0,
  "recent_30d_invoice_count_delta": 0,
  "recent_30d_revenue_delta": 0.0
}

Observation:
{
  "baseline_30d_invoice_count": 5,
  "baseline_30d_revenue": 3376.27,
  "observed_30d_invoice_count": 5,
  "observed_30d_revenue": 3376.27,
  "report_type": "hypothetical_multi_signal_trend",
  "revenue_change_pct": 0
}

Example: multi_signal::company-001::1::B_purchas

In [63]:

# STEP 60 — Audit synthetic financial states
# No model calls or API costs.

import json
from pathlib import Path
from collections import Counter

path = Path(
    "cross_company_benchmark/multi_signal_model_inputs.jsonl"
)

with path.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

issues = Counter()
examples_with_issues = []

for example in examples:
    record = example["input"]
    old = record["old_state"]
    current = record["current_state"]
    obs = record["new_event"]["observation"]

    found = []

    if (
        current["overdue_ar"] > 0
        and current["overdue_invoice_count"] == 0
    ):
        found.append("overdue_balance_without_overdue_invoice")

    if (
        current["overdue_invoice_count"] > 0
        and current["max_days_overdue"] == 0
    ):
        found.append("overdue_invoice_without_overdue_age")

    if (
        current["recent_30d_invoice_count"] == 0
        and current["recent_30d_revenue"] > 0
    ):
        found.append("revenue_without_recent_invoices")

    if (
        current["overdue_ar"]
        > current["outstanding_ar"] + 0.01
    ):
        found.append("overdue_exceeds_outstanding")

    if (
        obs["observed_30d_revenue"] == 0
        and obs["observed_30d_invoice_count"] > 0
    ):
        found.append("invoices_without_observed_revenue")

    for issue in found:
        issues[issue] += 1

    if found:
        examples_with_issues.append({
            "example_id": example["example_id"],
            "issues": found,
        })

print("========== STEP 60 AUDIT ==========")
print("Total examples:", len(examples))
print("Examples with issues:", len(examples_with_issues))

print("\nIssue counts:")
for issue, count in issues.most_common():
    print(f"{issue}: {count}")

print("\nFirst 5 affected examples:")
for item in examples_with_issues[:5]:
    print(item)

if not examples_with_issues:
    print("\nPASS — No flagged consistency issues.")
else:
    print("\nREVIEW REQUIRED — Fix flagged issues before Jev.")


========== STEP 60 AUDIT ==========
Total examples: 1848
Examples with issues: 114

Issue counts:
overdue_balance_without_overdue_invoice: 108
overdue_invoice_without_overdue_age: 6

First 5 affected examples:
{'example_id': 'multi_signal::company-001::1::E_improving_signals', 'issues': ['overdue_balance_without_overdue_invoice']}
{'example_id': 'multi_signal::company-001::10::E_improving_signals', 'issues': ['overdue_balance_without_overdue_invoice']}
{'example_id': 'multi_signal::company-001::15::E_improving_signals', 'issues': ['overdue_balance_without_overdue_invoice']}
{'example_id': 'multi_signal::company-001::17::E_improving_signals', 'issues': ['overdue_balance_without_overdue_invoice']}
{'example_id': 'multi_signal::company-001::22::E_improving_signals', 'issues': ['overdue_balance_without_overdue_invoice']}

REVIEW REQUIRED — Fix flagged issues before Jev.


In [64]:

# STEP 61 — Repair multi-signal benchmark consistency
# Creates a new version; does not overwrite frozen inputs.
# No model calls or API costs.

import json
import copy
import hashlib
from pathlib import Path
from collections import Counter

BASE = Path("cross_company_benchmark")
SOURCE = BASE / "multi_signal_model_inputs.jsonl"
OUTPUT = BASE / "multi_signal_model_inputs_v2.jsonl"

with SOURCE.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(examples) == 1848

corrected = []
repair_counts = Counter()

for example in examples:
    item = copy.deepcopy(example)
    record = item["input"]

    old = record["old_state"]
    current = record["current_state"]

    overdue_ar = float(current["overdue_ar"])
    overdue_count = int(current["overdue_invoice_count"])
    overdue_age = int(current["max_days_overdue"])

    if overdue_ar > 0:
        if overdue_count == 0:
            current["overdue_invoice_count"] = 1
            repair_counts["fixed_overdue_invoice_count"] += 1

        if overdue_age == 0:
            current["max_days_overdue"] = 1
            repair_counts["fixed_overdue_age"] += 1

    else:
        if overdue_count != 0:
            current["overdue_invoice_count"] = 0
            repair_counts["cleared_overdue_invoice_count"] += 1

        if overdue_age != 0:
            current["max_days_overdue"] = 0
            repair_counts["cleared_overdue_age"] += 1

    # Keep the event-impact fields synchronized with the states.
    impact_mapping = {
        "outstanding_ar_delta": "outstanding_ar",
        "overdue_ar_delta": "overdue_ar",
        "recent_30d_invoice_count_delta":
            "recent_30d_invoice_count",
        "recent_30d_revenue_delta":
            "recent_30d_revenue",
    }

    for impact_key, state_key in impact_mapping.items():
        delta = (
            float(current[state_key])
            - float(old[state_key])
        )

        if "count" in impact_key:
            record["event_impact"][impact_key] = int(delta)
        else:
            record["event_impact"][impact_key] = round(delta, 2)

    corrected.append(item)

assert len(corrected) == 1848
assert len(set(x["example_id"] for x in corrected)) == 1848

# Re-run the consistency checks.
remaining_issues = []

for item in corrected:
    current = item["input"]["current_state"]

    if current["overdue_ar"] > 0:
        if (
            current["overdue_invoice_count"] < 1
            or current["max_days_overdue"] < 1
        ):
            remaining_issues.append(item["example_id"])

    else:
        if (
            current["overdue_invoice_count"] != 0
            or current["max_days_overdue"] != 0
        ):
            remaining_issues.append(item["example_id"])

assert not remaining_issues, remaining_issues[:5]

serialized = "".join(
    json.dumps(item, sort_keys=True) + "\n"
    for item in corrected
)

if OUTPUT.exists():
    assert OUTPUT.read_text(encoding="utf-8") == serialized, (
        "Existing v2 file differs. Do not overwrite."
    )
    print("Existing corrected benchmark verified.")
else:
    OUTPUT.write_text(serialized, encoding="utf-8")
    print("Corrected benchmark saved.")

digest = hashlib.sha256(OUTPUT.read_bytes()).hexdigest()

print("\n========== STEP 61 ==========")
print("Total examples:", len(corrected))
print("Repairs:", dict(repair_counts))
print("Remaining flagged inconsistencies:", len(remaining_issues))
print("SHA-256:", digest)
print("Saved:", OUTPUT)
print("PASS — Corrected benchmark v2 ready.")


Corrected benchmark saved.

========== STEP 61 ==========
Total examples: 1848
Repairs: {'fixed_overdue_invoice_count': 108, 'fixed_overdue_age': 114}
Remaining flagged inconsistencies: 0
SHA-256: dc61530e14307db6cc407775e0fe2e2d68e7153055aeeba1df78de771110334b
Saved: cross_company_benchmark/multi_signal_model_inputs_v2.jsonl
PASS — Corrected benchmark v2 ready.


In [65]:

# STEP 62 — Run frozen Rules V1 and XGBoost on corrected v2
# No retraining. No Jev API calls.

import json
import hashlib
import time
from pathlib import Path
from collections import Counter

BASE = Path("cross_company_benchmark")

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

examples = read_jsonl(
    BASE / "multi_signal_model_inputs_v2.jsonl"
)
labels = read_jsonl(
    BASE / "multi_signal_reference_labels.jsonl"
)

assert len(examples) == len(labels) == 1848

ids = [item["example_id"] for item in examples]
label_ids = [item["example_id"] for item in labels]

assert len(set(ids)) == 1848
assert set(ids) == set(label_ids)

records = [item["input"] for item in examples]

assert all(
    "scenario_id" not in
    record["new_event"].get("observation", {})
    for record in records
)

# Frozen Rules V1
t0 = time.perf_counter()
rules_predictions = [
    rules_detector_v1(record)
    for record in records
]
rules_seconds = time.perf_counter() - t0

# Frozen XGBoost
t0 = time.perf_counter()
X = build_frozen_features(records)

meaningful = meaningful_model.predict(X)
escalation = escalation_model.predict(X)
direction = direction_model.predict(X)

direction_map = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

xgb_predictions = [
    {
        "meaningful_change": bool(meaningful[i]),
        "direction": direction_map[int(direction[i])],
        "escalation_needed": bool(escalation[i]),
    }
    for i in range(len(records))
]

xgb_seconds = time.perf_counter() - t0

def save_predictions(filename, predictions):
    rows = []

    for example_id, prediction in zip(ids, predictions):
        rows.append({
            "example_id": example_id,
            "meaningful_change": bool(
                prediction["meaningful_change"]
            ),
            "direction": str(
                prediction["direction"]
            ).lower(),
            "escalation_needed": bool(
                prediction["escalation_needed"]
            ),
        })

    path = BASE / filename

    content = "".join(
        json.dumps(row, sort_keys=True) + "\n"
        for row in rows
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"{filename} already exists with different content."
        )
    else:
        path.write_text(content, encoding="utf-8")

    return path, hashlib.sha256(
        path.read_bytes()
    ).hexdigest(), rows

rules_path, rules_hash, rules_rows = save_predictions(
    "multi_signal_rules_predictions_v2.jsonl",
    rules_predictions,
)

xgb_path, xgb_hash, xgb_rows = save_predictions(
    "multi_signal_xgboost_predictions_v2.jsonl",
    xgb_predictions,
)

print("========== STEP 62 ==========")
print("Examples:", len(examples))
print("Feature matrix:", X.shape)
print("Rules inference seconds:", round(rules_seconds, 4))
print("XGBoost inference seconds:", round(xgb_seconds, 4))

for name, rows in [
    ("Rules V1", rules_rows),
    ("XGBoost", xgb_rows),
]:
    print(f"\n{name} prediction distribution:")

    for field in [
        "meaningful_change",
        "escalation_needed",
        "direction",
    ]:
        print(field, dict(Counter(
            row[field] for row in rows
        )))

print("\nRules SHA-256:", rules_hash)
print("XGBoost SHA-256:", xgb_hash)
print("Saved:", rules_path)
print("Saved:", xgb_path)
print("PASS — Frozen v2 baseline predictions complete.")


========== STEP 62 ==========
Examples: 1848
Feature matrix: (1848, 1434)
Rules inference seconds: 0.001
XGBoost inference seconds: 0.0462

Rules V1 prediction distribution:
meaningful_change {False: 1232, True: 616}
escalation_needed {False: 1848}
direction {'neutral': 1232, 'negative': 616}

XGBoost prediction distribution:
meaningful_change {False: 1848}
escalation_needed {False: 1848}
direction {'neutral': 1232, 'negative': 616}

Rules SHA-256: 6ef206ecf60c9e7a89af3baa2dfc7c4d55c19fee894763c40dc24f37dac85b0d
XGBoost SHA-256: 36c582009dc79d36a3ba49858df848df3e3939a077fc96b9616535894348d5bf
Saved: cross_company_benchmark/multi_signal_rules_predictions_v2.jsonl
Saved: cross_company_benchmark/multi_signal_xgboost_predictions_v2.jsonl
PASS — Frozen v2 baseline predictions complete.


In [66]:

# STEP 63 — Audit whether scenario signals are observable
# Read-only. No model calls or API costs.

import json
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("cross_company_benchmark")
PATH = BASE / "multi_signal_model_inputs_v2.jsonl"

with PATH.open(encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(examples) == 1848

issues = Counter()
by_scenario = defaultdict(Counter)
samples = defaultdict(list)

for example in examples:
    example_id = example["example_id"]
    scenario = example_id.rsplit("::", 1)[-1]

    old = example["input"]["old_state"]
    current = example["input"]["current_state"]

    revenue_delta = (
        current["recent_30d_revenue"]
        - old["recent_30d_revenue"]
    )

    invoice_delta = (
        current["recent_30d_invoice_count"]
        - old["recent_30d_invoice_count"]
    )

    overdue_delta = (
        current["overdue_ar"]
        - old["overdue_ar"]
    )

    expected = {
        "A_stable": (0, 0, 0),
        "B_purchasing_slowdown": (-1, -1, 0),
        "C_payment_warning": (0, 0, 1),
        "D_reinforcing_warnings": (-1, -1, 1),
        "E_improving_signals": (1, 1, -1),
        "F_conflicting_signals": (1, 0, 1),
    }

    assert scenario in expected, scenario

    actual = (
        (revenue_delta > 0) - (revenue_delta < 0),
        (invoice_delta > 0) - (invoice_delta < 0),
        (overdue_delta > 0) - (overdue_delta < 0),
    )

    expected_signs = expected[scenario]

    for field, actual_sign, expected_sign in zip(
        ["revenue", "invoice_count", "overdue_ar"],
        actual,
        expected_signs,
    ):
        if actual_sign != expected_sign:
            issue = f"{field}_expected_{expected_sign}_actual_{actual_sign}"

            issues[issue] += 1
            by_scenario[scenario][issue] += 1

            if len(samples[issue]) < 3:
                samples[issue].append({
                    "example_id": example_id,
                    "old": old[
                        "recent_30d_revenue"
                        if field == "revenue"
                        else (
                            "recent_30d_invoice_count"
                            if field == "invoice_count"
                            else "overdue_ar"
                        )
                    ],
                    "new": current[
                        "recent_30d_revenue"
                        if field == "revenue"
                        else (
                            "recent_30d_invoice_count"
                            if field == "invoice_count"
                            else "overdue_ar"
                        )
                    ],
                })

print("========== STEP 63 ==========")
print("Total examples:", len(examples))

print("\nSignal observability issues:")
if issues:
    for issue, count in issues.most_common():
        print(f"{issue}: {count}")
else:
    print("None")

print("\nIssues by scenario:")
for scenario in sorted(by_scenario):
    print(scenario, dict(by_scenario[scenario]))

print("\nSample affected examples:")
for issue, rows in samples.items():
    print(f"\n{issue}")
    for row in rows:
        print(row)

if issues:
    print("\nREVIEW REQUIRED — Some intended signals are not observable.")
else:
    print("\nPASS — All intended signal directions are observable.")


========== STEP 63 ==========
Total examples: 1848

Signal observability issues:
overdue_ar_expected_-1_actual_0: 82
invoice_count_expected_-1_actual_0: 46
invoice_count_expected_1_actual_0: 32
revenue_expected_-1_actual_0: 26
revenue_expected_1_actual_0: 26

Issues by scenario:
B_purchasing_slowdown {'revenue_expected_-1_actual_0': 13, 'invoice_count_expected_-1_actual_0': 23}
D_reinforcing_warnings {'revenue_expected_-1_actual_0': 13, 'invoice_count_expected_-1_actual_0': 23}
E_improving_signals {'overdue_ar_expected_-1_actual_0': 82, 'invoice_count_expected_1_actual_0': 32, 'revenue_expected_1_actual_0': 13}
F_conflicting_signals {'revenue_expected_1_actual_0': 13}

Sample affected examples:

overdue_ar_expected_-1_actual_0
{'example_id': 'multi_signal::company-001::2::E_improving_signals', 'old': 0.0, 'new': 0.0}
{'example_id': 'multi_signal::company-001::19::E_improving_signals', 'old': 0.0, 'new': 0.0}
{'example_id': 'multi_signal::company-002::18::E_improving_signals', 'old': 0.

In [67]:

# STEP 64 — Create v3 benchmark with fully observable customers
# No API calls. Does not overwrite v2 or its labels.

import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("cross_company_benchmark")

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

inputs = read_jsonl(
    BASE / "multi_signal_model_inputs_v2.jsonl"
)
labels = read_jsonl(
    BASE / "multi_signal_reference_labels.jsonl"
)

expected = {
    "A_stable": (0, 0, 0),
    "B_purchasing_slowdown": (-1, -1, 0),
    "C_payment_warning": (0, 0, 1),
    "D_reinforcing_warnings": (-1, -1, 1),
    "E_improving_signals": (1, 1, -1),
    "F_conflicting_signals": (1, 0, 1),
}

def sign(value):
    return (value > 0) - (value < 0)

def customer_key(example_id):
    return example_id.rsplit("::", 1)[0]

def is_observable(item):
    record = item["input"]
    old = record["old_state"]
    new = record["current_state"]

    scenario = item["example_id"].rsplit("::", 1)[-1]

    actual = (
        sign(
            new["recent_30d_revenue"]
            - old["recent_30d_revenue"]
        ),
        sign(
            new["recent_30d_invoice_count"]
            - old["recent_30d_invoice_count"]
        ),
        sign(
            new["overdue_ar"] - old["overdue_ar"]
        ),
    )

    return actual == expected[scenario]

groups = defaultdict(list)

for item in inputs:
    groups[customer_key(item["example_id"])].append(item)

eligible_customers = {
    key
    for key, group in groups.items()
    if (
        len(group) == 6
        and {x["example_id"].rsplit("::", 1)[-1] for x in group}
            == set(expected)
        and all(is_observable(x) for x in group)
    )
}

eligible_ids = {
    item["example_id"]
    for item in inputs
    if customer_key(item["example_id"]) in eligible_customers
}

v3_inputs = [
    item for item in inputs
    if item["example_id"] in eligible_ids
]

v3_labels = [
    item for item in labels
    if item["example_id"] in eligible_ids
]

assert len(v3_inputs) == len(v3_labels)
assert len(v3_inputs) == len(eligible_customers) * 6
assert {
    item["example_id"] for item in v3_inputs
} == {
    item["example_id"] for item in v3_labels
}

def save_immutable(filename, rows):
    path = BASE / filename

    content = "".join(
        json.dumps(row, sort_keys=True) + "\n"
        for row in rows
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"{filename} exists with different content."
        )
    else:
        path.write_text(content, encoding="utf-8")

    return hashlib.sha256(path.read_bytes()).hexdigest()

inputs_hash = save_immutable(
    "multi_signal_model_inputs_v3.jsonl",
    v3_inputs,
)

labels_hash = save_immutable(
    "multi_signal_reference_labels_v3.jsonl",
    v3_labels,
)

scenario_counts = Counter(
    item["example_id"].rsplit("::", 1)[-1]
    for item in v3_inputs
)

print("========== STEP 64 ==========")
print("Original customers:", len(groups))
print("Eligible customers:", len(eligible_customers))
print("Excluded customers:", len(groups) - len(eligible_customers))
print("Original examples:", len(inputs))
print("Eligible examples:", len(v3_inputs))
print("Scenario counts:", dict(sorted(scenario_counts.items())))
print("Inputs SHA-256:", inputs_hash)
print("Labels SHA-256:", labels_hash)
print("PASS — Eligible-customer v3 benchmark frozen.")


========== STEP 64 ==========
Original customers: 308
Eligible customers: 217
Excluded customers: 91
Original examples: 1848
Eligible examples: 1302
Scenario counts: {'A_stable': 217, 'B_purchasing_slowdown': 217, 'C_payment_warning': 217, 'D_reinforcing_warnings': 217, 'E_improving_signals': 217, 'F_conflicting_signals': 217}
Inputs SHA-256: fa3f8045bfbb6cec852b2cd14e669099106043a55c8b49bb21b707cdc9fdc280
Labels SHA-256: 526adc49bd8eb42aec89e7a183f22a3b53ad99b261f61bdffe8edcac3d163ce6
PASS — Eligible-customer v3 benchmark frozen.


In [68]:

# STEP 65 — Evaluate frozen baselines on eligible v3 benchmark
# No retraining. No API calls.

import json
import hashlib
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

BASE = Path("cross_company_benchmark")

def load_jsonl(filename):
    with (BASE / filename).open(encoding="utf-8") as f:
        return pd.DataFrame(
            json.loads(line)
            for line in f
            if line.strip()
        )

labels = load_jsonl(
    "multi_signal_reference_labels_v3.jsonl"
)

inputs = load_jsonl(
    "multi_signal_model_inputs_v3.jsonl"
)

assert len(labels) == len(inputs) == 1302
assert labels["example_id"].is_unique
assert inputs["example_id"].is_unique
assert set(labels["example_id"]) == set(inputs["example_id"])

model_files = {
    "Rules V1": "multi_signal_rules_predictions_v2.jsonl",
    "XGBoost": "multi_signal_xgboost_predictions_v2.jsonl",
}

results = []
scenario_results = []

for model_name, filename in model_files.items():
    predictions = load_jsonl(filename)
    assert predictions["example_id"].is_unique

    merged = labels.merge(
        predictions,
        on="example_id",
        how="left",
        suffixes=("_true", "_pred"),
        validate="one_to_one",
        indicator=True,
    )

    assert len(merged) == 1302
    assert (merged["_merge"] == "both").all(), (
        f"Missing predictions for {model_name}"
    )

    result = {
        "model": model_name,
        "examples": len(merged),
    }

    for task, prefix in [
        ("meaningful_change", "meaningful"),
        ("escalation_needed", "escalation"),
    ]:
        truth = merged[f"{task}_true"].astype(bool)
        pred = merged[f"{task}_pred"].astype(bool)

        result[f"{prefix}_precision"] = precision_score(
            truth, pred, zero_division=0
        )
        result[f"{prefix}_recall"] = recall_score(
            truth, pred, zero_division=0
        )
        result[f"{prefix}_f1"] = f1_score(
            truth, pred, zero_division=0
        )

    result["direction_accuracy"] = accuracy_score(
        merged["direction_true"],
        merged["direction_pred"],
    )

    results.append(result)

    merged["scenario"] = merged["example_id"].str.rsplit(
        "::", n=1
    ).str[-1]

    for scenario, group in merged.groupby("scenario"):
        scenario_results.append({
            "model": model_name,
            "scenario": scenario,
            "examples": len(group),
            "meaningful_accuracy": accuracy_score(
                group["meaningful_change_true"],
                group["meaningful_change_pred"],
            ),
            "escalation_accuracy": accuracy_score(
                group["escalation_needed_true"],
                group["escalation_needed_pred"],
            ),
            "direction_accuracy": accuracy_score(
                group["direction_true"],
                group["direction_pred"],
            ),
        })

metrics_df = pd.DataFrame(results)
scenario_df = pd.DataFrame(scenario_results)

print("========== STEP 65 ==========")
print("\nOverall baseline metrics:")
display(metrics_df.round(4))

print("\nScenario breakdown:")
display(scenario_df.round(4))

def save_csv_immutable(filename, df):
    path = BASE / filename
    content = df.to_csv(index=False)

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"{filename} exists with different content."
        )
    else:
        path.write_text(content, encoding="utf-8")

    return hashlib.sha256(path.read_bytes()).hexdigest()

metrics_hash = save_csv_immutable(
    "multi_signal_baseline_metrics_v3.csv",
    metrics_df,
)

scenario_hash = save_csv_immutable(
    "multi_signal_scenario_metrics_v3.csv",
    scenario_df,
)

print("\nMetrics SHA-256:", metrics_hash)
print("Scenario SHA-256:", scenario_hash)
print("PASS — Frozen v3 baseline evaluation complete.")


========== STEP 65 ==========

Overall baseline metrics:


,model,examples,meaningful_precision,meaningful_recall,meaningful_f1,escalation_precision,escalation_recall,escalation_f1,direction_accuracy
0,Rules V1,1302,1.0,0.4,0.5714,0.0,0.0,0.0,0.6667
1,XGBoost,1302,0.0,0.0,0.0000,0.0,0.0,0.0,0.6667



Scenario breakdown:


,model,scenario,examples,meaningful_accuracy,escalation_accuracy,direction_accuracy
0,Rules V1,A_stable,217,1.0,1.0,1.0
1,Rules V1,B_purchasing_slowdown,217,1.0,1.0,1.0
2,Rules V1,C_payment_warning,217,0.0,1.0,0.0
3,Rules V1,D_reinforcing_warnings,217,1.0,0.0,1.0
4,Rules V1,E_improving_signals,217,0.0,1.0,0.0
5,Rules V1,F_conflicting_signals,217,0.0,0.0,1.0
6,XGBoost,A_stable,217,1.0,1.0,1.0
7,XGBoost,B_purchasing_slowdown,217,0.0,1.0,1.0
8,XGBoost,C_payment_warning,217,0.0,1.0,0.0
9,XGBoost,D_reinforcing_warnings,217,0.0,0.0,1.0



Metrics SHA-256: d1c7a3dd5b3f344474432dc032b98b44d7c7427ccaf9ba5d1f69964fd9100410
Scenario SHA-256: 5d1b29f3969ac22bce44c225297a0d8a806cbef876ab9fcaa633378be7dfe65a
PASS — Frozen v3 baseline evaluation complete.


In [69]:

# STEP 66 — Freeze a balanced 30-example Jev pilot
# No Jev calls yet. No API costs.

import json
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("cross_company_benchmark")
SEED = 20261018
CUSTOMERS = 5

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

inputs = read_jsonl(
    BASE / "multi_signal_model_inputs_v3.jsonl"
)
labels = read_jsonl(
    BASE / "multi_signal_reference_labels_v3.jsonl"
)

assert len(inputs) == len(labels) == 1302

def customer_key(example_id):
    return example_id.rsplit("::", 1)[0]

groups = defaultdict(list)

for item in inputs:
    groups[customer_key(item["example_id"])].append(item)

assert len(groups) == 217
assert all(len(group) == 6 for group in groups.values())

rng = random.Random(SEED)
selected_customers = set(
    rng.sample(sorted(groups), CUSTOMERS)
)

pilot_inputs = [
    item for item in inputs
    if customer_key(item["example_id"]) in selected_customers
]

pilot_ids = {
    item["example_id"] for item in pilot_inputs
}

pilot_labels = [
    item for item in labels
    if item["example_id"] in pilot_ids
]

assert len(pilot_inputs) == len(pilot_labels) == 30

# The runner must send ONLY item["input"] to Jev.
# Never send example_id or reference labels as model input.

assert all(
    "scenario_id" not in
    item["input"]["new_event"].get("observation", {})
    for item in pilot_inputs
)

def save_immutable(filename, rows):
    path = BASE / filename

    content = "".join(
        json.dumps(row, sort_keys=True) + "\n"
        for row in rows
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"{filename} already exists with different content."
        )
    else:
        path.write_text(content, encoding="utf-8")

    return hashlib.sha256(path.read_bytes()).hexdigest()

input_hash = save_immutable(
    "multi_signal_jev_pilot_inputs.jsonl",
    pilot_inputs,
)

label_hash = save_immutable(
    "multi_signal_jev_pilot_labels.jsonl",
    pilot_labels,
)

scenario_counts = Counter(
    item["example_id"].rsplit("::", 1)[-1]
    for item in pilot_inputs
)

print("========== STEP 66 ==========")
print("Seed:", SEED)
print("Selected customers:", len(selected_customers))
print("Pilot examples:", len(pilot_inputs))
print("Scenario counts:", dict(sorted(scenario_counts.items())))
print("Inputs SHA-256:", input_hash)
print("Labels SHA-256:", label_hash)
print("Saved:", BASE / "multi_signal_jev_pilot_inputs.jsonl")
print("Saved:", BASE / "multi_signal_jev_pilot_labels.jsonl")
print("PASS — Balanced Jev pilot frozen.")


========== STEP 66 ==========
Seed: 20261018
Selected customers: 5
Pilot examples: 30
Scenario counts: {'A_stable': 5, 'B_purchasing_slowdown': 5, 'C_payment_warning': 5, 'D_reinforcing_warnings': 5, 'E_improving_signals': 5, 'F_conflicting_signals': 5}
Inputs SHA-256: 1f0a21aa7b6a4ba619328ab4599fbea477871d9fd8d8d6c5c308ade768b61048
Labels SHA-256: dfa76955f210efcd51342c5227c282df21a0584a86e2e55d9bbdc3e0e66d92b2
Saved: cross_company_benchmark/multi_signal_jev_pilot_inputs.jsonl
Saved: cross_company_benchmark/multi_signal_jev_pilot_labels.jsonl
PASS — Balanced Jev pilot frozen.


In [70]:
from pathlib import Path

matches = list(Path.cwd().rglob("multi_signal_jev_pilot_inputs.jsonl"))

for path in matches:
    print(path.resolve())

print("Files found:", len(matches))

/Users/anushreenema/cross_company_benchmark/multi_signal_jev_pilot_inputs.jsonl
Files found: 1


In [71]:
import json
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

base = Path("/Users/anushreenema/cross_company_benchmark")

def load_jsonl(filename):
    with (base / filename).open() as f:
        return [json.loads(line) for line in f if line.strip()]

labels = load_jsonl("multi_signal_jev_pilot_labels.jsonl")
predictions = load_jsonl("multi_signal_jev_pilot_predictions.jsonl")

label_by_id = {r["example_id"]: r for r in labels}
pred_by_id = {r["example_id"]: r for r in predictions}

assert len(labels) == len(predictions) == 30
assert len(label_by_id) == len(pred_by_id) == 30
assert set(label_by_id) == set(pred_by_id), "Prediction IDs do not match labels"

ids = [r["example_id"] for r in labels]

print("JEV PILOT RESULTS — 30 EXAMPLES")
print("-" * 45)

for field in ["meaningful_change", "escalation_needed"]:
    y_true = [label_by_id[i][field] for i in ids]
    y_pred = [pred_by_id[i][field] for i in ids]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    print(f"{field}:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1:        {f1:.4f}")

direction_true = [label_by_id[i]["direction"] for i in ids]
direction_pred = [pred_by_id[i]["direction"] for i in ids]

print(f"direction_accuracy: {accuracy_score(direction_true, direction_pred):.4f}")

JEV PILOT RESULTS — 30 EXAMPLES
---------------------------------------------
meaningful_change:
  Precision: 1.0000
  Recall:    0.6000
  F1:        0.7500
escalation_needed:
  Precision: 0.6429
  Recall:    0.9000
  F1:        0.7500
direction_accuracy: 0.6667


In [72]:
import json
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

base = Path("/Users/anushreenema/cross_company_benchmark")

def load_rows(filename):
    with (base / filename).open() as f:
        return [json.loads(line) for line in f if line.strip()]

labels = load_rows("multi_signal_jev_pilot_labels.jsonl")
jev = load_rows("multi_signal_jev_pilot_predictions.jsonl")
rules = load_rows("multi_signal_rules_predictions_v2.jsonl")
xgb = load_rows("multi_signal_xgboost_predictions_v2.jsonl")

truth = {r["example_id"]: r for r in labels}
ids = list(truth)

assert len(ids) == len(set(ids)) == 30

def evaluate(name, rows):
    preds = {r["example_id"]: r for r in rows}
    assert len(preds) == len(rows), f"Duplicate IDs in {name}"
    assert all(i in preds for i in ids), f"Missing predictions in {name}"

    result = {"Model": name}

    for field, prefix in [
        ("meaningful_change", "Meaningful"),
        ("escalation_needed", "Escalation")
    ]:
        y_true = [truth[i][field] for i in ids]
        y_pred = [preds[i][field] for i in ids]

        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )
        result[f"{prefix} F1"] = round(f1, 4)

    result["Direction accuracy"] = round(
        accuracy_score(
            [truth[i]["direction"] for i in ids],
            [preds[i]["direction"] for i in ids]
        ), 4
    )

    return result

import pandas as pd

comparison = pd.DataFrame([
    evaluate("Rules V1", rules),
    evaluate("XGBoost", xgb),
    evaluate("Jev", jev),
])

print("SAME 30 EXAMPLES — THREE-MODEL COMPARISON")
display(comparison)

SAME 30 EXAMPLES — THREE-MODEL COMPARISON


,Model,Meaningful F1,Escalation F1,Direction accuracy
0,Rules V1,0.5714,0.00,0.6667
1,XGBoost,0.0000,0.00,0.6667
2,Jev,0.7500,0.75,0.6667


In [73]:
from pathlib import Path
import hashlib

path = Path(
    "/Users/anushreenema/cross_company_benchmark/"
    "multi_signal_model_inputs_v3.jsonl"
)

expected = "fa3f8045bfbb6cec852b2cd14e669099106043a55c8b49bb21b707cdc9fdc280"

assert path.exists(), "Full benchmark file missing"

digest = hashlib.sha256(path.read_bytes()).hexdigest()
assert digest == expected, "Frozen benchmark checksum mismatch"

print("PASS — Full benchmark verified")
print("SHA-256:", digest)
print("Examples:", sum(1 for line in path.open() if line.strip()))

PASS — Full benchmark verified
SHA-256: fa3f8045bfbb6cec852b2cd14e669099106043a55c8b49bb21b707cdc9fdc280
Examples: 1302


In [74]:
import json
import hashlib
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

full_path = base / "multi_signal_model_inputs_v3.jsonl"
pilot_path = base / "multi_signal_jev_pilot_predictions.jsonl"
remaining_path = base / "multi_signal_jev_remaining_inputs.jsonl"

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

full = read_jsonl(full_path)
pilot_predictions = read_jsonl(pilot_path)

assert len(full) == 1302
assert len(pilot_predictions) == 30

full_ids = [r["example_id"] for r in full]
pilot_ids = [r["example_id"] for r in pilot_predictions]

assert len(set(full_ids)) == 1302
assert len(set(pilot_ids)) == 30
assert set(pilot_ids).issubset(set(full_ids))

remaining = [
    row for row in full
    if row["example_id"] not in set(pilot_ids)
]

assert len(remaining) == 1272
assert all(
    isinstance(row["input"], dict)
    for row in remaining
)

# Preserve the frozen benchmark and existing pilot results.
assert not remaining_path.exists(), "Remaining-input file already exists"

with remaining_path.open("w", encoding="utf-8") as f:
    for row in remaining:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

digest = hashlib.sha256(remaining_path.read_bytes()).hexdigest()

print("PASS — Remaining Jev inputs created")
print("Full benchmark:", len(full))
print("Completed pilot:", len(pilot_predictions))
print("Remaining:", len(remaining))
print("SHA-256:", digest)
print("File:", remaining_path)
print("No Jev API calls made")

PASS — Remaining Jev inputs created
Full benchmark: 1302
Completed pilot: 30
Remaining: 1272
SHA-256: 7956bb325f9ff147b095639d43eafcc23174e6646ee6c174f0bb07ce8eba8c5d
File: /Users/anushreenema/cross_company_benchmark/multi_signal_jev_remaining_inputs.jsonl
No Jev API calls made


In [76]:
import json
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

def load(name):
    with (base / name).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

full = load("multi_signal_model_inputs_v3.jsonl")
pilot = load("multi_signal_jev_pilot_predictions.jsonl")
remaining = load("multi_signal_jev_remaining_predictions.jsonl")

assert len(full) == 1302
assert len(pilot) == 30
assert len(remaining) == 1272

combined = pilot + remaining
predictions = {r["example_id"]: r for r in combined}
expected_ids = [r["example_id"] for r in full]

assert len(predictions) == 1302, "Duplicate prediction IDs"
assert len(set(expected_ids)) == 1302
assert set(predictions) == set(expected_ids), "Prediction IDs mismatch"

output = base / "multi_signal_jev_predictions_v3.jsonl"

assert not output.exists(), "Output already exists; refusing to overwrite"

with output.open("w", encoding="utf-8") as f:
    for eid in expected_ids:
        f.write(json.dumps(predictions[eid], ensure_ascii=False) + "\n")

print("PASS — Full Jev benchmark assembled")
print("Pilot:", len(pilot))
print("Remaining:", len(remaining))
print("Total:", len(predictions))
print("Saved:", output)

PASS — Full Jev benchmark assembled
Pilot: 30
Remaining: 1272
Total: 1302
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_jev_predictions_v3.jsonl


In [77]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

base = Path("/Users/anushreenema/cross_company_benchmark")

def load_jsonl(filename):
    with (base / filename).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

labels = load_jsonl("multi_signal_reference_labels_v3.jsonl")
jev = load_jsonl("multi_signal_jev_predictions_v3.jsonl")
rules = load_jsonl("multi_signal_rules_predictions_v2.jsonl")
xgb = load_jsonl("multi_signal_xgboost_predictions_v2.jsonl")

truth = {r["example_id"]: r for r in labels}
ids = list(truth)

assert len(labels) == len(truth) == 1302

def evaluate(name, rows):
    preds = {r["example_id"]: r for r in rows}

    assert len(preds) == len(rows), f"Duplicate IDs in {name}"
    assert all(i in preds for i in ids), f"Missing predictions in {name}"

    result = {"Model": name}

    for field, label in [
        ("meaningful_change", "Meaningful"),
        ("escalation_needed", "Escalation")
    ]:
        y_true = [truth[i][field] for i in ids]
        y_pred = [preds[i][field] for i in ids]

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )

        result[f"{label} precision"] = round(precision, 4)
        result[f"{label} recall"] = round(recall, 4)
        result[f"{label} F1"] = round(f1, 4)

    result["Direction accuracy"] = round(
        accuracy_score(
            [truth[i]["direction"] for i in ids],
            [preds[i]["direction"] for i in ids]
        ),
        4
    )

    return result

comparison = pd.DataFrame([
    evaluate("Rules V1", rules),
    evaluate("XGBoost", xgb),
    evaluate("Jev", jev)
])

display(comparison)

output = base / "multi_signal_final_comparison_v3.csv"
comparison.to_csv(output, index=False)

print("PASS — Final comparison calculated")
print("Examples:", len(ids))
print("Saved:", output)

,Model,Meaningful precision,Meaningful recall,Meaningful F1,Escalation precision,Escalation recall,Escalation F1,Direction accuracy
0,Rules V1,1.0,0.4000,0.5714,0.0000,0.0000,0.0000,0.6667
1,XGBoost,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.6667
2,Jev,1.0,0.5926,0.7442,0.5943,0.7696,0.6707,0.6667


PASS — Final comparison calculated
Examples: 1302
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_final_comparison_v3.csv


In [78]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

base = Path("/Users/anushreenema/cross_company_benchmark")

def load(name):
    with (base / name).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

labels = load("multi_signal_reference_labels_v3.jsonl")
predictions = load("multi_signal_jev_predictions_v3.jsonl")

pred_by_id = {p["example_id"]: p for p in predictions}

rows = []

for label in labels:
    eid = label["example_id"]
    pred = pred_by_id[eid]

    scenario = eid.split("::")[-1]

    rows.append({
        "scenario": scenario,
        "meaningful_correct": (
            label["meaningful_change"] == pred["meaningful_change"]
        ),
        "escalation_correct": (
            label["escalation_needed"] == pred["escalation_needed"]
        ),
        "direction_correct": (
            label["direction"] == pred["direction"]
        ),
    })

df = pd.DataFrame(rows)

summary = df.groupby("scenario").agg(
    examples=("scenario", "size"),
    meaningful_accuracy=("meaningful_correct", "mean"),
    escalation_accuracy=("escalation_correct", "mean"),
    direction_accuracy=("direction_correct", "mean"),
).reset_index()

display(summary.round(4))

,scenario,examples,meaningful_accuracy,escalation_accuracy,direction_accuracy
0,A_stable,217,1.0000,1.0000,1.0
1,B_purchasing_slowdown,217,1.0000,0.0000,1.0
2,C_payment_warning,217,0.0000,1.0000,0.0
3,D_reinforcing_warnings,217,1.0000,1.0000,1.0
4,E_improving_signals,217,0.8710,0.9493,1.0
5,F_conflicting_signals,217,0.0922,0.5392,0.0


In [79]:
import json
import pandas as pd
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

with (base / "multi_signal_jev_predictions_v3.jsonl").open() as f:
    predictions = [json.loads(line) for line in f if line.strip()]

rows = []

for p in predictions:
    scenario = p["example_id"].split("::")[-1]

    if scenario in {
        "B_purchasing_slowdown",
        "C_payment_warning",
        "F_conflicting_signals",
    }:
        rows.append({
            "scenario": scenario,
            "meaningful": p["meaningful_change"],
            "direction": p["direction"],
            "escalation": p["escalation_needed"],
        })

df = pd.DataFrame(rows)

for scenario, group in df.groupby("scenario", sort=True):
    print(f"\n{scenario} ({len(group)} examples)")
    print("Meaningful:", group["meaningful"].value_counts().to_dict())
    print("Direction:", group["direction"].value_counts().to_dict())
    print("Escalation:", group["escalation"].value_counts().to_dict())


B_purchasing_slowdown (217 examples)
Meaningful: {True: 217}
Direction: {'negative': 217}
Escalation: {True: 217}

C_payment_warning (217 examples)
Meaningful: {False: 217}
Direction: {'neutral': 217}
Escalation: {False: 217}

F_conflicting_signals (217 examples)
Meaningful: {False: 197, True: 20}
Direction: {'positive': 217}
Escalation: {True: 117, False: 100}


In [80]:
import json
from pathlib import Path

path = Path(
    "/Users/anushreenema/cross_company_benchmark/"
    "multi_signal_model_inputs_v3.jsonl"
)

with path.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

for scenario in ["B_purchasing_slowdown", "C_payment_warning", "F_conflicting_signals"]:
    row = next(
        r for r in rows
        if r["example_id"].endswith("::" + scenario)
    )

    print(f"\n===== {scenario} =====")
    print(json.dumps(row["input"], indent=2)[:5000])


===== B_purchasing_slowdown =====
{
  "current_state": {
    "max_days_overdue": 1,
    "outstanding_ar": 192.38,
    "overdue_ar": 87.8,
    "overdue_invoice_count": 1,
    "previous_30d_invoice_count": 5,
    "previous_30d_revenue": 3376.27,
    "recent_30d_invoice_count": 3,
    "recent_30d_revenue": 2194.58
  },
  "event_date": "2026-10-06",
  "event_impact": {
    "outstanding_ar_delta": 0.0,
    "overdue_ar_delta": 0.0,
    "recent_30d_invoice_count_delta": -2,
    "recent_30d_revenue_delta": -1181.69
  },
  "new_event": {
    "amount": 0.0,
    "description": "Hypothetical multi-signal financial trend report comparing a customer baseline with a simulated subsequent 30-day period. The observations are synthetic and do not modify actual accounting records.",
    "event_date": "2026-10-06",
    "event_type": "financial_trend_observed",
    "observation": {
      "baseline_30d_invoice_count": 5,
      "baseline_30d_revenue": 3376.27,
      "observed_30d_invoice_count": 3,
      "ob

In [81]:
import json
from pathlib import Path

path = Path(
    "/Users/anushreenema/cross_company_benchmark/"
    "multi_signal_model_inputs_v3.jsonl"
)

rows = [
    json.loads(line)
    for line in path.open(encoding="utf-8")
    if line.strip()
]

payment_rows = [
    r for r in rows
    if r["example_id"].endswith("::C_payment_warning")
]

assert len(payment_rows) == 217

ar_increased = sum(
    r["input"]["current_state"]["overdue_ar"]
    > r["input"]["old_state"]["overdue_ar"]
    for r in payment_rows
)

age_increased = sum(
    r["input"]["current_state"]["max_days_overdue"]
    > r["input"]["old_state"]["max_days_overdue"]
    for r in payment_rows
)

print("Payment-warning examples:", len(payment_rows))
print("Overdue AR increased:", ar_increased)
print("Maximum overdue age increased:", age_increased)

Payment-warning examples: 217
Overdue AR increased: 217
Maximum overdue age increased: 217


In [82]:
import json
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")
output = base / "multi_signal_benchmark_findings_v3.json"

assert not output.exists(), "Findings file already exists"

findings = {
    "benchmark": "cross_company_multi_signal_v3",
    "examples": 1302,
    "eligible_customers": 217,
    "scenarios_per_customer": 6,
    "reference_type": "synthetic_scenario_policy",
    "metrics": {
        "rules_v1": {
            "meaningful_f1": 0.5714,
            "escalation_f1": 0.0,
            "direction_accuracy": 0.6667
        },
        "xgboost": {
            "meaningful_f1": 0.0,
            "escalation_f1": 0.0,
            "direction_accuracy": 0.6667
        },
        "jev": {
            "meaningful_f1": 0.7442,
            "escalation_f1": 0.6707,
            "direction_accuracy": 0.6667
        }
    },
    "verified_failure": {
        "scenario": "C_payment_warning",
        "examples": 217,
        "overdue_ar_increased": 217,
        "overdue_age_increased": 217,
        "jev_meaningful_detected": 0,
        "jev_negative_direction": 0
    },
    "limitations": [
        "Synthetic reference labels, not independently verified outcomes",
        "Eligible-customer subset excludes 91 customers",
        "Scenario labels, especially conflicting-signal escalation, require human review",
        "Frozen baseline models were not retrained for this distribution"
    ]
}

output.write_text(json.dumps(findings, indent=2) + "\n")

print("PASS — Benchmark findings frozen")
print("Saved:", output)

PASS — Benchmark findings frozen
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_benchmark_findings_v3.json


In [83]:
import json
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

path = base / "multi_signal_model_inputs_v3.jsonl"

with path.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

assert len(rows) == 1302
assert len({r["example_id"] for r in rows}) == 1302
assert all("input" in r for r in rows)

batch = [
    {
        "event_index": i,
        "event": rows[i]["input"]
    }
    for i in range(20)
]

assert all("example_id" not in item for item in batch)

print("PASS — LLM generalization inputs verified")
print("Total examples:", len(rows))
print("First batch:", len(batch))
print("Total API requests:", (len(rows) + 19) // 20)
print("First event keys:", list(batch[0]["event"].keys()))
print("Scenario IDs excluded from API payload: YES")

PASS — LLM generalization inputs verified
Total examples: 1302
First batch: 20
Total API requests: 66
First event keys: ['current_state', 'event_date', 'event_impact', 'new_event', 'old_state', 'previous_event']
Scenario IDs excluded from API payload: YES


In [84]:
import json
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

gpt_path = Path(
    "/Users/anushreenema/Desktop/la_ai_hacks_bulding/"
    "generalization_benchmark_llm/outputs/"
    "gpt_generalization_v3/gpt_predictions.jsonl"
)

with (base / "multi_signal_model_inputs_v3.jsonl").open() as f:
    inputs = [json.loads(line) for line in f if line.strip()]

with gpt_path.open() as f:
    predictions = [json.loads(line) for line in f if line.strip()]

assert len(inputs) == len(predictions) == 1302

indices = [p["event_index"] for p in predictions]

assert indices == list(range(1302)), (
    "GPT prediction indices are missing, duplicated, or out of order"
)

assert len({r["example_id"] for r in inputs}) == 1302

print("PASS — GPT generalization benchmark verified")
print("Inputs:", len(inputs))
print("Predictions:", len(predictions))
print("Unique indices:", len(set(indices)))
print("Missing predictions: 0")

PASS — GPT generalization benchmark verified
Inputs: 1302
Predictions: 1302
Unique indices: 1302
Missing predictions: 0


In [85]:
import shutil
import hashlib
from pathlib import Path

base = Path("/Users/anushreenema/cross_company_benchmark")

source = Path(
    "/Users/anushreenema/Desktop/la_ai_hacks_bulding/"
    "generalization_benchmark_llm/outputs/"
    "gpt_generalization_v3/gpt_predictions.jsonl"
)

destination = base / "multi_signal_gpt_predictions_v3.jsonl"

assert source.exists()
assert not destination.exists(), "Destination already exists; refusing to overwrite"

shutil.copy2(source, destination)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(source) == sha256(destination)

print("PASS — GPT predictions frozen")
print("Saved:", destination)
print("SHA256:", sha256(destination))

PASS — GPT predictions frozen
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_gpt_predictions_v3.jsonl
SHA256: dc1d070cf96131b178fecb8579e9ed0504b3c9d46e6da4f3a520f0f844aaa5dc


In [86]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
)

base = Path("/Users/anushreenema/cross_company_benchmark")

def load(name):
    with (base / name).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

inputs = load("multi_signal_model_inputs_v3.jsonl")
labels = load("multi_signal_reference_labels_v3.jsonl")

ids = [r["example_id"] for r in inputs]
truth = {r["example_id"]: r for r in labels}

assert len(ids) == len(set(ids)) == len(truth) == 1302
assert set(ids) == set(truth)

models = {
    "Rules V1": load("multi_signal_rules_predictions_v2.jsonl"),
    "XGBoost": load("multi_signal_xgboost_predictions_v2.jsonl"),
    "Jev": load("multi_signal_jev_predictions_v3.jsonl"),
}

# GPT uses local event_index rather than example_id.
gpt_rows = load("multi_signal_gpt_predictions_v3.jsonl")

assert len(gpt_rows) == 1302
assert [r["event_index"] for r in gpt_rows] == list(range(1302))

models["GPT-5.4-mini"] = [
    {"example_id": ids[i], **{
        k: gpt_rows[i][k]
        for k in (
            "meaningful_change",
            "direction",
            "escalation_needed",
        )
    }}
    for i in range(1302)
]

results = []

for model_name, rows in models.items():
    pred = {r["example_id"]: r for r in rows}

    assert len(pred) == len(rows), f"Duplicate IDs: {model_name}"
    assert set(ids).issubset(pred), f"Missing IDs: {model_name}"

    result = {"Model": model_name, "Examples": 1302}

    for field, prefix in [
        ("meaningful_change", "Meaningful"),
        ("escalation_needed", "Escalation"),
    ]:
        y_true = [truth[i][field] for i in ids]
        y_pred = [pred[i][field] for i in ids]

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred,
            average="binary",
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[False, True]
        ).ravel()

        result[f"{prefix} Precision"] = round(float(precision), 4)
        result[f"{prefix} Recall"] = round(float(recall), 4)
        result[f"{prefix} F1"] = round(float(f1), 4)
        result[f"{prefix} FP"] = int(fp)
        result[f"{prefix} FN"] = int(fn)

    result["Direction Accuracy"] = round(
        accuracy_score(
            [truth[i]["direction"] for i in ids],
            [pred[i]["direction"] for i in ids],
        ),
        4,
    )

    results.append(result)

comparison = pd.DataFrame(results)

output = base / "multi_signal_four_model_comparison_v3.csv"
assert not output.exists(), "Comparison CSV already exists"

comparison.to_csv(output, index=False)

display(comparison)

print("\nPASS — Four-model benchmark comparison saved")
print("Saved:", output)

,Model,Examples,Meaningful Precision,Meaningful Recall,Meaningful F1,Meaningful FP,Meaningful FN,Escalation Precision,Escalation Recall,Escalation F1,Escalation FP,Escalation FN,Direction Accuracy
0,Rules V1,1302,1.0,0.4000,0.5714,0,651,0.0000,0.0000,0.0000,0,434,0.6667
1,XGBoost,1302,0.0,0.0000,0.0000,0,1085,0.0000,0.0000,0.0000,0,434,0.6667
2,Jev,1302,1.0,0.5926,0.7442,0,442,0.5943,0.7696,0.6707,228,100,0.6667
3,GPT-5.4-mini,1302,1.0,0.9899,0.9949,0,11,0.4432,0.8548,0.5838,466,63,0.8226



PASS — Four-model benchmark comparison saved
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_four_model_comparison_v3.csv


In [87]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
)

base = Path("/Users/anushreenema/cross_company_benchmark")

def load(name):
    with (base / name).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

inputs = load("multi_signal_model_inputs_v3.jsonl")
labels = load("multi_signal_reference_labels_v3.jsonl")

ids = [r["example_id"] for r in inputs]
truth = {r["example_id"]: r for r in labels}

assert len(ids) == len(set(ids)) == len(truth) == 1302
assert set(ids) == set(truth)

# Models indexed by example_id
models = {}

for name, filename in [
    ("Rules V1", "multi_signal_rules_predictions_v2.jsonl"),
    ("XGBoost", "multi_signal_xgboost_predictions_v2.jsonl"),
    ("Jev", "multi_signal_jev_predictions_v3.jsonl"),
]:
    rows = load(filename)
    by_id = {r["example_id"]: r for r in rows}

    assert len(by_id) == len(rows)
    assert set(ids).issubset(by_id)

    models[name] = [by_id[i] for i in ids]

# GPT predictions indexed by event_index
gpt = load("multi_signal_gpt_predictions_v3.jsonl")

assert len(gpt) == 1302
assert [r["event_index"] for r in gpt] == list(range(1302))

models["GPT-5.4-mini"] = gpt

# An actionable alert requires BOTH conditions.
y_true = [
    bool(truth[i]["meaningful_change"])
    and bool(truth[i]["escalation_needed"])
    for i in ids
]

results = []

for name, predictions in models.items():
    y_pred = [
        bool(p["meaningful_change"])
        and bool(p["escalation_needed"])
        for p in predictions
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[False, True],
    ).ravel()

    results.append({
        "Model": name,
        "Actionable Precision": round(float(precision), 4),
        "Actionable Recall": round(float(recall), 4),
        "Actionable F1": round(float(f1), 4),
        "True Alerts": int(tp),
        "False Alerts": int(fp),
        "Missed Alerts": int(fn),
    })

comparison = pd.DataFrame(results)

display(comparison)

output = base / "multi_signal_actionable_alert_comparison_v3.csv"
assert not output.exists(), "Output already exists"

comparison.to_csv(output, index=False)

print("\nPASS — End-to-end actionable-alert evaluation complete")
print("Reference actionable alerts:", sum(y_true))
print("Saved:", output)

,Model,Actionable Precision,Actionable Recall,Actionable F1,True Alerts,False Alerts,Missed Alerts
0,Rules V1,0.0000,0.0000,0.0000,0,0,434
1,XGBoost,0.0000,0.0000,0.0000,0,0,434
2,Jev,0.5033,0.5323,0.5174,231,228,203
3,GPT-5.4-mini,0.4448,0.8548,0.5852,371,463,63



PASS — End-to-end actionable-alert evaluation complete
Reference actionable alerts: 434
Saved: /Users/anushreenema/cross_company_benchmark/multi_signal_actionable_alert_comparison_v3.csv
